# 3\.3\.4 Calibrate and Generate Gaussian Mixture Model Anomalies

Gaussian Mixture Models provide a probabilistic approach to anomaly detection that differs substantially from both DBSCAN and Isolation Forest\. Rather than identifying observations located in sparse regions of feature space or observations that are easily isolated through recursive partitioning, Gaussian Mixture Models estimate the probability distribution of mobility states and identify observations that fall within the low\-likelihood tails of the fitted distribution\.

This notebook calibrates Gaussian Mixture Models inside the shared anomaly framework established in 3\.3\.1\. The notebook is intentionally lean: PCA is the sole retained representation, the shared temporal\-bucket\-plus\-policy\-geography comparison universe remains inherited rather than re\-tested, and the goal is to produce a credible probabilistic anomaly surface for downstream comparison rather than to reopen another representation bakeoff\.

> 📝 Note\. Unlike DBSCAN, GMM does not require density\-homogeneous neighborhoods\. Unlike Isolation Forest, GMM explicitly models the probability distribution of mobility states\. That makes it a useful third anomaly\-detection lens: DBSCAN gives us a density\-based view, Isolation Forest gives us an isolation\-based view, and GMM gives us a probabilistic likelihood\-based view\. 

\-\-\-

In [1]:
# ---------------------------------------------------------------------
# Import libraries for GMM calibration and anomaly generation
# ---------------------------------------------------------------------

from pathlib import Path
from time import perf_counter
import json

import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from shapely import wkb

from sklearn.mixture import GaussianMixture

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_MAP_COLORS,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = BRAND_COLOR_SEQUENCE

# Keep existing downstream Set2 palette references brand-aligned without
# changing the plotting logic in those cells.
px.colors.qualitative.Set2 = BRAND_COLOR_SEQUENCE

# Make plotly_white behave like the project brand template so downstream
# cells that already use template="plotly_white" inherit the same look.
pio.templates["plotly_white"].layout.paper_bgcolor = "white"
pio.templates["plotly_white"].layout.plot_bgcolor = BRAND_COLORS["ice"]
pio.templates["plotly_white"].layout.font = {"color": BRAND_COLORS["dark_teal"]}
pio.templates["plotly_white"].layout.title = {"font": {"color": BRAND_COLORS["dark_teal"]}}
pio.templates["plotly_white"].layout.legend = {
    "bgcolor": "rgba(255,255,255,0.85)",
    "bordercolor": BRAND_COLORS["seafoam"],
    "borderwidth": 1,
    "font": {"color": BRAND_COLORS["dark_teal"]},
}
pio.templates["plotly_white"].layout.xaxis = {
    "gridcolor": "rgba(11, 78, 74, 0.14)",
    "zerolinecolor": "rgba(11, 78, 74, 0.20)",
    "tickfont": {"color": BRAND_COLORS["dark_teal"]},
    "title": {"font": {"color": BRAND_COLORS["dark_teal"]}},
}
pio.templates["plotly_white"].layout.yaxis = {
    "gridcolor": "rgba(11, 78, 74, 0.14)",
    "zerolinecolor": "rgba(11, 78, 74, 0.20)",
    "tickfont": {"color": BRAND_COLORS["dark_teal"]},
    "title": {"font": {"color": BRAND_COLORS["dark_teal"]}},
}

LEGACY_COLOR_MAP = {
    "#4c78a8": BRAND_COLORS["dark_teal"],
    "#e45756": BRAND_COLORS["terracotta"],
    "#72b7b2": BRAND_COLORS["seafoam"],
    "gray": BRAND_COLORS["seafoam"],
    "grey": BRAND_COLORS["seafoam"],
}


def remap_legacy_color(value):
    if isinstance(value, str):
        return LEGACY_COLOR_MAP.get(value.strip().lower(), value)
    if isinstance(value, (list, tuple)):
        return [remap_legacy_color(item) for item in value]
    return value


def apply_brand_plotly_layout(
    fig,
    *,
    title=None,
    width=None,
    height=None,
    legend_title=None,
    margin=None,
):
    fig.update_layout(
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor=BRAND_COLORS["ice"],
        font={"color": BRAND_COLORS["dark_teal"]},
        title={
            "text": title,
            "font": {"color": BRAND_COLORS["dark_teal"]},
        } if title else None,
        width=width,
        height=height,
        margin=margin or {"l": 70, "r": 40, "t": 80, "b": 70},
    )

    fig.update_xaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )
    fig.update_yaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )

    if legend_title is not None:
        fig.update_layout(
            legend={
                "title": {"text": legend_title},
                "orientation": "h",
                "yanchor": "top",
                "y": -0.18,
                "xanchor": "center",
                "x": 0.5,
                "bgcolor": "rgba(255,255,255,0.85)",
                "bordercolor": BRAND_COLORS["seafoam"],
                "borderwidth": 1,
                "font": {"color": BRAND_COLORS["dark_teal"]},
            }
        )

    return fig


def remap_legacy_plotly_colors(fig):
    apply_brand_plotly_layout(fig)

    for trace in fig.data:
        if hasattr(trace, "marker") and trace.marker is not None:
            if hasattr(trace.marker, "color"):
                trace.marker.color = remap_legacy_color(trace.marker.color)
            if hasattr(trace.marker, "line") and trace.marker.line is not None:
                if hasattr(trace.marker.line, "color"):
                    trace.marker.line.color = remap_legacy_color(trace.marker.line.color)

        if hasattr(trace, "line") and trace.line is not None:
            if hasattr(trace.line, "color"):
                trace.line.color = remap_legacy_color(trace.line.color)

        if hasattr(trace, "fillcolor"):
            trace.fillcolor = remap_legacy_color(trace.fillcolor)

        if hasattr(trace, "increasing") and trace.increasing is not None:
            if hasattr(trace.increasing, "marker") and trace.increasing.marker is not None:
                if hasattr(trace.increasing.marker, "color"):
                    trace.increasing.marker.color = remap_legacy_color(trace.increasing.marker.color)
        if hasattr(trace, "decreasing") and trace.decreasing is not None:
            if hasattr(trace.decreasing, "marker") and trace.decreasing.marker is not None:
                if hasattr(trace.decreasing.marker, "color"):
                    trace.decreasing.marker.color = remap_legacy_color(trace.decreasing.marker.color)

    if fig.layout.shapes:
        for shape in fig.layout.shapes:
            if hasattr(shape, "line") and shape.line is not None and hasattr(shape.line, "color"):
                shape.line.color = remap_legacy_color(shape.line.color)
            if hasattr(shape, "fillcolor"):
                shape.fillcolor = remap_legacy_color(shape.fillcolor)

    if fig.layout.annotations:
        for annotation in fig.layout.annotations:
            if hasattr(annotation, "font") and annotation.font is not None and hasattr(annotation.font, "color"):
                annotation.font.color = remap_legacy_color(annotation.font.color)

    return fig


if not hasattr(go.Figure, "_brand_original_show"):
    go.Figure._brand_original_show = go.Figure.show

    def _brand_show(self, *args, **kwargs):
        remap_legacy_plotly_colors(self)
        return go.Figure._brand_original_show(self, *args, **kwargs)

    go.Figure.show = _brand_show

In [2]:
# ---------------------------------------------------------------------
# Define project directories and notebook output locations
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/datasets/_deepnote_work")
PIPELINE_DATA_DIR = PROJECT_ROOT / "pipeline_data"

FINAL_311_DIR = PIPELINE_DATA_DIR / "3.1.1.final_tables"
FINAL_321_DIR = PIPELINE_DATA_DIR / "3.2.1.final_tables"
FINAL_322_DIR = PIPELINE_DATA_DIR / "3.2.2.final_tables"
FINAL_331_DIR = PIPELINE_DATA_DIR / "3.3.1.final_tables"
FINAL_332_DIR = PIPELINE_DATA_DIR / "3.3.2.final_tables"
FINAL_333_DIR = PIPELINE_DATA_DIR / "3.3.3.final_tables"

INTERMEDIATE_334_DIR = PIPELINE_DATA_DIR / "3.3.4.intermediate_tables"
FINAL_334_DIR = PIPELINE_DATA_DIR / "3.3.4.final_tables"

INTERMEDIATE_334_DIR.mkdir(parents=True, exist_ok=True)
FINAL_334_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# ---------------------------------------------------------------------
# Define inherited anomaly-framework defaults and core columns
# ---------------------------------------------------------------------

MODEL_FEATURE_SET_NAMES = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
GMM_REPRESENTATION_NAMES = ["pca_80pct"]

MODEL_ID_COLUMN = "modeling_row_id"
DATE_COLUMN = "date"
TEMPORAL_BUCKET_COLUMN = "temporal_bucket"
PRE_POST_CP_COLUMN = "pre_post_cp"
ZONE_ID_COLUMN = "taxi_zone_id"
ZONE_COLUMN = "zone"
BOROUGH_COLUMN = "borough"
POLICY_GEOGRAPHY_COLUMN = "policy_geography_class"

DEFAULT_METADATA_COLUMNS = [
    MODEL_ID_COLUMN,
    DATE_COLUMN,
    TEMPORAL_BUCKET_COLUMN,
    PRE_POST_CP_COLUMN,
    ZONE_ID_COLUMN,
    ZONE_COLUMN,
    BOROUGH_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
]

SHARED_COMPARISON_UNIVERSE_NAME = "same_temporal_bucket_and_policy_geography"
SHARED_COMPARISON_GROUPING_COLUMNS = [
    TEMPORAL_BUCKET_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
]

TAXI_PCA_HANDLING_DECISION = "split_prepost_period"

REPRESENTATION_ROLE_BY_NAME = {
    "pca_80pct": "primary_linear_representation",
}

In [4]:
# ---------------------------------------------------------------------
# Define candidate GMM grids and notebook rebuild toggles
# ---------------------------------------------------------------------

GMM_COMPONENT_GRID = [1, 2, 3, 5]
GMM_COVARIANCE_TYPE_GRID = ["diag", "full"]
GMM_TAIL_THRESHOLD_GRID = [0.005, 0.010, 0.020, 0.050]

GMM_RANDOM_STATE = 42
GMM_N_INIT = 5
GMM_MAX_ITER = 300
GMM_REG_COVAR = 1e-6

GMM_MAP_MODALITY_TO_REVIEW = "taxi"
GMM_TEMPORAL_BUCKET_TO_REVIEW = "weekday_pm_peak"

REBUILD_GMM_INPUT_SUMMARIES = False
REBUILD_GMM_STRUCTURE_CALIBRATION = False
REBUILD_GMM_TAIL_CALIBRATION = False
REBUILD_GMM_FRAMEWORK_COMPARISONS = False
REBUILD_GMM_AGGREGATE_REVIEW = False
REBUILD_GMM_FINAL_OUTPUTS = False
WRITE_334_OUTPUTS = True

In [5]:
# ---------------------------------------------------------------------
# Define inherited framework asset paths from 3.3.1, 3.3.2, and 3.3.3
# ---------------------------------------------------------------------

ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS = {
    "shared_representation_defaults": FINAL_331_DIR / "shared_representation_defaults.csv",
    "shared_calibration_defaults": FINAL_331_DIR / "shared_calibration_defaults.csv",
    "shared_comparison_universe_defaults": FINAL_331_DIR / "shared_comparison_universe_defaults.csv",
    "shared_framework_interpretation_defaults": FINAL_331_DIR / "shared_framework_interpretation_defaults.csv",
    "shared_framework_asset_manifest": FINAL_331_DIR / "shared_framework_asset_manifest.csv",
}

ROW_METADATA_PATHS_BY_SET = {
    feature_set: FINAL_311_DIR / f"{feature_set}_row_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

ANOMALY_CALIBRATION_ID_PATHS_BY_SET = {
    feature_set: FINAL_331_DIR / f"{feature_set}_anomaly_calibration_ids.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET = {
    feature_set: FINAL_331_DIR / f"{feature_set}_anomaly_calibration_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

PCA_80_FINAL_PATHS_BY_SET = {
    "bus": FINAL_321_DIR / "bus_pca_modeling_scores.parquet",
    "subway": FINAL_321_DIR / "subway_pca_modeling_scores.parquet",
    "taxi": FINAL_321_DIR / "taxi_pca_modeling_scores.parquet",
    "fhvhv": FINAL_321_DIR / "fhvhv_pca_modeling_scores.parquet",
    "multimodal": FINAL_321_DIR / "multimodal_pca_modeling_scores.parquet",
}

TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD = {
    "pre_cp": FINAL_322_DIR / "taxi_pre_cp_pca_80pct_modeling_scores.parquet",
    "post_cp": FINAL_322_DIR / "taxi_post_cp_pca_80pct_modeling_scores.parquet",
}

DBSCAN_SELECTED_CONFIGURATION_PATH = FINAL_332_DIR / "selected_dbscan_configurations.csv"
DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH = FINAL_332_DIR / "dbscan_anomaly_export_manifest.csv"
DBSCAN_FINAL_ROW_LEVEL_DIR = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts"

IF_SELECTED_CONFIGURATION_PATH = FINAL_333_DIR / "selected_isolation_forest_configurations.csv"
IF_ANOMALY_EXPORT_MANIFEST_PATH = FINAL_333_DIR / "isolation_forest_anomaly_export_manifest.csv"
IF_FINAL_ROW_LEVEL_DIR = FINAL_333_DIR / "isolation_forest_candidate_anomaly_row_level_parts"

In [6]:
# ---------------------------------------------------------------------
# Define 3.3.4 intermediate and final output paths
# ---------------------------------------------------------------------

GMM_INPUT_ASSET_SUMMARY_PATH = INTERMEDIATE_334_DIR / "gmm_input_asset_summary.parquet"
GMM_STRUCTURE_SWEEP_SUMMARY_PATH = INTERMEDIATE_334_DIR / "gmm_structure_sweep_summary.parquet"
GMM_STRUCTURE_SWEEP_DETAIL_PATH = INTERMEDIATE_334_DIR / "gmm_structure_sweep_detail.parquet"
GMM_STRUCTURE_SCREEN_PATH = INTERMEDIATE_334_DIR / "gmm_structure_screen.parquet"
GMM_ADVANCING_STRUCTURES_PATH = INTERMEDIATE_334_DIR / "gmm_advancing_structures.parquet"
GMM_TAIL_ROW_LEVEL_DIR = INTERMEDIATE_334_DIR / "gmm_tail_row_level_parts"
GMM_TAIL_SWEEP_SUMMARY_PATH = INTERMEDIATE_334_DIR / "gmm_tail_sweep_summary.parquet"
GMM_TAIL_SWEEP_DETAIL_PATH = INTERMEDIATE_334_DIR / "gmm_tail_sweep_detail.parquet"
GMM_FRAMEWORK_COMPARISON_PATH = INTERMEDIATE_334_DIR / "gmm_framework_comparison.parquet"
GMM_FRAMEWORK_COMPARISON_MANIFEST_PATH = INTERMEDIATE_334_DIR / "gmm_framework_comparison_manifest.parquet"
GMM_FRAMEWORK_COMPARISON_ROW_LEVEL_PATH = GMM_FRAMEWORK_COMPARISON_PATH
GMM_SPATIAL_FOOTPRINT_SUMMARY_PATH = INTERMEDIATE_334_DIR / "gmm_spatial_footprint_summary.parquet"
GMM_ZONE_MAP_SUMMARY_PATH = INTERMEDIATE_334_DIR / "gmm_zone_map_summary.parquet"

GMM_SELECTED_STRUCTURE_PATH = FINAL_334_DIR / "selected_gmm_structures.csv"
GMM_SELECTED_THRESHOLD_PATH = FINAL_334_DIR / "selected_gmm_tail_thresholds.csv"
GMM_SELECTED_CONFIGURATION_PATH = FINAL_334_DIR / "selected_gmm_configurations.csv"
GMM_ANOMALY_EXPORT_MANIFEST_PATH = FINAL_334_DIR / "gmm_anomaly_export_manifest.csv"
GMM_FINAL_ROW_LEVEL_DIR = FINAL_334_DIR / "gmm_candidate_anomaly_row_level_parts"

GMM_FINAL_PREPARED_PANELS_DIR = INTERMEDIATE_334_DIR / "gmm_final_prepared_panels"
GMM_FINAL_ROW_LEVEL_TEMP_DIR = INTERMEDIATE_334_DIR / "gmm_final_row_level_temp_parts"

GMM_FINAL_PREPARED_PANELS_DIR.mkdir(parents=True, exist_ok=True)
GMM_FINAL_ROW_LEVEL_TEMP_DIR.mkdir(parents=True, exist_ok=True)

GMM_TAIL_ROW_LEVEL_DIR.mkdir(parents=True, exist_ok=True)
GMM_FINAL_ROW_LEVEL_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# ---------------------------------------------------------------------
# Define general helper functions used throughout the notebook
# ---------------------------------------------------------------------

def sql_path(path_like):
    return str(Path(path_like)).replace("'", "''")


def duckdb_identifier(identifier):
    return '"' + str(identifier).replace('"', '""') + '"'


def should_rebuild(flag, output_path):
    return flag or not Path(output_path).exists()


def load_parquet_columns(parquet_path):
    return pd.read_parquet(parquet_path, engine="pyarrow").columns.tolist()


def load_shared_framework_table(table_key):
    table_path = ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS[table_key]
    if not table_path.exists():
        raise FileNotFoundError(f"Missing shared framework asset: {table_path}")
    if table_path.suffix.lower() == ".csv":
        return pd.read_csv(table_path)
    return pd.read_parquet(table_path)


def get_pca_representation_path(feature_set, row_metadata=None):
    if feature_set != "taxi" or TAXI_PCA_HANDLING_DECISION != "split_prepost_period":
        return PCA_80_FINAL_PATHS_BY_SET[feature_set]

    if row_metadata is None:
        raise ValueError("Taxi PCA path resolution requires row metadata with pre/post CP labels.")

    if PRE_POST_CP_COLUMN not in row_metadata.columns:
        raise KeyError(f"Expected '{PRE_POST_CP_COLUMN}' in taxi row metadata.")

    unique_periods = pd.Index(row_metadata[PRE_POST_CP_COLUMN].dropna().unique()).tolist()
    if len(unique_periods) != 1:
        raise ValueError(
            "Taxi PCA path resolution expects one policy period at a time; "
            f"found {unique_periods}."
        )

    return TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD[unique_periods[0]]


def load_calibration_metadata(feature_set):
    calibration_ids_path = ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set]
    calibration_metadata_path = ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set]
    row_metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set]

    if not calibration_ids_path.exists():
        raise FileNotFoundError(f"Missing calibration IDs for {feature_set}: {calibration_ids_path}")
    if not calibration_metadata_path.exists():
        raise FileNotFoundError(
            f"Missing calibration metadata for {feature_set}: {calibration_metadata_path}"
        )
    if not row_metadata_path.exists():
        raise FileNotFoundError(f"Missing row metadata for {feature_set}: {row_metadata_path}")

    calibration_ids_df = pd.read_parquet(calibration_ids_path)
    calibration_metadata_df = pd.read_parquet(calibration_metadata_path)

    merged_df = calibration_ids_df.merge(
        calibration_metadata_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )

    missing_default_columns = [
        column for column in DEFAULT_METADATA_COLUMNS if column not in merged_df.columns
    ]
    if missing_default_columns:
        row_metadata_df = pd.read_parquet(
            row_metadata_path,
            columns=[MODEL_ID_COLUMN, *missing_default_columns],
        )
        merged_df = merged_df.merge(
            row_metadata_df,
            on=MODEL_ID_COLUMN,
            how="left",
            validate="one_to_one",
        )

    return merged_df


def get_final_pca_representation_df(feature_set, metadata_df):
    representation_path = get_pca_representation_path(feature_set, row_metadata=metadata_df)
    all_columns = load_parquet_columns(representation_path)
    numeric_columns = [column for column in all_columns if str(column).startswith("PC")]

    if not numeric_columns:
        raise ValueError(
            f"No PCA columns found for {feature_set} at {representation_path}."
        )

    representation_df = pd.read_parquet(
        representation_path,
        columns=[MODEL_ID_COLUMN, *numeric_columns],
    )

    merged_df = metadata_df.merge(
        representation_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )

    return merged_df, numeric_columns, representation_path

In [8]:
# # ---------------------------------------------------------------------
# # Optional cleanup: remove old full-export artifacts before rerunning 3.3.4.5
# # ---------------------------------------------------------------------

# import shutil

# cleanup_targets = [
#     GMM_ANOMALY_EXPORT_MANIFEST_PATH,
#     GMM_FINAL_ROW_LEVEL_DIR,
#     GMM_FINAL_ROW_LEVEL_TEMP_DIR,
#     GMM_FINAL_PREPARED_PANELS_DIR,
# ]

# for target in cleanup_targets:
#     target = Path(target)
#     if target.is_dir():
#         shutil.rmtree(target, ignore_errors=True)
#         print(f"Removed directory: {target}")
#     elif target.exists():
#         target.unlink()
#         print(f"Removed file: {target}")

# GMM_FINAL_ROW_LEVEL_DIR.mkdir(parents=True, exist_ok=True)
# GMM_FINAL_ROW_LEVEL_TEMP_DIR.mkdir(parents=True, exist_ok=True)
# GMM_FINAL_PREPARED_PANELS_DIR.mkdir(parents=True, exist_ok=True)

# print("GMM final-export cleanup complete.")

## 3\.3\.4\.1 Load Shared Anomaly Framework Assets

Load the anomaly\-framework assets inherited from 3\.3\.1 together with the retained PCA representation branches, calibration metadata, and the candidate anomaly outputs already generated by DBSCAN and Isolation Forest\.

Start with a compact inheritance inventory\. Before fitting any Gaussian mixture, we want to confirm that the shared framework assets from 3\.3\.1, the retained DBSCAN outputs from 3\.3\.2, and the retained Isolation Forest outputs from 3\.3\.3 are actually present and available for reuse here\.

In [9]:
# ---------------------------------------------------------------------
# Inventory inherited anomaly-framework assets and retained framework handoff files
# ---------------------------------------------------------------------

gmm_framework_asset_inventory_records = []

for asset_name, asset_path in ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS.items():
    gmm_framework_asset_inventory_records.append(
        {
            "asset_group": "3.3.1 shared framework",
            "asset_name": asset_name,
            "path": str(asset_path),
            "path_exists": asset_path.exists(),
        }
    )

for feature_set in MODEL_FEATURE_SET_NAMES:
    gmm_framework_asset_inventory_records.extend(
        [
            {
                "asset_group": "3.3.1 calibration metadata",
                "asset_name": f"{feature_set}_anomaly_calibration_ids",
                "path": str(ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set]),
                "path_exists": ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set].exists(),
            },
            {
                "asset_group": "3.3.1 calibration metadata",
                "asset_name": f"{feature_set}_anomaly_calibration_metadata",
                "path": str(ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set]),
                "path_exists": ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set].exists(),
            },
            {
                "asset_group": "3.2.x retained PCA branch",
                "asset_name": f"{feature_set}_pca_80pct",
                "path": str(PCA_80_FINAL_PATHS_BY_SET[feature_set]),
                "path_exists": PCA_80_FINAL_PATHS_BY_SET[feature_set].exists(),
            },
        ]
    )

for policy_period, asset_path in TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD.items():
    gmm_framework_asset_inventory_records.append(
        {
            "asset_group": "3.2.2 taxi split PCA",
            "asset_name": f"taxi_{policy_period}_pca_80pct",
            "path": str(asset_path),
            "path_exists": asset_path.exists(),
        }
    )

for asset_name, asset_path in {
    "selected_dbscan_configurations": DBSCAN_SELECTED_CONFIGURATION_PATH,
    "dbscan_anomaly_export_manifest": DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH,
    "dbscan_final_row_level_dir": DBSCAN_FINAL_ROW_LEVEL_DIR,
}.items():
    gmm_framework_asset_inventory_records.append(
        {
            "asset_group": "3.3.2 DBSCAN handoff",
            "asset_name": asset_name,
            "path": str(asset_path),
            "path_exists": asset_path.exists(),
        }
    )

for asset_name, asset_path in {
    "selected_isolation_forest_configurations": IF_SELECTED_CONFIGURATION_PATH,
    "isolation_forest_anomaly_export_manifest": IF_ANOMALY_EXPORT_MANIFEST_PATH,
    "isolation_forest_final_row_level_dir": IF_FINAL_ROW_LEVEL_DIR,
}.items():
    gmm_framework_asset_inventory_records.append(
        {
            "asset_group": "3.3.3 Isolation Forest handoff",
            "asset_name": asset_name,
            "path": str(asset_path),
            "path_exists": asset_path.exists(),
        }
    )

gmm_framework_asset_inventory_df = pd.DataFrame(gmm_framework_asset_inventory_records)

gmm_framework_asset_inventory_summary_df = (
    gmm_framework_asset_inventory_df.groupby("asset_group", dropna=False)
    .agg(
        assets_expected=("asset_name", "size"),
        assets_found=("path_exists", "sum"),
    )
    .reset_index()
)
gmm_framework_asset_inventory_summary_df["status"] = np.where(
    gmm_framework_asset_inventory_summary_df["assets_expected"]
    == gmm_framework_asset_inventory_summary_df["assets_found"],
    "pass",
    "review",
)

display(gmm_framework_asset_inventory_summary_df)
display(gmm_framework_asset_inventory_df)

assert gmm_framework_asset_inventory_df["path_exists"].all(), (
    "One or more inherited anomaly-framework assets is missing."
)

,asset_group,assets_expected,assets_found,status
0,3.2.2 taxi split PCA,2,2,pass
1,3.2.x retained PCA branch,5,5,pass
2,3.3.1 calibration metadata,10,10,pass
3,3.3.1 shared framework,5,5,pass
4,3.3.2 DBSCAN handoff,3,3,pass
5,3.3.3 Isolation Forest handoff,3,3,pass


,asset_group,asset_name,path,path_exists
0,3.3.1 shared framework,shared_representation_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_representation_defaults.csv,True
1,3.3.1 shared framework,shared_calibration_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_calibration_defaults.csv,True
2,3.3.1 shared framework,shared_comparison_universe_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_comparison_universe_defaults.csv,True
3,3.3.1 shared framework,shared_framework_interpretation_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_framework_interpretation_defaults.csv,True
4,3.3.1 shared framework,shared_framework_asset_manifest,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_framework_asset_manifest.csv,True
5,3.3.1 calibration metadata,bus_anomaly_calibration_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/bus_anomaly_calibration_ids.parquet,True
6,3.3.1 calibration metadata,bus_anomaly_calibration_metadata,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/bus_anomaly_calibration_metadata.parquet,True
7,3.2.x retained PCA branch,bus_pca_80pct,/datasets/_deepnote_work/pipeline_data/3.2.1.final_tables/bus_pca_modeling_scores.parquet,True
8,3.3.1 calibration metadata,subway_anomaly_calibration_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/subway_anomaly_calibration_ids.parquet,True
9,3.3.1 calibration metadata,subway_anomaly_calibration_metadata,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/subway_anomaly_calibration_metadata.parquet,True


Findings\. This inventory is just a readiness check, and the intended result is simple: the shared framework tables, retained PCA assets, Taxi split\-PCA files, retained DBSCAN outputs, and retained Isolation Forest outputs should all already exist before GMM calibration begins\. If this block passes, then the probabilistic notebook is starting from a complete inherited framework rather than quietly reconstructing missing pieces\.

Next, load the inherited defaults and prove that the shared calibration panel really lines up with the retained PCA branch\. The main thing to check here is row alignment: every calibration row should resolve cleanly into the retained PCA representation for each modality, with no dropped rows and no missing PCA values in the inherited branch\.

In [10]:
# ---------------------------------------------------------------------
# Load inherited framework defaults and validate row alignment to retained PCA
# ---------------------------------------------------------------------

shared_representation_defaults_df = load_shared_framework_table(
    "shared_representation_defaults"
)
shared_calibration_defaults_df = load_shared_framework_table(
    "shared_calibration_defaults"
)
shared_comparison_universe_defaults_df = load_shared_framework_table(
    "shared_comparison_universe_defaults"
)
shared_framework_interpretation_defaults_df = load_shared_framework_table(
    "shared_framework_interpretation_defaults"
)

dbscan_selected_configurations_df = pd.read_csv(DBSCAN_SELECTED_CONFIGURATION_PATH)
dbscan_anomaly_export_manifest_df = pd.read_csv(DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH)
if_selected_configurations_df = pd.read_csv(IF_SELECTED_CONFIGURATION_PATH)
if_anomaly_export_manifest_df = pd.read_csv(IF_ANOMALY_EXPORT_MANIFEST_PATH)

gmm_input_alignment_records = []

for feature_set in MODEL_FEATURE_SET_NAMES:
    calibration_metadata_df = load_calibration_metadata(feature_set)

    # -------------------------------------------------------------
    # Non-Taxi case: load the retained PCA branch directly
    # -------------------------------------------------------------
    if feature_set != "taxi" or TAXI_PCA_HANDLING_DECISION != "split_prepost_period":
        retained_pca_df, pca_columns, retained_pca_path = get_final_pca_representation_df(
            feature_set,
            calibration_metadata_df,
        )

        rows_in_panel = len(calibration_metadata_df)
        rows_with_any_missing = int(retained_pca_df[pca_columns].isna().any(axis=1).sum())
        total_missing_cells = int(retained_pca_df[pca_columns].isna().sum().sum())
        missing_representation_columns = int(retained_pca_df[pca_columns].isna().sum().gt(0).sum())

        gmm_input_alignment_records.append(
            {
                "feature_set": feature_set,
                "rows_in_panel": rows_in_panel,
                "pca_columns": len(pca_columns),
                "rows_with_any_missing": rows_with_any_missing,
                "total_missing_cells": total_missing_cells,
                "columns_with_missing": missing_representation_columns,
                "retained_pca_path": str(retained_pca_path),
                "status": "pass" if rows_with_any_missing == 0 else "review",
            }
        )

    # -------------------------------------------------------------
    # Taxi case: split the shared calibration panel by policy period,
    # load each PCA branch separately, then stitch back together
    # -------------------------------------------------------------
    else:
        assert PRE_POST_CP_COLUMN in calibration_metadata_df.columns, (
            f"Expected '{PRE_POST_CP_COLUMN}' in Taxi calibration metadata."
        )

        taxi_branch_frames = []
        taxi_branch_paths = []
        taxi_pca_columns = None

        for policy_period in ["pre_cp", "post_cp"]:
            period_metadata_df = (
                calibration_metadata_df.loc[
                    calibration_metadata_df[PRE_POST_CP_COLUMN].eq(policy_period)
                ]
                .copy()
                .reset_index(drop=True)
            )

            if period_metadata_df.empty:
                continue

            period_pca_df, period_pca_columns, period_pca_path = get_final_pca_representation_df(
                feature_set,
                period_metadata_df,
            )

            taxi_branch_frames.append(period_pca_df)
            taxi_branch_paths.append(f"{policy_period}: {period_pca_path}")

            if taxi_pca_columns is None:
                taxi_pca_columns = period_pca_columns
            else:
                assert taxi_pca_columns == period_pca_columns, (
                    "Taxi pre/post PCA branches do not expose the same PCA column set."
                )

        assert taxi_branch_frames, "Taxi PCA validation found no pre/post branches to load."

        retained_pca_df = (
            pd.concat(taxi_branch_frames, ignore_index=True)
            .sort_values(MODEL_ID_COLUMN)
            .reset_index(drop=True)
        )
        taxi_pca_columns = list(taxi_pca_columns)

        rows_in_panel = len(calibration_metadata_df)
        rows_with_any_missing = int(retained_pca_df[taxi_pca_columns].isna().any(axis=1).sum())
        total_missing_cells = int(retained_pca_df[taxi_pca_columns].isna().sum().sum())
        missing_representation_columns = int(
            retained_pca_df[taxi_pca_columns].isna().sum().gt(0).sum()
        )

        gmm_input_alignment_records.append(
            {
                "feature_set": feature_set,
                "rows_in_panel": rows_in_panel,
                "pca_columns": len(taxi_pca_columns),
                "rows_with_any_missing": rows_with_any_missing,
                "total_missing_cells": total_missing_cells,
                "columns_with_missing": missing_representation_columns,
                "retained_pca_path": " | ".join(taxi_branch_paths),
                "status": "pass" if rows_with_any_missing == 0 else "review",
            }
        )

gmm_input_alignment_df = pd.DataFrame(gmm_input_alignment_records)

gmm_framework_handoff_summary_df = pd.DataFrame(
    [
        {
            "dbscan_selected_rows": len(dbscan_selected_configurations_df),
            "dbscan_manifest_rows": len(dbscan_anomaly_export_manifest_df),
            "if_selected_rows": len(if_selected_configurations_df),
            "if_manifest_rows": len(if_anomaly_export_manifest_df),
            "status": (
                "pass"
                if len(dbscan_selected_configurations_df) > 0
                and len(dbscan_anomaly_export_manifest_df) > 0
                and len(if_selected_configurations_df) > 0
                and len(if_anomaly_export_manifest_df) > 0
                else "review"
            ),
        }
    ]
)

display(gmm_input_alignment_df)
display(gmm_framework_handoff_summary_df)

assert gmm_input_alignment_df["status"].eq("pass").all(), (
    "One or more retained PCA branches contains missing representation values within the shared calibration panel."
)
assert gmm_framework_handoff_summary_df["status"].eq("pass").all(), (
    "One or more inherited DBSCAN or Isolation Forest handoff tables is empty."
)

,feature_set,rows_in_panel,pca_columns,rows_with_any_missing,total_missing_cells,columns_with_missing,retained_pca_path,status
0,bus,50000,14,0,0,0,/datasets/_deepnote_work/pipeline_data/3.2.1.final_tables/bus_pca_modeling_scores.parquet,pass
1,subway,50000,11,0,0,0,/datasets/_deepnote_work/pipeline_data/3.2.1.final_tables/subway_pca_modeling_scores.parquet,pass
2,taxi,50000,15,0,0,0,pre_cp: /datasets/_deepnote_work/pipeline_data/3.2.2.final_tables/taxi_pre_cp_pca_80pct_modeling_scores.parquet | post_cp: /datasets/_deepnote_work/pipeline_data/3.2.2.final_tables/taxi_post_cp_pca_80pct_modeling_scores.parquet,pass
3,fhvhv,50000,13,0,0,0,/datasets/_deepnote_work/pipeline_data/3.2.1.final_tables/fhvhv_pca_modeling_scores.parquet,pass
4,multimodal,50000,46,0,0,0,/datasets/_deepnote_work/pipeline_data/3.2.1.final_tables/multimodal_pca_modeling_scores.parquet,pass


,dbscan_selected_rows,dbscan_manifest_rows,if_selected_rows,if_manifest_rows,status
0,5,5,5,5,pass


Findings\. The retained PCA branch is now confirmed as a clean inherited input surface for GMM: every shared calibration row resolves into the retained PCA representation without dropped rows or missing PCA values, and the retained DBSCAN and Isolation Forest handoff tables are both available for later comparison\. That means this notebook can focus on probabilistic calibration rather than spending time rebuilding framework inputs\.

\-\-\-

## 3\.3\.4\.2 Calibrate Candidate GMM Structures

This section tries to determine whether a small set of Gaussian Mixture Model structures can produce believable likelihood surfaces that are stable enough to support anomaly scoring downstream\. That means the main job of this section is screening: eliminate obviously weak model structures, retain plausible ones, and avoid turning model\-selection statistics into an academic detour\.

The section should move from broad structural diagnostics to a compact retained candidate set\. First, verify that each candidate model actually fits cleanly and uses its components sensibly\. Then compare fit\-quality diagnostics like BIC and AIC together with compact likelihood summaries, so we can see whether additional complexity is buying anything meaningful\. Finally, translate those diagnostics into a transparent screening rule that records which component\-count and covariance combinations are worth carrying into tail\-threshold calibration\.

> 📝 Note\. AIC \(Akaike Information Criterion\) and BIC \(Bayesian Information Criterion\) are compact model\-selection diagnostics that balance fit quality against model complexity\. Lower values are better, but they should be read as screening aids rather than as automatic decision rules\. In this notebook, they help us spot structures that are clearly over\- or under\-complex, but the final goal is not to minimize AIC or BIC in the abstract\. The real goal is to retain GMM structures that produce stable, interpretable likelihood tails for anomaly detection\.

### Fit the candidate GMM structure grid and validate basic model behavior

This first part materializes the candidate GMM structure sweep across modalities using the retained PCA branch only\. The goal here is not yet to decide which structures are best; it is to establish the raw evidence we’ll need for that decision\. For each candidate combination of component count and covariance type, we fit a Gaussian Mixture Model within each modality’s shared calibration panel and record the basic diagnostics that tell us whether the fit is even worth taking seriously\.

These helpers make the GMM structure sweep resumable and Taxi\-safe\. They handle candidate naming, loading and saving intermediate outputs, and the one special inheritance case in this notebook: Taxi’s retained PCA panel must be rebuilt by stitching together the split pre\_cp and post\_cp PCA branches before any candidate GMM can be fit against the shared calibration panel\.

In [11]:
# ---------------------------------------------------------------------
# Fit the candidate GMM structure grid and record basic model diagnostics: Helpers
# ---------------------------------------------------------------------

def build_gmm_candidate_id(feature_set, n_components, covariance_type):
    return f"{feature_set}__k{int(n_components)}__cov_{covariance_type}"


def load_gmm_existing_outputs():
    summary_exists = GMM_STRUCTURE_SWEEP_SUMMARY_PATH.exists()
    detail_exists = GMM_STRUCTURE_SWEEP_DETAIL_PATH.exists()

    if summary_exists:
        summary_df = pd.read_parquet(GMM_STRUCTURE_SWEEP_SUMMARY_PATH)
    else:
        summary_df = pd.DataFrame()

    if detail_exists:
        detail_df = pd.read_parquet(GMM_STRUCTURE_SWEEP_DETAIL_PATH)
    else:
        detail_df = pd.DataFrame(
            columns=[
                "candidate_id",
                "feature_set",
                "representation_name",
                "n_components",
                "covariance_type",
                "error_type",
                "error_message",
            ]
        )

    return summary_df, detail_df


def save_gmm_existing_outputs(summary_df, detail_df):
    summary_df.to_parquet(GMM_STRUCTURE_SWEEP_SUMMARY_PATH, index=False)
    detail_df.to_parquet(GMM_STRUCTURE_SWEEP_DETAIL_PATH, index=False)


def load_gmm_retained_pca_panel(feature_set):
    calibration_metadata_df = load_calibration_metadata(feature_set)

    # Non-Taxi case: direct retained PCA load
    if feature_set != "taxi" or TAXI_PCA_HANDLING_DECISION != "split_prepost_period":
        retained_pca_df, pca_columns, retained_pca_path = get_final_pca_representation_df(
            feature_set,
            calibration_metadata_df,
        )
        return retained_pca_df, pca_columns, str(retained_pca_path)

    # Taxi case: split by policy period, load each branch separately, then stitch back
    assert PRE_POST_CP_COLUMN in calibration_metadata_df.columns, (
        f"Expected '{PRE_POST_CP_COLUMN}' in Taxi calibration metadata."
    )

    taxi_branch_frames = []
    taxi_branch_paths = []
    taxi_pca_columns = None

    for policy_period in ["pre_cp", "post_cp"]:
        period_metadata_df = (
            calibration_metadata_df.loc[
                calibration_metadata_df[PRE_POST_CP_COLUMN].eq(policy_period)
            ]
            .copy()
            .reset_index(drop=True)
        )

        if period_metadata_df.empty:
            continue

        period_pca_df, period_pca_columns, period_pca_path = get_final_pca_representation_df(
            feature_set,
            period_metadata_df,
        )

        taxi_branch_frames.append(period_pca_df)
        taxi_branch_paths.append(f"{policy_period}: {period_pca_path}")

        if taxi_pca_columns is None:
            taxi_pca_columns = period_pca_columns
        else:
            assert taxi_pca_columns == period_pca_columns, (
                "Taxi pre/post PCA branches do not expose the same PCA column set."
            )

    assert taxi_branch_frames, "Taxi PCA structure calibration found no pre/post branches to load."

    retained_pca_df = (
        pd.concat(taxi_branch_frames, ignore_index=True)
        .sort_values(MODEL_ID_COLUMN)
        .reset_index(drop=True)
    )

    return retained_pca_df, list(taxi_pca_columns), " | ".join(taxi_branch_paths)



This first pass is only a health check on the candidate GMM grid\. Before comparing model quality, we need to know whether each candidate structure actually fits cleanly, converges, and uses its components in a believable way\. At this stage, the important question is not which structure wins; it is whether the grid is mechanically healthy enough to support model\-selection analysis in the next part\.

In [12]:
# ---------------------------------------------------------------------
# Fit the candidate GMM structure grid and record basic model diagnostics
# ---------------------------------------------------------------------

candidate_grid_records = []
for feature_set in MODEL_FEATURE_SET_NAMES:
    for n_components in GMM_COMPONENT_GRID:
        for covariance_type in GMM_COVARIANCE_TYPE_GRID:
            candidate_grid_records.append(
                {
                    "candidate_id": build_gmm_candidate_id(
                        feature_set,
                        n_components,
                        covariance_type,
                    ),
                    "feature_set": feature_set,
                    "representation_name": "pca_80pct",
                    "n_components": int(n_components),
                    "covariance_type": covariance_type,
                }
            )

candidate_grid_df = pd.DataFrame(candidate_grid_records)
total_candidates = len(candidate_grid_df)

if should_rebuild(REBUILD_GMM_STRUCTURE_CALIBRATION, GMM_STRUCTURE_SWEEP_SUMMARY_PATH):
    gmm_structure_sweep_df = pd.DataFrame()
    gmm_structure_fit_failures_df = pd.DataFrame(
        columns=[
            "candidate_id",
            "feature_set",
            "representation_name",
            "n_components",
            "covariance_type",
            "error_type",
            "error_message",
        ]
    )
else:
    gmm_structure_sweep_df, gmm_structure_fit_failures_df = load_gmm_existing_outputs()

completed_candidate_ids = set()
if not gmm_structure_sweep_df.empty and "candidate_id" in gmm_structure_sweep_df.columns:
    completed_candidate_ids = set(gmm_structure_sweep_df["candidate_id"].dropna().astype(str).tolist())

for iteration_idx, candidate_row in enumerate(candidate_grid_df.itertuples(index=False), start=1):
    candidate_id = candidate_row.candidate_id
    feature_set = candidate_row.feature_set
    representation_name = candidate_row.representation_name
    n_components = int(candidate_row.n_components)
    covariance_type = candidate_row.covariance_type

    if candidate_id in completed_candidate_ids:
        print(f"[{iteration_idx}/{total_candidates}] Skipping completed {candidate_id}")
        continue

    print(
        f"[{iteration_idx}/{total_candidates}] Fitting {candidate_id} "
        f"({feature_set}, k={n_components}, cov={covariance_type})"
    )

    started_at = perf_counter()

    try:
        retained_pca_df, pca_columns, retained_pca_path = load_gmm_retained_pca_panel(feature_set)

        model_input_df = retained_pca_df[[MODEL_ID_COLUMN, *pca_columns]].copy()
        X = model_input_df[pca_columns].to_numpy(dtype=np.float64)

        gmm_model = GaussianMixture(
            n_components=n_components,
            covariance_type=covariance_type,
            random_state=GMM_RANDOM_STATE,
            n_init=GMM_N_INIT,
            max_iter=GMM_MAX_ITER,
            reg_covar=GMM_REG_COVAR,
        )
        gmm_model.fit(X)

        log_likelihood_scores = gmm_model.score_samples(X)
        component_assignments = gmm_model.predict(X)
        component_shares = (
            pd.Series(component_assignments)
            .value_counts(normalize=True, sort=False)
            .sort_index()
        )

        elapsed_seconds = perf_counter() - started_at

        summary_row = {
            "candidate_id": candidate_id,
            "feature_set": feature_set,
            "representation_name": representation_name,
            "rows_evaluated": int(X.shape[0]),
            "pca_dimensions": int(len(pca_columns)),
            "n_components": int(n_components),
            "covariance_type": covariance_type,
            "converged": bool(gmm_model.converged_),
            "n_iter": int(gmm_model.n_iter_),
            "lower_bound": float(gmm_model.lower_bound_),
            "bic": float(gmm_model.bic(X)),
            "aic": float(gmm_model.aic(X)),
            "loglik_q01": float(np.quantile(log_likelihood_scores, 0.01)),
            "loglik_q05": float(np.quantile(log_likelihood_scores, 0.05)),
            "loglik_q50": float(np.quantile(log_likelihood_scores, 0.50)),
            "loglik_mean": float(np.mean(log_likelihood_scores)),
            "loglik_std": float(np.std(log_likelihood_scores)),
            "active_components": int(component_shares.shape[0]),
            "smallest_active_component_share": float(component_shares.min()),
            "largest_component_share": float(component_shares.max()),
            "retained_pca_path": str(retained_pca_path),
            "fit_elapsed_seconds": float(elapsed_seconds),
            "status": "pass",
        }

        gmm_structure_sweep_df = pd.concat(
            [gmm_structure_sweep_df, pd.DataFrame([summary_row])],
            ignore_index=True,
        )
        completed_candidate_ids.add(candidate_id)
        save_gmm_existing_outputs(gmm_structure_sweep_df, gmm_structure_fit_failures_df)

    except Exception as exc:
        elapsed_seconds = perf_counter() - started_at

        failure_summary_row = {
            "candidate_id": candidate_id,
            "feature_set": feature_set,
            "representation_name": representation_name,
            "rows_evaluated": np.nan,
            "pca_dimensions": np.nan,
            "n_components": int(n_components),
            "covariance_type": covariance_type,
            "converged": False,
            "n_iter": np.nan,
            "lower_bound": np.nan,
            "bic": np.nan,
            "aic": np.nan,
            "loglik_q01": np.nan,
            "loglik_q05": np.nan,
            "loglik_q50": np.nan,
            "loglik_mean": np.nan,
            "loglik_std": np.nan,
            "active_components": np.nan,
            "smallest_active_component_share": np.nan,
            "largest_component_share": np.nan,
            "retained_pca_path": None,
            "fit_elapsed_seconds": float(elapsed_seconds),
            "status": "fit_failed",
        }

        failure_detail_row = {
            "candidate_id": candidate_id,
            "feature_set": feature_set,
            "representation_name": representation_name,
            "n_components": int(n_components),
            "covariance_type": covariance_type,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }

        gmm_structure_sweep_df = pd.concat(
            [gmm_structure_sweep_df, pd.DataFrame([failure_summary_row])],
            ignore_index=True,
        )
        gmm_structure_fit_failures_df = pd.concat(
            [gmm_structure_fit_failures_df, pd.DataFrame([failure_detail_row])],
            ignore_index=True,
        )
        completed_candidate_ids.add(candidate_id)
        save_gmm_existing_outputs(gmm_structure_sweep_df, gmm_structure_fit_failures_df)

        print(f"    -> failed: {type(exc).__name__}: {exc}")

gmm_structure_sweep_df = (
    gmm_structure_sweep_df.sort_values(
        ["feature_set", "covariance_type", "n_components", "candidate_id"],
        na_position="last",
    )
    .reset_index(drop=True)
)

save_gmm_existing_outputs(gmm_structure_sweep_df, gmm_structure_fit_failures_df)

gmm_structure_fit_health_preview_df = (
    gmm_structure_sweep_df[
        [
            "feature_set",
            "n_components",
            "covariance_type",
            "rows_evaluated",
            "pca_dimensions",
            "converged",
            "n_iter",
            "active_components",
            "smallest_active_component_share",
            "largest_component_share",
            "fit_elapsed_seconds",
            "status",
        ]
    ]
    .sort_values(["feature_set", "covariance_type", "n_components"])
    .reset_index(drop=True)
)

display(
    gmm_structure_fit_health_preview_df.style.format(
        {
            "smallest_active_component_share": "{:.3f}",
            "largest_component_share": "{:.3f}",
            "fit_elapsed_seconds": "{:.3f}",
        }
    )
)

if not gmm_structure_fit_failures_df.empty:
    display(gmm_structure_fit_failures_df)


[1/40] Skipping completed bus__k1__cov_diag
[2/40] Skipping completed bus__k1__cov_full
[3/40] Skipping completed bus__k2__cov_diag
[4/40] Skipping completed bus__k2__cov_full
[5/40] Skipping completed bus__k3__cov_diag
[6/40] Skipping completed bus__k3__cov_full
[7/40] Skipping completed bus__k5__cov_diag
[8/40] Skipping completed bus__k5__cov_full
[9/40] Skipping completed subway__k1__cov_diag
[10/40] Skipping completed subway__k1__cov_full
[11/40] Skipping completed subway__k2__cov_diag
[12/40] Skipping completed subway__k2__cov_full
[13/40] Skipping completed subway__k3__cov_diag
[14/40] Skipping completed subway__k3__cov_full
[15/40] Skipping completed subway__k5__cov_diag
[16/40] Skipping completed subway__k5__cov_full
[17/40] Skipping completed taxi__k1__cov_diag
[18/40] Skipping completed taxi__k1__cov_full
[19/40] Skipping completed taxi__k2__cov_diag
[20/40] Skipping completed taxi__k2__cov_full
[21/40] Skipping completed taxi__k3__cov_diag
[22/40] Skipping completed taxi__k3

,feature_set,n_components,covariance_type,rows_evaluated,pca_dimensions,converged,n_iter,active_components,smallest_active_component_share,largest_component_share,fit_elapsed_seconds,status
0,bus,1,diag,50000,14,True,2,1,1.000,1.000,3.261,pass
1,bus,2,diag,50000,14,True,13,2,0.200,0.800,7.054,pass
2,bus,3,diag,50000,14,True,30,3,0.124,0.672,13.002,pass
3,bus,5,diag,50000,14,True,19,5,0.028,0.621,13.304,pass
4,bus,1,full,50000,14,True,2,1,1.000,1.000,4.590,pass
5,bus,2,full,50000,14,True,15,2,0.188,0.812,17.670,pass
6,bus,3,full,50000,14,True,18,3,0.168,0.644,26.147,pass
7,bus,5,full,50000,14,True,26,5,0.072,0.528,28.655,pass
8,fhvhv,1,diag,50000,13,True,2,1,1.000,1.000,3.405,pass
9,fhvhv,2,diag,50000,13,True,8,2,0.242,0.758,4.875,pass


Findings\. All 40 candidate structures fit successfully and converged, with no obvious component\-usage failures\. That means the grid is healthy enough to compare on fit quality rather than debug on mechanics\.

> 📝 Note\. Lower BIC and AIC are better, so downward curves indicate improved penalized fit\. Fit time moves in the opposite direction: higher values mean the model is becoming more expensive to use\. The point of this chart is not to choose winners yet, but to show where added structure is clearly helping and where it may be buying diminishing returns\.

Now compress that raw run into a compact health summary\. This validation block checks whether the expected grid was fully produced and whether any candidates show obvious mechanical warning signs such as failed fits, non\-convergence, or suspicious component usage\.

In [13]:
# ---------------------------------------------------------------------
# Validate candidate-grid coverage and summarize basic fit health
# ---------------------------------------------------------------------

expected_candidate_rows = (
    len(MODEL_FEATURE_SET_NAMES)
    * len(GMM_COMPONENT_GRID)
    * len(GMM_COVARIANCE_TYPE_GRID)
)

gmm_structure_grid_validation_df = pd.DataFrame(
    [
        {
            "expected_candidate_rows": expected_candidate_rows,
            "actual_candidate_rows": len(gmm_structure_sweep_df),
            "fit_failed_rows": int(gmm_structure_sweep_df["status"].eq("fit_failed").sum()),
            "converged_rows": int(
                (
                    gmm_structure_sweep_df["status"].eq("pass")
                    & gmm_structure_sweep_df["converged"].fillna(False)
                ).sum()
            ),
            "nonconverged_pass_rows": int(
                (
                    gmm_structure_sweep_df["status"].eq("pass")
                    & ~gmm_structure_sweep_df["converged"].fillna(False)
                ).sum()
            ),
            "status": (
                "pass"
                if len(gmm_structure_sweep_df) == expected_candidate_rows
                else "review"
            ),
        }
    ]
)

gmm_structure_fit_health_df = (
    gmm_structure_sweep_df.assign(
        weak_component_usage=lambda df: (
            df["smallest_active_component_share"].fillna(0).lt(0.01)
        ),
        dominant_component=lambda df: (
            df["largest_component_share"].fillna(0).gt(0.90)
        ),
    )
    .groupby(["feature_set", "covariance_type"], dropna=False)
    .agg(
        candidate_structures=("n_components", "size"),
        converged_structures=("converged", lambda s: int(s.fillna(False).sum())),
        fit_failed_structures=("status", lambda s: int((s == "fit_failed").sum())),
        structures_with_weak_component_usage=("weak_component_usage", lambda s: int(s.sum())),
        structures_with_dominant_component=("dominant_component", lambda s: int(s.sum())),
    )
    .reset_index()
)

display(gmm_structure_grid_validation_df)
display(gmm_structure_fit_health_df)

assert gmm_structure_grid_validation_df["status"].eq("pass").all(), (
    "The GMM candidate structure sweep did not produce the expected number of rows."
)

,expected_candidate_rows,actual_candidate_rows,fit_failed_rows,converged_rows,nonconverged_pass_rows,status
0,40,40,0,40,0,pass


,feature_set,covariance_type,candidate_structures,converged_structures,fit_failed_structures,structures_with_weak_component_usage,structures_with_dominant_component
0,bus,diag,4,4,0,0,1
1,bus,full,4,4,0,0,1
2,fhvhv,diag,4,4,0,0,1
3,fhvhv,full,4,4,0,0,1
4,multimodal,diag,4,4,0,0,1
5,multimodal,full,4,4,0,0,1
6,subway,diag,4,4,0,0,1
7,subway,full,4,4,0,0,1
8,taxi,diag,4,4,0,0,1
9,taxi,full,4,4,0,0,1


Findings\. The structure sweep ran cleanly end to end: all 40 expected candidates were produced, none failed, and all converged\. The grouped health summary also shows no weak component\-usage cases under the current threshold\. The one dominant\-component flag per modality\-covariance pair is just the expected k=1 baseline rather than a pathology\. So the candidate grid passes the basic health screen and is ready for comparative model\-selection analysis\.

### Compare fit\-quality and likelihood behavior across candidate structures

Now that the grid has fit cleanly, we can compare the two actual structure choices in play: how many components the model should use, and whether those components need diagonal or full covariance\. The diagnostics in this part are decision aids rather than knobs\. Lower BIC and AIC indicate better penalized fit, higher lower bound indicates a better fitted likelihood surface, and runtime plus component\-usage behavior tell us what we are paying for that extra flexibility\. The practical question is simple: which structures improve fit enough to be worth carrying into anomaly\-tail calibration?

At this stage, the most important things to compare are:

- BIC and AIC, which summarize penalized fit quality

- lower\_bound and compact likelihood summaries, which tell us how the fitted probability surface is changing

- runtime and component\-usage behavior, which tell us what we are paying for those improvements

The goal is to make the candidate structure tradeoffs readable enough that the next step — screening and retaining plausible structures — can be explicit rather than impressionistic\.

The table below is meant to answer one practical question: within each modality, which structures look strongest on fit quality without becoming gratuitously expensive or mechanically suspicious? The within\-modality ranks help avoid comparing raw BIC or AIC values across very different feature spaces, while the baseline\-improvement columns show how much each candidate buys relative to the simplest local model\.

In [14]:
# ---------------------------------------------------------------------
# Summarize candidate structure behavior in a reader-friendly comparison table
# ---------------------------------------------------------------------

gmm_structure_comparison_df = (
    gmm_structure_sweep_df.loc[
        gmm_structure_sweep_df["status"].eq("pass")
    ].copy()
)

# Lower is better for BIC/AIC/runtime; higher is better for lower_bound.
gmm_structure_comparison_df["bic_rank_within_modality"] = (
    gmm_structure_comparison_df.groupby("feature_set")["bic"]
    .rank(method="dense", ascending=True)
    .astype(int)
)
gmm_structure_comparison_df["aic_rank_within_modality"] = (
    gmm_structure_comparison_df.groupby("feature_set")["aic"]
    .rank(method="dense", ascending=True)
    .astype(int)
)
gmm_structure_comparison_df["lower_bound_rank_within_modality"] = (
    gmm_structure_comparison_df.groupby("feature_set")["lower_bound"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
gmm_structure_comparison_df["fit_time_rank_within_modality"] = (
    gmm_structure_comparison_df.groupby("feature_set")["fit_elapsed_seconds"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

# Compare each candidate to the simplest local baseline: k=1, diag.
baseline_df = (
    gmm_structure_comparison_df.loc[
        gmm_structure_comparison_df["n_components"].eq(1)
        & gmm_structure_comparison_df["covariance_type"].eq("diag"),
        ["feature_set", "bic", "aic", "lower_bound", "fit_elapsed_seconds"],
    ]
    .rename(
        columns={
            "bic": "baseline_bic",
            "aic": "baseline_aic",
            "lower_bound": "baseline_lower_bound",
            "fit_elapsed_seconds": "baseline_fit_elapsed_seconds",
        }
    )
    .reset_index(drop=True)
)

gmm_structure_comparison_df = gmm_structure_comparison_df.merge(
    baseline_df,
    on="feature_set",
    how="left",
    validate="many_to_one",
)

gmm_structure_comparison_df["bic_improvement_vs_baseline"] = (
    gmm_structure_comparison_df["baseline_bic"] - gmm_structure_comparison_df["bic"]
)
gmm_structure_comparison_df["aic_improvement_vs_baseline"] = (
    gmm_structure_comparison_df["baseline_aic"] - gmm_structure_comparison_df["aic"]
)
gmm_structure_comparison_df["lower_bound_gain_vs_baseline"] = (
    gmm_structure_comparison_df["lower_bound"] - gmm_structure_comparison_df["baseline_lower_bound"]
)
gmm_structure_comparison_df["fit_time_multiplier_vs_baseline"] = (
    gmm_structure_comparison_df["fit_elapsed_seconds"]
    / gmm_structure_comparison_df["baseline_fit_elapsed_seconds"]
)

display(
    gmm_structure_comparison_df[
        [
            "feature_set",
            "n_components",
            "covariance_type",
            "converged",
            "active_components",
            "smallest_active_component_share",
            "largest_component_share",
            "bic",
            "aic",
            "lower_bound",
            "loglik_q01",
            "loglik_q50",
            "fit_elapsed_seconds",
            "bic_rank_within_modality",
            "aic_rank_within_modality",
            "lower_bound_rank_within_modality",
            "fit_time_rank_within_modality",
            "bic_improvement_vs_baseline",
            "aic_improvement_vs_baseline",
            "lower_bound_gain_vs_baseline",
            "fit_time_multiplier_vs_baseline",
        ]
    ]
    .sort_values(["feature_set", "covariance_type", "n_components"])
    .style.format(
        {
            "smallest_active_component_share": "{:.3f}",
            "largest_component_share": "{:.3f}",
            "bic": "{:.3f}",
            "aic": "{:.3f}",
            "lower_bound": "{:.3f}",
            "loglik_q01": "{:.3f}",
            "loglik_q50": "{:.3f}",
            "fit_elapsed_seconds": "{:.3f}",
            "bic_improvement_vs_baseline": "{:.3f}",
            "aic_improvement_vs_baseline": "{:.3f}",
            "lower_bound_gain_vs_baseline": "{:.3f}",
            "fit_time_multiplier_vs_baseline": "{:.2f}",
        }
    )
)

,feature_set,n_components,covariance_type,converged,active_components,smallest_active_component_share,largest_component_share,bic,aic,lower_bound,loglik_q01,loglik_q50,fit_elapsed_seconds,bic_rank_within_modality,aic_rank_within_modality,lower_bound_rank_within_modality,fit_time_rank_within_modality,bic_improvement_vs_baseline,aic_improvement_vs_baseline,lower_bound_gain_vs_baseline,fit_time_multiplier_vs_baseline
0,bus,1,diag,True,1,1.000,1.000,2464732.093,2464485.140,-24.644,-58.193,-21.750,3.261,7,8,8,1,0.000,0.000,0.000,1.00
1,bus,2,diag,True,2,0.200,0.800,2251888.456,2251385.729,-22.513,-40.588,-20.508,7.054,6,6,6,3,212843.637,213099.411,2.131,2.16
2,bus,3,diag,True,3,0.124,0.672,2198305.505,2197547.004,-21.974,-39.483,-20.363,13.002,4,4,4,4,266426.588,266938.135,2.670,3.99
3,bus,5,diag,True,5,0.028,0.621,2137031.662,2135761.614,-21.355,-38.222,-19.998,13.304,3,3,3,5,327700.431,328723.525,3.289,4.08
4,bus,1,full,True,1,1.000,1.000,2465103.177,2464053.623,-24.638,-58.073,-21.790,4.590,8,7,7,2,-371.084,431.516,0.006,1.41
5,bus,2,full,True,2,0.188,0.812,2236624.980,2234517.053,-22.341,-50.805,-20.313,17.670,5,5,5,6,228107.113,229968.086,2.303,5.42
6,bus,3,full,True,3,0.168,0.644,2052621.141,2049454.841,-20.488,-45.670,-18.955,26.147,2,2,2,7,412110.952,415030.299,4.157,8.02
7,bus,5,full,True,5,0.072,0.528,1877438.997,1872155.949,-18.710,-37.716,-17.525,28.655,1,1,1,8,587293.097,592329.190,5.934,8.79
8,fhvhv,1,diag,True,1,1.000,1.000,2250329.695,2250100.380,-22.500,-69.532,-19.148,3.405,8,8,8,1,0.000,0.000,0.000,1.00
9,fhvhv,2,diag,True,2,0.242,0.758,1942489.635,1942022.187,-19.419,-42.909,-16.873,4.875,6,6,6,3,307840.059,308078.193,3.081,1.43


Findings\. All 40 candidate GMM structures fit successfully and converged, so there is no immediate evidence that GMM is breaking down on the retained PCA panels\. Across modalities, moving from k=1 toward larger component counts generally improves fit statistics and likelihood behavior, and full covariance almost always outperforms diag on BIC, AIC, and lower\-bound fit quality\. At the same time, that extra flexibility is not free: fit time rises sharply for full covariance, especially in Taxi and Multimodal\. Component usage also looks healthy overall, which means the richer models are not just adding empty or degenerate mixture pieces\. So the practical takeaway from the raw fit table is that GMM is viable here, and the real calibration question is no longer “does it fit?” but “how much structural complexity is worth paying for?”

This visual turns the comparison table into a more intuitive read\. The most useful way to read it is by modality: within each feature set, ask how much BIC, AIC, and lower\_bound improve as structure gets richer, and whether those gains come mainly from adding components, switching from DIAG to FULL, or both\. The rightmost panel keeps the operational cost visible so that we do not confuse “best fit” with “best practical candidate\.”

In [15]:
# ---------------------------------------------------------------------
# Visualize fit-quality and likelihood tradeoffs within each modality
# ---------------------------------------------------------------------

feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
covariance_order = ["diag", "full"]

gmm_tradeoff_plot_df = gmm_structure_comparison_df.copy()
gmm_tradeoff_plot_df["feature_set"] = pd.Categorical(
    gmm_tradeoff_plot_df["feature_set"],
    categories=feature_set_order,
    ordered=True,
)
gmm_tradeoff_plot_df["covariance_type"] = pd.Categorical(
    gmm_tradeoff_plot_df["covariance_type"],
    categories=covariance_order,
    ordered=True,
)

metric_specs = [
    ("bic", "BIC"),
    ("aic", "AIC"),
    ("lower_bound", "Lower bound"),
    ("fit_elapsed_seconds", "Fit time (s)"),
]

fig = make_subplots(
    rows=5,
    cols=4,
    shared_xaxes=True,
    vertical_spacing=0.045,
    horizontal_spacing=0.08,
    subplot_titles=[
        f"{feature_set.upper()} | {metric_label}"
        for feature_set in feature_set_order
        for _, metric_label in metric_specs
    ],
)

color_map = {
    "diag": BRAND_COLORS["dark_teal"],
    "full": BRAND_COLORS["terracotta"],
}

for row_idx, feature_set in enumerate(feature_set_order, start=1):
    feature_df = gmm_tradeoff_plot_df.loc[
        gmm_tradeoff_plot_df["feature_set"].eq(feature_set)
    ].copy()

    for col_idx, (metric_col, metric_label) in enumerate(metric_specs, start=1):
        for covariance_type in covariance_order:
            trace_df = feature_df.loc[
                feature_df["covariance_type"].eq(covariance_type)
            ].sort_values("n_components")

            fig.add_trace(
                go.Scatter(
                    x=trace_df["n_components"],
                    y=trace_df[metric_col],
                    mode="lines+markers",
                    name=covariance_type.upper(),
                    legendgroup=covariance_type,
                    showlegend=(row_idx == 1 and col_idx == 1),
                    line=dict(color=color_map[covariance_type], width=2),
                    marker=dict(size=7),
                    hovertemplate=(
                        "Feature set=" + feature_set
                        + "<br>Covariance=" + covariance_type
                        + "<br>Components=%{x}"
                        + f"<br>{metric_label}=%{{y:.3f}}"
                        + "<extra></extra>"
                    ),
                ),
                row=row_idx,
                col=col_idx,
            )

for row_idx in range(1, 6):
    for col_idx in range(1, 5):
        fig.update_yaxes(
            ticklabelstandoff=4,
            row=row_idx,
            col=col_idx,
        )

for col_idx in range(1, 5):
    fig.update_xaxes(
        title_text="Components",
        tickmode="array",
        tickvals=GMM_COMPONENT_GRID,
        row=5,
        col=col_idx,
    )

fig.update_layout(
    title="GMM Candidate-Structure Tradeoffs by Modality",
    width=960,
    height=640,
    margin=dict(l=70, r=40, t=90, b=60),
    legend=dict(
        title="Covariance",
        orientation="h",
        yanchor="bottom",
        y=1.07,
        xanchor="right",
        x=1.0,
    ),
)

fig.show()

print("GMM candidate-structure tradeoff chart rendered.")

GMM candidate-structure tradeoff chart rendered.


Findings\. The chart makes the tradeoff much easier to see modality by modality\. In every feature set, the orange FULL line drops below the blue DIAG line on both BIC and AIC as component count increases, and its lower bound rises faster as well, which means fuller covariance structure is buying materially better probabilistic fit\. The cost shows up in the rightmost column: FULL fit time climbs much faster than DIAG, with the steepest penalties in Taxi and especially Multimodal\. The overall pattern is pretty consistent: k=1 looks too simple, k=2 and k=3 capture a large share of the fit improvement, and k=5 often keeps improving fit but with a much heavier runtime bill\. So the practical read is that we probably do want to carry some FULL covariance candidates forward, but we should be selective rather than automatically treating the biggest model as the winner\.

### Screen and record GMM structures advancing to tail\-threshold calibration

Now we can turn the structure\-comparison evidence into a simple screening rule\. We are still not choosing the final GMM configuration here; we are just retiring structures that are too simple, too weak on fit, or too costly relative to what they buy us\.  

For this screen, the main ideas are straightforward:

- k=1 is treated as a local baseline rather than a serious anomaly\-scoring candidate

- a candidate should meaningfully use the components it asked for

- a candidate should be locally competitive on fit quality within its modality

- unnecessarily expensive structures should not survive unless they are also clearly strong on fit

The goal is to keep the next section focused on plausible GMM structures rather than dragging the full grid forward\.

Record the candidate GMM structures advancing to tail\-threshold calibration\. At this point, we do not need to keep staring at the full screening table\. The important thing is which structures survived by modality, how many remain, and whether every modality still has at least one credible candidate to carry into tail\-threshold calibration\.

In [16]:
# ---------------------------------------------------------------------
# Record retained GMM structures advancing to tail-threshold calibration
# ---------------------------------------------------------------------

assert GMM_STRUCTURE_SWEEP_SUMMARY_PATH.exists(), (
    "The GMM structure sweep summary was not found. Please run the structure-fit block first."
)

gmm_structure_summary_df = pd.read_parquet(GMM_STRUCTURE_SWEEP_SUMMARY_PATH).copy()

gmm_structure_screen_df = (
    gmm_structure_summary_df.loc[
        gmm_structure_summary_df["status"].eq("pass")
    ]
    .copy()
    .reset_index(drop=True)
)

gmm_structure_screen_df["bic_rank_within_modality"] = (
    gmm_structure_screen_df.groupby("feature_set", dropna=False)["bic"]
    .rank(method="dense", ascending=True)
    .astype(int)
)
gmm_structure_screen_df["aic_rank_within_modality"] = (
    gmm_structure_screen_df.groupby("feature_set", dropna=False)["aic"]
    .rank(method="dense", ascending=True)
    .astype(int)
)
gmm_structure_screen_df["lower_bound_rank_within_modality"] = (
    gmm_structure_screen_df.groupby("feature_set", dropna=False)["lower_bound"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
gmm_structure_screen_df["fit_time_rank_within_modality"] = (
    gmm_structure_screen_df.groupby("feature_set", dropna=False)["fit_elapsed_seconds"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

gmm_structure_screen_df["fit_competitive_metric_count"] = (
    gmm_structure_screen_df[
        [
            "bic_rank_within_modality",
            "aic_rank_within_modality",
            "lower_bound_rank_within_modality",
        ]
    ]
    .le(4)
    .sum(axis=1)
    .astype(int)
)


def classify_gmm_structure_screen(row):
    if int(row["n_components"]) == 1:
        return "retire", "Baseline-only structure"
    if row["covariance_type"] == "full" and int(row["fit_competitive_metric_count"]) == 3:
        return "advance", "Strong local fit competitiveness"
    if row["covariance_type"] == "diag" and int(row["fit_competitive_metric_count"]) == 3:
        return "caution", "Plausible but not clearly preferred"
    if row["covariance_type"] == "full" and int(row["fit_competitive_metric_count"]) == 2:
        return "caution", "Plausible but not clearly preferred"
    return "retire", "Weak fit competitiveness"

screen_labels = gmm_structure_screen_df.apply(classify_gmm_structure_screen, axis=1)
gmm_structure_screen_df["screen_decision"] = [label for label, _ in screen_labels]
gmm_structure_screen_df["screen_rationale"] = [reason for _, reason in screen_labels]

gmm_advancing_structures_df = (
    gmm_structure_screen_df.loc[
        gmm_structure_screen_df["screen_decision"].eq("advance")
    ]
    .copy()
    .sort_values(["feature_set", "n_components", "covariance_type"])
    .reset_index(drop=True)
)

gmm_advancing_structure_inventory_df = (
    gmm_advancing_structures_df.groupby("feature_set", dropna=False)
    .agg(
        retained_structures=("candidate_id", "size"),
        retained_component_counts=(
            "n_components",
            lambda s: ", ".join(map(str, sorted(pd.Index(s).unique()))),
        ),
        retained_covariance_types=(
            "covariance_type",
            lambda s: ", ".join(pd.Index(s).drop_duplicates().tolist()),
        ),
    )
    .reset_index()
)

gmm_advancing_structure_validation_df = pd.DataFrame(
    [
        {
            "feature_sets_with_advancing_structures": int(gmm_advancing_structures_df["feature_set"].nunique()),
            "expected_feature_sets": int(len(MODEL_FEATURE_SET_NAMES)),
            "retained_structure_rows": int(len(gmm_advancing_structures_df)),
            "status": (
                "pass"
                if gmm_advancing_structures_df["feature_set"].nunique()
                == len(MODEL_FEATURE_SET_NAMES)
                else "review"
            ),
        }
    ]
)

if WRITE_334_OUTPUTS:
    if "GMM_STRUCTURE_SCREEN_PATH" in globals():
        gmm_structure_screen_df.to_parquet(GMM_STRUCTURE_SCREEN_PATH, index=False)
    if "GMM_ADVANCING_STRUCTURES_PATH" in globals():
        gmm_advancing_structures_df.to_parquet(
            GMM_ADVANCING_STRUCTURES_PATH,
            index=False,
        )

display(gmm_advancing_structure_inventory_df)
display(gmm_advancing_structure_validation_df)

assert gmm_advancing_structure_validation_df["status"].eq("pass").all(), (
    "One or more modalities has no GMM structure advancing into tail-threshold calibration."
)

,feature_set,retained_structures,retained_component_counts,retained_covariance_types
0,bus,2,"3, 5",full
1,fhvhv,3,"2, 3, 5",full
2,multimodal,3,"2, 3, 5",full
3,subway,3,"2, 3, 5",full
4,taxi,2,"3, 5",full


,feature_sets_with_advancing_structures,expected_feature_sets,retained_structure_rows,status
0,5,5,13,pass


Findings\. The screening step leaves us with a compact retained region rather than a messy spread of candidates\. Every modality still has advancing structures, all of them in the full covariance branch\. Bus and Taxi narrow immediately to k=3 and k=5, while Subway, FHVHV, and Multimodal keep k=2, k=3, and k=5 in play\. So the structure screen has already done meaningful work: it has retired the baseline and weaker candidates, kept the stronger higher\-capacity region, and preserved at least one plausible path forward for every modality\.

Visualize the structure screen as a modality scorecard\. The retained\-structure inventory tells us what survived; this scorecard shows why\. Higher fit\-competitiveness values are better, lower runtime\-rank values are better, and the decision column summarizes whether each candidate advances, stays in caution, or retires\.

In [17]:
# ---------------------------------------------------------------------
# Screen retained GMM structures and prepare the scorecard source table
# ---------------------------------------------------------------------

assert GMM_STRUCTURE_SWEEP_SUMMARY_PATH.exists(), (
    "The GMM structure sweep summary does not exist yet. Please run the structure-fit block first."
)

gmm_structure_summary_df = pd.read_parquet(GMM_STRUCTURE_SWEEP_SUMMARY_PATH).copy()

# Keep only successfully fitted candidates before ranking fit quality.
gmm_structure_summary_df = (
    gmm_structure_summary_df.loc[
        gmm_structure_summary_df["status"].eq("pass")
    ]
    .copy()
    .reset_index(drop=True)
)

gmm_structure_screen_df = gmm_structure_summary_df.copy()

gmm_structure_screen_df["bic_rank_within_modality"] = (
    gmm_structure_screen_df.groupby("feature_set", dropna=False)["bic"]
    .rank(method="dense", ascending=True)
    .astype(int)
)
gmm_structure_screen_df["aic_rank_within_modality"] = (
    gmm_structure_screen_df.groupby("feature_set", dropna=False)["aic"]
    .rank(method="dense", ascending=True)
    .astype(int)
)
gmm_structure_screen_df["lower_bound_rank_within_modality"] = (
    gmm_structure_screen_df.groupby("feature_set", dropna=False)["lower_bound"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
gmm_structure_screen_df["fit_time_rank_within_modality"] = (
    gmm_structure_screen_df.groupby("feature_set", dropna=False)["fit_elapsed_seconds"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

# A candidate is fit-competitive if it stays in the top four on the three fit-quality metrics.
gmm_structure_screen_df["fit_competitive_metric_count"] = (
    gmm_structure_screen_df[
        [
            "bic_rank_within_modality",
            "aic_rank_within_modality",
            "lower_bound_rank_within_modality",
        ]
    ]
    .le(4)
    .sum(axis=1)
    .astype(int)
)


def classify_gmm_structure_screen(row):
    if int(row["n_components"]) == 1:
        return "retire", "Baseline-only structure"
    if row["covariance_type"] == "full" and int(row["fit_competitive_metric_count"]) == 3:
        return "advance", "Strong local fit competitiveness"
    if row["covariance_type"] == "diag" and int(row["fit_competitive_metric_count"]) == 3:
        return "caution", "Plausible but not clearly preferred"
    if row["covariance_type"] == "full" and int(row["fit_competitive_metric_count"]) == 2:
        return "caution", "Plausible but not clearly preferred"
    return "retire", "Weak fit competitiveness"

screen_labels = gmm_structure_screen_df.apply(classify_gmm_structure_screen, axis=1)
gmm_structure_screen_df["screen_decision"] = [label for label, _ in screen_labels]
gmm_structure_screen_df["screen_rationale"] = [reason for _, reason in screen_labels]

gmm_structure_screen_df = (
    gmm_structure_screen_df.sort_values(
        ["feature_set", "screen_decision", "covariance_type", "n_components"],
        ascending=[True, True, False, True],
    )
    .reset_index(drop=True)
)

if WRITE_334_OUTPUTS:
    gmm_structure_screen_df.to_parquet(GMM_STRUCTURE_SCREEN_PATH, index=False)

scorecard_df = gmm_structure_screen_df.copy()
scorecard_df["fit_competitiveness_score"] = scorecard_df["fit_competitive_metric_count"] / 3.0
scorecard_df["runtime_efficiency_score"] = (
    1.0
    - (
        scorecard_df["fit_time_rank_within_modality"] - 1
    )
    /
    (
        scorecard_df.groupby("feature_set", dropna=False)["fit_time_rank_within_modality"].transform("max") - 1
    ).replace(0, 1)
)

scorecard_df["decision_score"] = scorecard_df["screen_decision"].map(
    {"advance": 1.0, "caution": 0.5, "retire": 0.0}
)
scorecard_df["row_label"] = (
    "k=" + scorecard_df["n_components"].astype(int).astype(str)
    + " | " + scorecard_df["covariance_type"].str.upper()
)

display(
    gmm_structure_screen_df[
        [
            "feature_set",
            "n_components",
            "covariance_type",
            "fit_competitive_metric_count",
            "bic_rank_within_modality",
            "aic_rank_within_modality",
            "lower_bound_rank_within_modality",
            "fit_time_rank_within_modality",
            "screen_decision",
            "screen_rationale",
        ]
    ].style.format()
)

,feature_set,n_components,covariance_type,fit_competitive_metric_count,bic_rank_within_modality,aic_rank_within_modality,lower_bound_rank_within_modality,fit_time_rank_within_modality,screen_decision,screen_rationale
0,bus,3,full,3,2,2,2,7,advance,Strong local fit competitiveness
1,bus,5,full,3,1,1,1,8,advance,Strong local fit competitiveness
2,bus,3,diag,3,4,4,4,4,caution,Plausible but not clearly preferred
3,bus,5,diag,3,3,3,3,5,caution,Plausible but not clearly preferred
4,bus,1,full,0,8,7,7,2,retire,Baseline-only structure
5,bus,2,full,0,5,5,5,6,retire,Weak fit competitiveness
6,bus,1,diag,0,7,8,8,1,retire,Baseline-only structure
7,bus,2,diag,0,6,6,6,3,retire,Weak fit competitiveness
8,fhvhv,2,full,3,4,4,4,6,advance,Strong local fit competitiveness
9,fhvhv,3,full,3,2,2,2,7,advance,Strong local fit competitiveness


Findings\. The scorecard makes the screening pattern much easier to read than the raw tables\. Across all five modalities, the strongest candidates sit in the full covariance branch, while diag mostly falls into caution or retire territory\. The k=1 baselines are retired everywhere, which is what we would want from a structure that is too simple to support useful anomaly scoring\. Bus and Taxi already narrow to full \| k=3 and full \| k=5, while Subway, FHVHV, and Multimodal still keep full \| k=2 alive alongside k=3 and k=5\. So the practical takeaway is that the screen has reduced the grid to a compact, higher\-capacity structure region that is strong enough to carry into tail\-threshold calibration without dragging obviously weak candidates forward\.

### Section Summary

The candidate GMM structure grid proved mechanically healthy: all 40 structures fit successfully, all converged, and none showed obvious component\-usage failure\. Once we moved from fit health to model comparison, the main pattern was consistent across modalities: full covariance materially outperformed diag on BIC, AIC, and lower\-bound fit quality, while larger component counts generally improved fit at the cost of longer runtime\. The structure screen then narrowed the grid to a compact retained region made up entirely of full covariance candidates\. Bus and Taxi retain k=3 and k=5, while Subway, FHVHV, and Multimodal retain k=2, k=3, and k=5\. That leaves the next section with a much cleaner question: not which broad GMM structure family to use, but which likelihood\-tail threshold produces anomaly behavior that is sparse, stable, and interpretable within this retained probabilistic structure region\.

\-\-\-

## 3\.3\.4\.3 Evaluate Likelihood\-Tail Thresholds

With the candidate GMM structures narrowed, the center of gravity now shifts from model form to anomaly definition\. GMM does not label anomalies directly; it gives us likelihood scores, and we have to decide how deep into the low\-likelihood tail to cut\. This section calibrates that cut by testing a small threshold grid and asking which retained tail levels produce anomaly surfaces that are sparse enough to be interpretable, distinct enough from the bulk of observations, and stable enough to carry forward into downstream framework comparison\.

### Materialize likelihood\-tail candidates across the retained GMM structures

Start by materializing the candidate anomaly surfaces\. Each retained GMM structure is re\-fit on its retained PCA panel, then each candidate tail threshold is applied to the resulting likelihood scores\. The purpose of this subsection is not interpretation yet; it is simply to create the candidate row\-level anomaly outputs and confirm that the threshold grid has been built cleanly\.

Materialize the candidate likelihood\-tail surfaces and write compact summary outputs\. The key output here is simple: for each retained structure and each candidate threshold, we want a row\-level anomaly file plus a small summary table showing how many rows were flagged\.

In [18]:
# ---------------------------------------------------------------------
# Materialize likelihood-tail candidates across the retained GMM structures
# ---------------------------------------------------------------------

GMM_TAIL_ROW_LEVEL_DIR = INTERMEDIATE_334_DIR / "gmm_tail_row_level_parts"
GMM_TAIL_ROW_LEVEL_DIR.mkdir(parents=True, exist_ok=True)


def build_gmm_tail_candidate_id(feature_set, n_components, covariance_type, tail_threshold):
    threshold_label = str(tail_threshold).replace(".", "p")
    return f"{feature_set}__k{int(n_components)}__cov_{covariance_type}__tail_{threshold_label}"


def get_gmm_tail_row_level_path(feature_set, n_components, covariance_type, tail_threshold):
    tail_candidate_id = build_gmm_tail_candidate_id(
        feature_set,
        n_components,
        covariance_type,
        tail_threshold,
    )
    return GMM_TAIL_ROW_LEVEL_DIR / f"{tail_candidate_id}.parquet"


assert GMM_ADVANCING_STRUCTURES_PATH.exists(), (
    "The retained GMM structures table was not found. Please run the retained-structures block first."
)

gmm_advancing_structures_df = pd.read_parquet(GMM_ADVANCING_STRUCTURES_PATH).copy()

# Rebuild from scratch only when explicitly requested.
if should_rebuild(REBUILD_GMM_TAIL_CALIBRATION, GMM_TAIL_SWEEP_SUMMARY_PATH):
    gmm_tail_sweep_summary_df = pd.DataFrame()
else:
    if GMM_TAIL_SWEEP_SUMMARY_PATH.exists():
        gmm_tail_sweep_summary_df = pd.read_parquet(GMM_TAIL_SWEEP_SUMMARY_PATH)
    else:
        gmm_tail_sweep_summary_df = pd.DataFrame()

completed_tail_candidate_ids = set()
if not gmm_tail_sweep_summary_df.empty and "tail_candidate_id" in gmm_tail_sweep_summary_df.columns:
    completed_tail_candidate_ids = set(
        gmm_tail_sweep_summary_df["tail_candidate_id"].dropna().astype(str).tolist()
    )

total_tail_candidates = len(gmm_advancing_structures_df) * len(GMM_TAIL_THRESHOLD_GRID)
completed_before_run = len(completed_tail_candidate_ids)

print(
    f"Tail materialization status: {completed_before_run}/{total_tail_candidates} "
    f"candidates already available."
)

# Fast exit when everything is already cached.
if completed_before_run == total_tail_candidates and not REBUILD_GMM_TAIL_CALIBRATION:
    print("All tail candidates already materialized; loading cached summary only.")
else:
    iteration_idx = 0

    for retained_row in gmm_advancing_structures_df.itertuples(index=False):
        feature_set = retained_row.feature_set
        representation_name = retained_row.representation_name
        n_components = int(retained_row.n_components)
        covariance_type = retained_row.covariance_type

        structure_tail_ids = [
            build_gmm_tail_candidate_id(
                feature_set,
                n_components,
                covariance_type,
                tail_threshold,
            )
            for tail_threshold in GMM_TAIL_THRESHOLD_GRID
        ]

        completed_structure_tail_ids = [
            tail_candidate_id
            for tail_candidate_id in structure_tail_ids
            if tail_candidate_id in completed_tail_candidate_ids
        ]

        # If every tail threshold for this retained structure is already available,
        # skip the expensive PCA load + GMM refit entirely.
        if len(completed_structure_tail_ids) == len(structure_tail_ids):
            iteration_idx += len(structure_tail_ids)
            print(
                f"[{iteration_idx}/{total_tail_candidates}] "
                f"Skipping fully completed structure "
                f"{feature_set} | k={n_components} | {covariance_type}"
            )
            continue

        retained_pca_df, pca_columns, retained_pca_path = load_gmm_retained_pca_panel(feature_set)
        X = retained_pca_df[pca_columns].to_numpy(dtype=np.float64)

        # Fit each retained structure once, then apply all requested tail thresholds.
        gmm_model = GaussianMixture(
            n_components=n_components,
            covariance_type=covariance_type,
            random_state=GMM_RANDOM_STATE,
            n_init=GMM_N_INIT,
            max_iter=GMM_MAX_ITER,
            reg_covar=GMM_REG_COVAR,
        )
        gmm_model.fit(X)
        log_likelihood_scores = gmm_model.score_samples(X)

        for tail_threshold in GMM_TAIL_THRESHOLD_GRID:
            iteration_idx += 1

            tail_candidate_id = build_gmm_tail_candidate_id(
                feature_set,
                n_components,
                covariance_type,
                tail_threshold,
            )

            if tail_candidate_id in completed_tail_candidate_ids:
                continue

            print(f"[{iteration_idx}/{total_tail_candidates}] Materializing {tail_candidate_id}")

            tail_cutoff = float(np.quantile(log_likelihood_scores, tail_threshold))
            anomaly_flag = (log_likelihood_scores <= tail_cutoff).astype(int)

            row_level_output_path = get_gmm_tail_row_level_path(
                feature_set,
                n_components,
                covariance_type,
                tail_threshold,
            )

            row_level_df = retained_pca_df[
                [
                    MODEL_ID_COLUMN,
                    DATE_COLUMN,
                    TEMPORAL_BUCKET_COLUMN,
                    PRE_POST_CP_COLUMN,
                    ZONE_ID_COLUMN,
                    ZONE_COLUMN,
                    BOROUGH_COLUMN,
                    POLICY_GEOGRAPHY_COLUMN,
                ]
            ].copy()

            row_level_df["feature_set"] = feature_set
            row_level_df["representation_name"] = representation_name
            row_level_df["n_components"] = n_components
            row_level_df["covariance_type"] = covariance_type
            row_level_df["tail_threshold"] = float(tail_threshold)
            row_level_df["gmm_log_likelihood"] = log_likelihood_scores
            row_level_df["gmm_tail_cutoff"] = tail_cutoff
            row_level_df["gmm_anomaly_flag"] = anomaly_flag

            if WRITE_334_OUTPUTS:
                row_level_df.to_parquet(row_level_output_path, index=False)

            anomaly_scores = log_likelihood_scores[anomaly_flag == 1]
            normal_scores = log_likelihood_scores[anomaly_flag == 0]

            summary_row = {
                "tail_candidate_id": tail_candidate_id,
                "feature_set": feature_set,
                "representation_name": representation_name,
                "n_components": n_components,
                "covariance_type": covariance_type,
                "tail_threshold": float(tail_threshold),
                "rows_evaluated": int(len(row_level_df)),
                "anomaly_rows": int(anomaly_flag.sum()),
                "anomaly_share": float(anomaly_flag.mean()),
                "anomaly_score_q95": float(np.quantile(anomaly_scores, 0.95)),
                "normal_score_q05": float(np.quantile(normal_scores, 0.05)),
                "anomaly_score_median": float(np.median(anomaly_scores)),
                "normal_score_median": float(np.median(normal_scores)),
                "tail_separation_gap": float(
                    np.quantile(normal_scores, 0.05) - np.quantile(anomaly_scores, 0.95)
                ),
                "median_separation": float(
                    np.median(normal_scores) - np.median(anomaly_scores)
                ),
                "row_level_output_path": str(row_level_output_path),
                "retained_pca_path": str(retained_pca_path),
                "status": "pass",
            }

            gmm_tail_sweep_summary_df = pd.concat(
                [gmm_tail_sweep_summary_df, pd.DataFrame([summary_row])],
                ignore_index=True,
            )
            completed_tail_candidate_ids.add(tail_candidate_id)

            if WRITE_334_OUTPUTS:
                gmm_tail_sweep_summary_df.to_parquet(
                    GMM_TAIL_SWEEP_SUMMARY_PATH,
                    index=False,
                )

gmm_tail_sweep_summary_df = (
    gmm_tail_sweep_summary_df.sort_values(
        ["feature_set", "n_components", "covariance_type", "tail_threshold"]
    )
    .reset_index(drop=True)
)

if WRITE_334_OUTPUTS:
    gmm_tail_sweep_summary_df.to_parquet(
        GMM_TAIL_SWEEP_SUMMARY_PATH,
        index=False,
    )

gmm_tail_materialization_inventory_df = (
    gmm_tail_sweep_summary_df.groupby("feature_set", dropna=False)
    .agg(
        retained_structures=(
            "tail_candidate_id",
            lambda s: s.str.replace(r"__tail_.*$", "", regex=True).nunique(),
        ),
        thresholds_materialized=("tail_candidate_id", "size"),
        min_anomaly_share=("anomaly_share", "min"),
        max_anomaly_share=("anomaly_share", "max"),
    )
    .reset_index()
)

display(
    gmm_tail_materialization_inventory_df.style.format(
        {
            "min_anomaly_share": "{:.3f}",
            "max_anomaly_share": "{:.3f}",
        }
    )
)

Tail materialization status: 52/52 candidates already available.
All tail candidates already materialized; loading cached summary only.


,feature_set,retained_structures,thresholds_materialized,min_anomaly_share,max_anomaly_share
0,bus,2,8,0.005,0.050
1,fhvhv,3,12,0.005,0.050
2,multimodal,3,12,0.005,0.050
3,subway,3,12,0.005,0.050
4,taxi,2,8,0.005,0.050


Validate that the full tail\-threshold grid was materialized\. If this block passes, then every retained structure has a full set of threshold candidates and we can move into actual threshold comparison instead of worrying about missing files or partial outputs\.

In [19]:
# ---------------------------------------------------------------------
# Validate likelihood-tail candidate coverage
# ---------------------------------------------------------------------

expected_tail_candidate_rows = len(gmm_advancing_structures_df) * len(GMM_TAIL_THRESHOLD_GRID)

gmm_tail_candidate_validation_df = pd.DataFrame(
    [
        {
            "expected_tail_candidate_rows": expected_tail_candidate_rows,
            "actual_tail_candidate_rows": len(gmm_tail_sweep_summary_df),
            "feature_sets_covered": gmm_tail_sweep_summary_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "status": (
                "pass"
                if (
                    len(gmm_tail_sweep_summary_df) == expected_tail_candidate_rows
                    and gmm_tail_sweep_summary_df["feature_set"].nunique() == len(MODEL_FEATURE_SET_NAMES)
                )
                else "review"
            ),
        }
    ]
)

display(gmm_tail_candidate_validation_df)

assert gmm_tail_candidate_validation_df["status"].eq("pass").all(), (
    "The likelihood-tail candidate grid is incomplete."
)

,expected_tail_candidate_rows,actual_tail_candidate_rows,feature_sets_covered,expected_feature_sets,status
0,52,52,5,5,pass


Findings\. The full retained\-structure\-by\-threshold grid was materialized successfully, so the next step can focus entirely on threshold behavior rather than missing candidate surfaces\.

### Compare anomaly prevalence and tail\-separation behavior across candidate thresholds

Now that the candidate tail surfaces exist, we can compare what each threshold actually does\. Two things matter most here\. First, anomaly share tells us the size of the anomaly budget: how much of the panel we are calling anomalous\. Second, the tail\-separation metrics tell us whether the anomaly tail still looks distinct from the bulk of normal rows\. In practical terms, we want thresholds that are sparse enough to be interpretable and still preserve a visible low\-likelihood tail rather than smearing anomalies too far into the middle of the distribution\.

Compare anomaly prevalence across candidate thresholds\. Anomaly share is the simplest threshold diagnostic: it tells us how much of the panel is being pulled into the anomaly tail\. In practical terms, this is the anomaly budget\. Lower thresholds should produce sparser anomaly surfaces, while higher thresholds should spend that budget more aggressively\.

In [20]:
# ---------------------------------------------------------------------
# Compare anomaly prevalence across candidate thresholds
# ---------------------------------------------------------------------

gmm_tail_prevalence_plot_df = gmm_tail_sweep_summary_df.copy()
gmm_tail_prevalence_plot_df["structure_label"] = (
    "k="
    + gmm_tail_prevalence_plot_df["n_components"].astype(int).astype(str)
    + " | "
    + gmm_tail_prevalence_plot_df["covariance_type"].str.upper()
)

gmm_tail_prevalence_inventory_df = (
    gmm_tail_prevalence_plot_df.groupby(["feature_set", "structure_label"], dropna=False)
    .agg(
        min_anomaly_share=("anomaly_share", "min"),
        max_anomaly_share=("anomaly_share", "max"),
    )
    .reset_index()
    .sort_values(["feature_set", "structure_label"])
    .reset_index(drop=True)
)

display(
    gmm_tail_prevalence_inventory_df.style.format(
        {
            "min_anomaly_share": "{:.3f}",
            "max_anomaly_share": "{:.3f}",
        }
    )
)

feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]

fig = px.line(
    gmm_tail_prevalence_plot_df.sort_values(["feature_set", "structure_label", "tail_threshold"]),
    x="tail_threshold",
    y="anomaly_share",
    color="structure_label",
    facet_col="feature_set",
    facet_col_wrap=2,
    category_orders={"feature_set": feature_set_order},
    markers=True,
    labels={
        "tail_threshold": "Tail threshold",
        "anomaly_share": "Anomaly share",
        "structure_label": "Structure",
        "feature_set": "Feature set",
    },
    title="GMM Anomaly Prevalence Across Candidate Tail Thresholds",
)

fig.update_yaxes(tickformat=".1%", ticklabelstandoff=4)
fig.update_layout(
    width=960,
    height=640,
    margin=dict(l=60, r=40, t=80, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=-0.20, xanchor="center", x=0.5),
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].upper()))
fig.show()

print("GMM prevalence calibration chart rendered.")

,feature_set,structure_label,min_anomaly_share,max_anomaly_share
0,bus,k=3 | FULL,0.005,0.050
1,bus,k=5 | FULL,0.005,0.050
2,fhvhv,k=2 | FULL,0.005,0.050
3,fhvhv,k=3 | FULL,0.005,0.050
4,fhvhv,k=5 | FULL,0.005,0.050
5,multimodal,k=2 | FULL,0.005,0.050
6,multimodal,k=3 | FULL,0.005,0.050
7,multimodal,k=5 | FULL,0.005,0.050
8,subway,k=2 | FULL,0.005,0.050
9,subway,k=3 | FULL,0.005,0.050


GMM prevalence calibration chart rendered.


Findings\. This block is mostly a calibration sanity check, and it behaves exactly as it should\. Across every modality and retained GMM structure, anomaly share tracks the requested tail threshold almost one\-for\-one: 0\.005, 0\.010, 0\.020, and 0\.050 produce roughly 0\.5%, 1\.0%, 2\.0%, and 5\.0% anomaly surfaces\. That means the tail\-threshold knob is functioning cleanly as an anomaly\-budget control\. Practically, this block does not help us choose among structures or thresholds by itself; it just confirms that larger thresholds spend more anomaly budget, so the real decision has to come from how well the anomaly tail stays separated from normal observations\.

Compare tail\-separation behavior across candidate thresholds\. The two metrics here measure whether the anomaly tail is still meaningfully distinct from the bulk of normal rows\. tail\_separation\_gap asks whether the least\-extreme anomalies still sit below the worst normal rows; larger positive values mean cleaner tail separation\. median\_separation is a broader check on whether anomaly scores and normal scores are still meaningfully apart in the middle, not just at the edge\. In practical terms, we want thresholds that do not blur the anomaly boundary too aggressively\.

In [21]:
# ---------------------------------------------------------------------
# Compare tail-separation behavior across candidate thresholds
# ---------------------------------------------------------------------

gmm_tail_separation_plot_df = gmm_tail_sweep_summary_df.copy()
gmm_tail_separation_plot_df["structure_label"] = (
    "k="
    + gmm_tail_separation_plot_df["n_components"].astype(int).astype(str)
    + " | "
    + gmm_tail_separation_plot_df["covariance_type"].str.upper()
)

gmm_tail_separation_inventory_df = (
    gmm_tail_separation_plot_df.groupby(["feature_set", "structure_label"], dropna=False)
    .agg(
        min_tail_gap=("tail_separation_gap", "min"),
        max_tail_gap=("tail_separation_gap", "max"),
        min_median_gap=("median_separation", "min"),
        max_median_gap=("median_separation", "max"),
    )
    .reset_index()
    .sort_values(["feature_set", "structure_label"])
    .reset_index(drop=True)
)

display(
    gmm_tail_separation_inventory_df.style.format(
        {
            "min_tail_gap": "{:.3f}",
            "max_tail_gap": "{:.3f}",
            "min_median_gap": "{:.3f}",
            "max_median_gap": "{:.3f}",
        }
    )
)

metric_specs = [
    ("tail_separation_gap", "Tail separation gap"),
    ("median_separation", "Median separation"),
]

fig = make_subplots(
    rows=5,
    cols=2,
    shared_xaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.08,
    subplot_titles=[
        f"{feature_set.upper()} | {metric_label}"
        for feature_set in feature_set_order
        for _, metric_label in metric_specs
    ],
)

color_map = {
    label: color
    for label, color in zip(
        sorted(gmm_tail_separation_plot_df["structure_label"].unique()),
        px.colors.qualitative.Set2,
    )
}

shown_legend_labels = set()

for row_idx, feature_set in enumerate(feature_set_order, start=1):
    feature_df = gmm_tail_separation_plot_df.loc[
        gmm_tail_separation_plot_df["feature_set"].eq(feature_set)
    ].copy()

    for col_idx, (metric_col, metric_label) in enumerate(metric_specs, start=1):
        for structure_label in sorted(feature_df["structure_label"].unique()):
            trace_df = feature_df.loc[
                feature_df["structure_label"].eq(structure_label)
            ].sort_values("tail_threshold")

            showlegend = structure_label not in shown_legend_labels
            shown_legend_labels.add(structure_label)

            fig.add_trace(
                go.Scatter(
                    x=trace_df["tail_threshold"],
                    y=trace_df[metric_col],
                    mode="lines+markers",
                    name=structure_label,
                    legendgroup=structure_label,
                    showlegend=showlegend,
                    line=dict(color=color_map[structure_label], width=2),
                    marker=dict(size=6),
                    hovertemplate=(
                        "Feature set=" + feature_set
                        + "<br>Structure=" + structure_label
                        + "<br>Tail threshold=%{x:.3f}"
                        + f"<br>{metric_label}=%{{y:.3f}}"
                        + "<extra></extra>"
                    ),
                ),
                row=row_idx,
                col=col_idx,
            )

for row_idx in range(1, 6):
    for col_idx in range(1, 3):
        fig.update_yaxes(ticklabelstandoff=4, row=row_idx, col=col_idx)

for col_idx in range(1, 3):
    fig.update_xaxes(title_text="Tail threshold", row=5, col=col_idx)

fig.update_layout(
    title="GMM Tail-Separation Behavior Across Candidate Thresholds",
    width=960,
    height=650,
    margin=dict(l=65, r=40, t=90, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="right", x=1.0),
)

fig.show()

print("GMM tail-separation chart rendered.")

,feature_set,structure_label,min_tail_gap,max_tail_gap,min_median_gap,max_median_gap
0,bus,k=3 | FULL,3.230,39.664,15.006,80.333
1,bus,k=5 | FULL,2.581,20.951,14.040,43.506
2,fhvhv,k=2 | FULL,4.356,59.431,24.118,110.933
3,fhvhv,k=3 | FULL,2.783,25.163,18.860,52.014
4,fhvhv,k=5 | FULL,2.957,16.353,18.198,38.964
5,multimodal,k=2 | FULL,17.285,122.257,70.317,209.334
6,multimodal,k=3 | FULL,16.258,106.351,62.751,188.749
7,multimodal,k=5 | FULL,10.077,50.480,53.090,121.988
8,subway,k=2 | FULL,2.496,33.880,30.128,74.218
9,subway,k=3 | FULL,3.414,36.673,26.010,70.565


GMM tail-separation chart rendered.


Findings\. The chart and summary table tell a consistent story: as the tail threshold gets looser, both tail\_separation\_gap and median\_separation fall in every modality and every retained structure\. In practical terms, that means broader anomaly cuts make the flagged rows look less distinctly unusual relative to the normal rows\. Here, a “stronger separation” or “clearer anomaly boundary” simply means a bigger score gap between the anomaly tail and the bulk of normal observations; bigger gaps are cleaner, smaller gaps are blurrier\. The structure comparison is also clearer now\. In Bus, k=3 \| FULL stays above k=5 \| FULL on both metrics; in FHVHV and Multimodal, k=2 \| FULL preserves the largest gaps; in Subway, k=2 \| FULL and k=3 \| FULL both stay well above k=5 \| FULL; and in Taxi, the two retained structures behave similarly\. So the practical takeaway is that lower thresholds give us cleaner, more defensible anomaly cuts, while higher thresholds start to dilute the tail with less clearly unusual rows\.

### Screen and record thresholds advancing beyond calibration

At this point, we have enough evidence to stop treating every threshold as equally plausible\. The purpose of this subsection is to apply a simple screening rule: keep thresholds that remain sparse enough to be usable and still preserve a reasonably distinct low\-likelihood tail, caution thresholds that are still arguable but weaker, and retire thresholds that are already too diffuse or too blurred to be attractive anomaly candidates\.

The single\-modality scorecards apply the same threshold screen across Bus, Subway, Taxi, and FHVHV\. Each panel weighs three practical checks at once: anomaly share should remain reasonably sparse, tail\_separation\_gap should stay positive, and median\_separation should remain competitive within the retained structure\. The goal is to make the threshold tradeoff readable at a glance for the four single\-modality branches rather than forcing the comparison back into long calibration tables\.

In [22]:
# ---------------------------------------------------------------------
# Score candidate thresholds with a transparent calibration rule:
# Single-modality scorecards
# ---------------------------------------------------------------------

gmm_tail_screen_df = pd.read_parquet(GMM_TAIL_SWEEP_SUMMARY_PATH).copy()

gmm_tail_screen_df["structure_label"] = (
    "k="
    + gmm_tail_screen_df["n_components"].astype(int).astype(str)
    + " | "
    + gmm_tail_screen_df["covariance_type"].str.upper()
)

gmm_tail_screen_df["tail_gap_rank_within_structure"] = (
    gmm_tail_screen_df.groupby(["feature_set", "structure_label"])["tail_separation_gap"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

gmm_tail_screen_df["median_gap_rank_within_structure"] = (
    gmm_tail_screen_df.groupby(["feature_set", "structure_label"])["median_separation"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

gmm_tail_screen_df["sparsity_score"] = np.where(
    gmm_tail_screen_df["tail_threshold"] <= 0.020,
    1.0,
    np.where(gmm_tail_screen_df["tail_threshold"] <= 0.050, 0.5, 0.0),
)

gmm_tail_screen_df["tail_gap_score"] = np.where(
    gmm_tail_screen_df["tail_separation_gap"] > 0,
    1.0,
    0.0,
)

gmm_tail_screen_df["median_gap_score"] = np.where(
    gmm_tail_screen_df["median_gap_rank_within_structure"] <= 2,
    1.0,
    np.where(gmm_tail_screen_df["median_gap_rank_within_structure"] == 3, 0.5, 0.0),
)

def classify_gmm_tail_threshold(row):
    if row["tail_threshold"] == max(GMM_TAIL_THRESHOLD_GRID):
        return "retire", "Tail cut is too diffuse"
    if row["tail_separation_gap"] <= 0:
        return "retire", "Tail boundary is no longer clean"
    if (
        row["tail_threshold"] <= 0.020
        and row["tail_gap_rank_within_structure"] <= 2
        and row["median_gap_rank_within_structure"] <= 2
    ):
        return "advance", "Sparse threshold with strong tail separation"
    return "caution", "Plausible threshold but less cleanly separated"

gmm_tail_screen_df[["screen_decision", "screen_rationale"]] = (
    gmm_tail_screen_df.apply(
        lambda row: pd.Series(classify_gmm_tail_threshold(row)),
        axis=1,
    )
)

gmm_tail_screen_df["decision_score"] = gmm_tail_screen_df["screen_decision"].map(
    {"advance": 1.0, "caution": 0.5, "retire": 0.0}
)

single_modality_order = ["bus", "subway", "taxi", "fhvhv"]
metric_order = ["Anomaly<br>share", "Tail gap", "Median<br>gap", "Decision"]

fig = make_subplots(
    rows=2,
    cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.07,
    horizontal_spacing=0.15,
    subplot_titles=[feature_set.upper() for feature_set in single_modality_order],
)

position_map = {
    "bus": (1, 1),
    "subway": (1, 2),
    "taxi": (2, 1),
    "fhvhv": (2, 2),
}

brand_screen_scale = [
    [0.00, BRAND_COLORS["terracotta"]],
    [0.50, BRAND_COLORS["pale_peach"]],
    [0.75, BRAND_COLORS["seafoam"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

single_modality_df = gmm_tail_screen_df.loc[
    gmm_tail_screen_df["feature_set"].isin(single_modality_order)
].copy()

anomaly_share_max = max(single_modality_df["anomaly_share"].max(), 1e-9)
tail_gap_max = max(single_modality_df["tail_separation_gap"].max(), 1e-9)
median_gap_max = max(single_modality_df["median_separation"].max(), 1e-9)

row_orders_by_feature = {}

for feature_set in single_modality_order:
    row_idx, col_idx = position_map[feature_set]

    feature_df = single_modality_df.loc[
        single_modality_df["feature_set"].eq(feature_set)
    ].copy()

    feature_df = feature_df.sort_values(
        ["n_components", "tail_threshold"],
        ascending=[False, False],
    ).reset_index(drop=True)

    feature_df["row_label"] = (
        "k="
        + feature_df["n_components"].astype(int).astype(str)
        + " | "
        + feature_df["tail_threshold"].map("{:.3f}".format)
    )

    row_order = feature_df["row_label"].tolist()
    row_orders_by_feature[feature_set] = row_order

    z_matrix = []
    text_matrix = []

    for _, row in feature_df.iterrows():
        z_matrix.append(
            [
                float(1 - (row["anomaly_share"] / anomaly_share_max)),
                float(max(row["tail_separation_gap"], 0) / tail_gap_max),
                float(max(row["median_separation"], 0) / median_gap_max),
                float(row["decision_score"]),
            ]
        )
        text_matrix.append(
            [
                f"{row['anomaly_share']:.3f}",
                f"{row['tail_separation_gap']:.3f}",
                f"{row['median_separation']:.3f}",
                str(row["screen_decision"]),
            ]
        )

    fig.add_trace(
        go.Heatmap(
            z=z_matrix,
            x=metric_order,
            y=row_order,
            colorscale=brand_screen_scale,
            zmin=0,
            zmax=1,
            showscale=(feature_set == "fhvhv"),
            colorbar=dict(
                title=dict(
                    text="Relative score",
                    font=dict(color=BRAND_COLORS["dark_teal"]),
                ),
                tickfont=dict(color=BRAND_COLORS["dark_teal"]),
                len=0.84,
                y=0.50,
            ) if feature_set == "fhvhv" else None,
            hovertemplate=(
                "Feature set=" + feature_set
                + "<br>Setting=%{y}"
                + "<br>Metric=%{x}"
                + "<br>Relative score=%{z:.3f}"
                + "<extra></extra>"
            ),
        ),
        row=row_idx,
        col=col_idx,
    )

    for y_index, row_label in enumerate(row_order):
        for x_index, metric_label in enumerate(metric_order):
            cell_text = text_matrix[y_index][x_index]
            cell_score = z_matrix[y_index][x_index]

            fig.add_annotation(
                x=metric_label,
                y=row_label,
                text=cell_text,
                showarrow=False,
                font=dict(
                    size=10,
                    color="white" if cell_score >= 0.78 else BRAND_COLORS["dark_teal"],
                ),
                row=row_idx,
                col=col_idx,
            )

for feature_set in single_modality_order:
    row_idx, col_idx = position_map[feature_set]
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=row_orders_by_feature[feature_set],
        ticklabelstandoff=10,
        automargin=True,
        row=row_idx,
        col=col_idx,
    )

fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(showticklabels=False, row=1, col=2)
fig.update_xaxes(showticklabels=True, row=2, col=1)
fig.update_xaxes(showticklabels=True, row=2, col=2)

fig.update_xaxes(title_text="Threshold dimension", row=2, col=1)
fig.update_xaxes(title_text="Threshold dimension", row=2, col=2)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=980,
    height=780,
    margin=dict(l=210, r=65, t=95, b=55),
)

fig.add_annotation(
    text="GMM Tail-Threshold Screening Scorecards",
    x=0.5,
    y=1.10,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="center",
    yanchor="bottom",
    font=dict(size=18, color=BRAND_COLORS["dark_teal"]),
)

fig.show()

print("Single-modality GMM tail-threshold screening scorecards rendered.")

Single-modality GMM tail-threshold screening scorecards rendered.


Findings\. The single\-modality picture is highly consistent\. Across Bus, Subway, Taxi, and FHVHV, the lowest tail cuts remain the strongest candidates: 0\.005 and 0\.010 most often advance, 0\.020 usually drops into caution, and 0\.050 is generally retired\. In practical terms, the useful single\-modality GMM region sits in the sparse tail, where anomaly prevalence stays controlled and the separation metrics remain strongest\. So the main remaining judgment is not whether loose thresholds survive, but whether each modality should standardize on the sparsest defensible cut or retain a slightly wider 0\.010 tail where the boundary still looks clean\.

The Multimodal branch is easier to read on its own, so this second scorecard isolates it from the four single\-modality panels\. The same screening logic still applies: anomaly share should stay reasonably sparse, tail\_separation\_gap should remain positive, and median\_separation should stay locally competitive within the retained structure\. Pulling Multimodal into its own view makes it easier to judge whether its larger and more heterogeneous mobility surface still supports the same low\-tail threshold logic\.

In [23]:
# ---------------------------------------------------------------------
# Score candidate thresholds with a transparent calibration rule:
# Multimodal scorecard
# ---------------------------------------------------------------------

gmm_tail_screen_df = pd.read_parquet(GMM_TAIL_SWEEP_SUMMARY_PATH).copy()

gmm_tail_screen_df["structure_label"] = (
    "k="
    + gmm_tail_screen_df["n_components"].astype(int).astype(str)
    + " | "
    + gmm_tail_screen_df["covariance_type"].str.upper()
)

gmm_tail_screen_df["tail_gap_rank_within_structure"] = (
    gmm_tail_screen_df.groupby(["feature_set", "structure_label"])["tail_separation_gap"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

gmm_tail_screen_df["median_gap_rank_within_structure"] = (
    gmm_tail_screen_df.groupby(["feature_set", "structure_label"])["median_separation"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

def classify_gmm_tail_threshold(row):
    if row["tail_threshold"] == max(GMM_TAIL_THRESHOLD_GRID):
        return "retire", "Tail cut is too diffuse"
    if row["tail_separation_gap"] <= 0:
        return "retire", "Tail boundary is no longer clean"
    if (
        row["tail_threshold"] <= 0.020
        and row["tail_gap_rank_within_structure"] <= 2
        and row["median_gap_rank_within_structure"] <= 2
    ):
        return "advance", "Sparse threshold with strong tail separation"
    return "caution", "Plausible threshold but less cleanly separated"

gmm_tail_screen_df[["screen_decision", "screen_rationale"]] = (
    gmm_tail_screen_df.apply(
        lambda row: pd.Series(classify_gmm_tail_threshold(row)),
        axis=1,
    )
)

gmm_tail_screen_df["decision_score"] = gmm_tail_screen_df["screen_decision"].map(
    {"advance": 1.0, "caution": 0.5, "retire": 0.0}
)

multimodal_df = gmm_tail_screen_df.loc[
    gmm_tail_screen_df["feature_set"].eq("multimodal")
].copy()

multimodal_df = multimodal_df.sort_values(
    ["n_components", "tail_threshold"],
    ascending=[False, False],
).reset_index(drop=True)

multimodal_df["row_label"] = (
    "k="
    + multimodal_df["n_components"].astype(int).astype(str)
    + " | "
    + multimodal_df["tail_threshold"].map("{:.3f}".format)
)

metric_order = ["Anomaly<br>share", "Tail gap", "Median<br>gap", "Decision"]
row_order = multimodal_df["row_label"].tolist()

brand_screen_scale = [
    [0.00, BRAND_COLORS["terracotta"]],
    [0.50, BRAND_COLORS["pale_peach"]],
    [0.75, BRAND_COLORS["seafoam"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

anomaly_share_max = max(multimodal_df["anomaly_share"].max(), 1e-9)
tail_gap_max = max(multimodal_df["tail_separation_gap"].max(), 1e-9)
median_gap_max = max(multimodal_df["median_separation"].max(), 1e-9)

z_matrix = []
text_matrix = []

for _, row in multimodal_df.iterrows():
    z_matrix.append(
        [
            float(1 - (row["anomaly_share"] / anomaly_share_max)),
            float(max(row["tail_separation_gap"], 0) / tail_gap_max),
            float(max(row["median_separation"], 0) / median_gap_max),
            float(row["decision_score"]),
        ]
    )
    text_matrix.append(
        [
            f"{row['anomaly_share']:.3f}",
            f"{row['tail_separation_gap']:.3f}",
            f"{row['median_separation']:.3f}",
            str(row["screen_decision"]),
        ]
    )

fig = go.Figure(
    go.Heatmap(
        z=z_matrix,
        x=metric_order,
        y=row_order,
        colorscale=brand_screen_scale,
        zmin=0,
        zmax=1,
        colorbar=dict(
            title=dict(
                text="Relative score",
                font=dict(color=BRAND_COLORS["dark_teal"]),
            ),
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            len=0.92,
            y=0.50,
        ),
        hovertemplate=(
            "Feature set=multimodal"
            + "<br>Setting=%{y}"
            + "<br>Metric=%{x}"
            + "<br>Relative score=%{z:.3f}"
            + "<extra></extra>"
        ),
    )
)

for y_index, row_label in enumerate(row_order):
    for x_index, metric_label in enumerate(metric_order):
        cell_text = text_matrix[y_index][x_index]
        cell_score = z_matrix[y_index][x_index]

        fig.add_annotation(
            x=metric_label,
            y=row_label,
            text=cell_text,
            showarrow=False,
            font=dict(
                size=11,
                color="white" if cell_score >= 0.78 else BRAND_COLORS["dark_teal"],
            ),
        )

fig.update_yaxes(
    categoryorder="array",
    categoryarray=row_order,
    ticklabelstandoff=10,
    automargin=True,
)
fig.update_xaxes(title_text="Threshold dimension", tickangle=0, showticklabels=True)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=760,
    height=720,
    margin=dict(l=205, r=70, t=110, b=55),
)

fig.add_annotation(
    text="Multimodal GMM Tail-Threshold Screening Scorecard",
    x=0.5,
    y=1.10,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="center",
    yanchor="bottom",
    font=dict(size=18, color=BRAND_COLORS["dark_teal"]),
)

fig.show()

print("Multimodal GMM tail-threshold screening scorecard rendered.")

Multimodal GMM tail-threshold screening scorecard rendered.


Findings\. The Multimodal branch is easier to read on its own, so this second scorecard isolates it from the four single\-modality panels\. The same screening logic still applies: anomaly share should stay reasonably sparse, tail\_separation\_gap should remain positive, and median\_separation should stay locally competitive within the retained structure\. Pulling Multimodal into its own view makes it easier to judge whether its larger and more heterogeneous mobility surface still supports the same low\-tail threshold logic\.

Retain one final GMM tail threshold for downstream use\. At this point we no longer need a threshold menu; later notebook cells need one deterministic anomaly branch per retained GMM structure\. This block records that final threshold choice explicitly so the rest of the notebook can proceed against a single retained probabilistic anomaly surface\.

In [24]:
# ---------------------------------------------------------------------
# Select the final retained GMM structure and tail threshold per modality
# ---------------------------------------------------------------------

assert GMM_ADVANCING_STRUCTURES_PATH.exists(), (
    "The retained GMM structure table was not found. Please run the structure-screen recording block first."
)
assert GMM_TAIL_SWEEP_SUMMARY_PATH.exists(), (
    "The GMM tail-sweep summary was not found. Please run the tail-threshold materialization block first."
)

FINAL_GMM_TAIL_THRESHOLD = 0.005

# The retained structure decision is finalized here so downstream framework
# comparison runs on one concrete probabilistic surface per modality.
final_structure_choice_by_feature_set = {
    "bus": {"n_components": 3, "covariance_type": "full"},
    "subway": {"n_components": 3, "covariance_type": "full"},
    "taxi": {"n_components": 3, "covariance_type": "full"},
    "fhvhv": {"n_components": 2, "covariance_type": "full"},
    "multimodal": {"n_components": 2, "covariance_type": "full"},
}

selection_rationale_by_feature_set = {
    "bus": "k=3 preserves the stronger anomaly boundary without paying for the extra complexity of k=5.",
    "subway": "k=3 keeps the likelihood tail strong while avoiding the extra fragmentation implied by larger mixtures.",
    "taxi": "k=3 and k=5 behave similarly at the retained tail, so the simpler k=3 structure is carried forward.",
    "fhvhv": "k=2 already delivers the strongest separation and added components do not justify their extra complexity.",
    "multimodal": "k=2 provides the clearest probabilistic boundary; larger mixtures soften separation without a compensating gain.",
}

gmm_advancing_structures_df = pd.read_parquet(GMM_ADVANCING_STRUCTURES_PATH).copy()
gmm_tail_sweep_summary_df = pd.read_parquet(GMM_TAIL_SWEEP_SUMMARY_PATH).copy()

# Normalize the saved tail-summary schema to the compact names used downstream.
gmm_tail_sweep_summary_df = gmm_tail_sweep_summary_df.rename(
    columns={
        "tail_separation_gap": "tail_gap",
        "median_separation": "median_gap",
    }
)

gmm_tail_at_default_df = (
    gmm_tail_sweep_summary_df.loc[
        np.isclose(gmm_tail_sweep_summary_df["tail_threshold"], FINAL_GMM_TAIL_THRESHOLD)
    ]
    .merge(
        gmm_advancing_structures_df[
            [
                "candidate_id",
                "feature_set",
                "representation_name",
                "n_components",
                "covariance_type",
            ]
        ].drop_duplicates(),
        on=["feature_set", "representation_name", "n_components", "covariance_type"],
        how="inner",
        validate="many_to_one",
    )
    .copy()
)

final_structure_choice_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set,
            "n_components": choice["n_components"],
            "covariance_type": choice["covariance_type"],
            "tail_threshold": FINAL_GMM_TAIL_THRESHOLD,
            "selection_rationale": selection_rationale_by_feature_set[feature_set],
        }
        for feature_set, choice in final_structure_choice_by_feature_set.items()
    ]
)

gmm_selected_configurations_df = (
    final_structure_choice_df.merge(
        gmm_tail_at_default_df,
        on=["feature_set", "n_components", "covariance_type", "tail_threshold"],
        how="left",
        validate="one_to_one",
    )
    .sort_values("feature_set")
    .reset_index(drop=True)
)

required_selected_columns = [
    "candidate_id",
    "feature_set",
    "representation_name",
    "n_components",
    "covariance_type",
    "tail_threshold",
    "anomaly_rows",
    "anomaly_share",
    "tail_gap",
    "median_gap",
    "selection_rationale",
]

missing_selected_columns = [
    column for column in required_selected_columns if column not in gmm_selected_configurations_df.columns
]
assert not missing_selected_columns, (
    f"The selected GMM configuration table is missing columns: {missing_selected_columns}"
)
assert gmm_selected_configurations_df["candidate_id"].notna().all(), (
    "One or more selected GMM structures could not be matched to the retained tail-threshold summary."
)

gmm_selected_structure_export_df = gmm_selected_configurations_df[
    [
        "feature_set",
        "representation_name",
        "candidate_id",
        "n_components",
        "covariance_type",
        "selection_rationale",
    ]
].copy()

gmm_selected_threshold_export_df = gmm_selected_configurations_df[
    [
        "feature_set",
        "representation_name",
        "candidate_id",
        "tail_threshold",
        "anomaly_rows",
        "anomaly_share",
        "tail_gap",
        "median_gap",
    ]
].copy()

gmm_final_selection_validation_df = pd.DataFrame(
    [
        {
            "feature_sets_selected": gmm_selected_configurations_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "selected_configuration_rows": len(gmm_selected_configurations_df),
            "status": (
                "pass"
                if (
                    gmm_selected_configurations_df["feature_set"].nunique()
                    == len(MODEL_FEATURE_SET_NAMES)
                    and len(gmm_selected_configurations_df) == len(MODEL_FEATURE_SET_NAMES)
                )
                else "review"
            ),
        }
    ]
)

if WRITE_334_OUTPUTS:
    gmm_selected_structure_export_df.to_csv(GMM_SELECTED_STRUCTURE_PATH, index=False)
    gmm_selected_threshold_export_df.to_csv(GMM_SELECTED_THRESHOLD_PATH, index=False)
    gmm_selected_configurations_df.to_csv(GMM_SELECTED_CONFIGURATION_PATH, index=False)

display(
    gmm_selected_configurations_df[
        [
            "feature_set",
            "representation_name",
            "n_components",
            "covariance_type",
            "tail_threshold",
            "anomaly_share",
            "tail_gap",
            "median_gap",
            "selection_rationale",
        ]
    ].style.format(
        {
            "tail_threshold": "{:.3f}",
            "anomaly_share": "{:.3f}",
            "tail_gap": "{:.3f}",
            "median_gap": "{:.3f}",
        }
    )
)
display(gmm_final_selection_validation_df)

assert gmm_final_selection_validation_df["status"].eq("pass").all(), (
    "The final GMM selection did not resolve to exactly one retained configuration per modality."
)

,feature_set,representation_name,n_components,covariance_type,tail_threshold,anomaly_share,tail_gap,median_gap,selection_rationale
0,bus,pca_80pct,3,full,0.005,0.005,39.664,80.333,k=3 preserves the stronger anomaly boundary without paying for the extra complexity of k=5.
1,fhvhv,pca_80pct,2,full,0.005,0.005,59.431,110.933,k=2 already delivers the strongest separation and added components do not justify their extra complexity.
2,multimodal,pca_80pct,2,full,0.005,0.005,122.257,209.334,k=2 provides the clearest probabilistic boundary; larger mixtures soften separation without a compensating gain.
3,subway,pca_80pct,3,full,0.005,0.005,36.673,70.565,k=3 keeps the likelihood tail strong while avoiding the extra fragmentation implied by larger mixtures.
4,taxi,pca_80pct,3,full,0.005,0.005,15.305,38.527,"k=3 and k=5 behave similarly at the retained tail, so the simpler k=3 structure is carried forward."


,feature_sets_selected,expected_feature_sets,selected_configuration_rows,status
0,5,5,5,pass


Findings\. The GMM calibration now resolves to one retained probabilistic configuration per modality, and all five selections pass the final handoff check\. Every modality keeps the same sparse tail threshold of 0\.5%, but the retained component count is modality\-specific: k=3 \| full for Bus, Subway, and Taxi, and k=2 \| full for FHVHV and Multimodal\. The practical takeaway is that we are no longer carrying a candidate grid forward\. We now have one concrete GMM anomaly surface per modality, chosen to preserve a strong low\-likelihood boundary without paying for extra mixture complexity that did not add enough downstream value\.

> 🎯 Decision\. We will retain one final GMM configuration per modality for downstream use, and every retained branch will use the same sparse likelihood\-tail threshold of 0\.005\. Across modalities, 0\.005 consistently preserves the strongest likelihood separation and the clearest anomaly boundary while keeping anomaly volume sparse and interpretable, so thresholds 0\.010 and 0\.020 will remain calibration evidence only, and 0\.050 is retired\. The retained covariance structure is full in every modality, while the retained component count is modality\-specific: Bus, Subway, and Taxi will carry forward k=3, and FHVHV and Multimodal will carry forward k=2\. In practical terms, this means GMM now advances as one concrete probabilistic anomaly surface per modality rather than as an unresolved candidate grid\.

Record the thresholds advancing beyond calibration\. The scorecard explains the retention logic; this final block just makes the retained threshold region explicit and verifies that every modality still has at least one viable threshold candidate\.

In [25]:
# ---------------------------------------------------------------------
# Record retained GMM configurations advancing beyond calibration
# ---------------------------------------------------------------------

assert GMM_SELECTED_CONFIGURATION_PATH.exists(), (
    "The final selected GMM configuration file was not found. Please run the final GMM selection block first."
)

gmm_selected_configurations_df = pd.read_csv(GMM_SELECTED_CONFIGURATION_PATH)

gmm_retained_configuration_inventory_df = (
    gmm_selected_configurations_df.groupby("feature_set", dropna=False)
    .agg(
        retained_configurations=("feature_set", "size"),
        retained_component_count=("n_components", "first"),
        retained_covariance_type=("covariance_type", "first"),
        retained_tail_threshold=("tail_threshold", "first"),
    )
    .reset_index()
)

gmm_retained_configuration_validation_df = pd.DataFrame(
    [
        {
            "feature_sets_with_selected_configuration": gmm_selected_configurations_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "selected_configuration_rows": len(gmm_selected_configurations_df),
            "status": (
                "pass"
                if (
                    gmm_selected_configurations_df["feature_set"].nunique()
                    == len(MODEL_FEATURE_SET_NAMES)
                    and len(gmm_selected_configurations_df) == len(MODEL_FEATURE_SET_NAMES)
                )
                else "review"
            ),
        }
    ]
)

display(
    gmm_retained_configuration_inventory_df.style.format(
        {"retained_tail_threshold": "{:.3f}"}
    )
)
display(gmm_retained_configuration_validation_df)

assert gmm_retained_configuration_validation_df["status"].eq("pass").all(), (
    "The final GMM calibration handoff did not resolve to exactly one retained configuration per modality."
)

,feature_set,retained_configurations,retained_component_count,retained_covariance_type,retained_tail_threshold
0,bus,1,3,full,0.005
1,fhvhv,1,2,full,0.005
2,multimodal,1,2,full,0.005
3,subway,1,3,full,0.005
4,taxi,1,3,full,0.005


,feature_sets_with_selected_configuration,expected_feature_sets,selected_configuration_rows,status
0,5,5,5,pass


Findings\. The final GMM calibration handoff is now clean and fully resolved: each modality carries exactly one retained configuration, every retained branch uses full covariance, and the selected tail threshold is 0\.005 across the board\. The only remaining modality\-level difference is component count, with Bus, Subway, and Taxi retaining k=3 and FHVHV and Multimodal retaining k=2\. So the practical takeaway is that 3\.3\.4\.3 has finished its job: the GMM calibration space has been collapsed from a candidate grid into one concrete downstream anomaly surface per modality\.

### Section Summary

This section turned the retained GMM structure grid into final probabilistic anomaly configurations\. First, the tail\-threshold sweep showed that anomaly prevalence behaves mechanically as expected while likelihood separation weakens as the retained tail grows wider, making 0\.005 the clearest and most interpretable shared threshold\. Then the screening step confirmed that only a small subset of structure\-threshold combinations preserve a strong enough low\-likelihood boundary to justify downstream use\. Finally, the notebook resolved that candidate set to one retained GMM configuration per modality: full covariance everywhere, k=3 for Bus, Subway, and Taxi, k=2 for FHVHV and Multimodal, and a shared tail threshold of 0\.005\. At this point, GMM has a finalized probabilistic anomaly surface ready for cross\-framework comparison against DBSCAN and Isolation Forest\.

\-\-\-

## 3\.3\.4\.4 Compare GMM, DBSCAN, and Isolation Forest Anomaly Surfaces

This section performs a lightweight complementarity check across the retained anomaly surfaces from GMM, DBSCAN, and Isolation Forest\. The point is not to declare a winning framework yet\. It is to understand whether GMM is mostly reproducing anomaly structure that the other methods already capture or whether it contributes a distinct probabilistic anomaly slice worth preserving for downstream framework comparison\.

### Load and align the retained anomaly surfaces across all three frameworks

Start by putting the three retained framework surfaces on the same footing\. We want one comparison\-ready anomaly branch per modality from DBSCAN, Isolation Forest, and GMM, together with a simple alignment check proving that they refer to the same row universe\.

In [26]:
# ---------------------------------------------------------------------
# Load and align the retained anomaly surfaces across all three frameworks
# Calibration-universe only
# ---------------------------------------------------------------------

def identify_existing_column(df, candidate_columns, required=True):
    for column in candidate_columns:
        if column in df.columns:
            return column
    if required:
        raise AssertionError(
            f"Could not identify any of the expected columns {candidate_columns}; "
            f"available columns were {df.columns.tolist()}."
        )
    return None


def identify_framework_row_level_path_column(df, required=True):
    return identify_existing_column(
        df,
        [
            "row_level_output_path",
            "final_row_level_output_path",
            "candidate_anomaly_row_level_path",
            "output_path",
        ],
        required=required,
    )


def derive_framework_anomaly_flag(row_level_df, framework_label):
    candidate_columns_by_framework = {
        "DBSCAN": ["is_anomaly_like", "anomaly_like_flag", "anomaly_flag", "dbscan_anomaly_flag"],
        "Isolation Forest": ["is_anomaly", "anomaly_flag", "if_anomaly_flag"],
        "GMM": ["is_anomaly", "anomaly_flag", "gmm_anomaly_flag"],
    }

    for column in candidate_columns_by_framework[framework_label]:
        if column in row_level_df.columns:
            return row_level_df[column].fillna(0).astype(int)

    if framework_label == "DBSCAN" and "cluster_label" in row_level_df.columns:
        return row_level_df["cluster_label"].fillna(0).eq(-1).astype(int)

    raise AssertionError(
        f"Could not derive an anomaly flag for {framework_label}; "
        f"available columns were {row_level_df.columns.tolist()}."
    )


def build_gmm_tail_candidate_id(feature_set, n_components, covariance_type, tail_threshold):
    tail_token = str(tail_threshold).replace("0.", "0p").replace(".", "p")
    return f"{feature_set}__k{int(n_components)}__cov_{covariance_type}__tail_{tail_token}"


def resolve_dbscan_row_level_path(feature_set, representation_name=None):
    candidate_paths = []

    if representation_name:
        candidate_paths.append(
            DBSCAN_FINAL_ROW_LEVEL_DIR / f"{feature_set}_{representation_name}_dbscan_candidate_anomalies.parquet"
        )
        candidate_paths.append(
            DBSCAN_FINAL_ROW_LEVEL_DIR / f"{feature_set}_{representation_name}_dbscan_candidate_anomaly_rows.parquet"
        )

    candidate_paths.append(
        DBSCAN_FINAL_ROW_LEVEL_DIR / f"{feature_set}_dbscan_candidate_anomalies.parquet"
    )
    candidate_paths.append(
        DBSCAN_FINAL_ROW_LEVEL_DIR / f"{feature_set}_dbscan_candidate_anomaly_rows.parquet"
    )

    for candidate_path in candidate_paths:
        if candidate_path.exists():
            return candidate_path

    matches = sorted(DBSCAN_FINAL_ROW_LEVEL_DIR.glob(f"{feature_set}*.parquet"))
    if representation_name:
        representation_matches = [
            path for path in matches if representation_name in path.stem
        ]
        if len(representation_matches) == 1:
            return representation_matches[0]
        if len(representation_matches) > 1:
            raise AssertionError(
                f"Found multiple DBSCAN row-level files for {feature_set} / {representation_name}: "
                f"{[str(path) for path in representation_matches]}"
            )

    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise AssertionError(
            f"Found multiple DBSCAN row-level files for {feature_set}: {[str(path) for path in matches]}"
        )

    return candidate_paths[0]


def resolve_if_row_level_path(feature_set, manifest_path, representation_name=None):
    manifest_path = Path(manifest_path)
    if manifest_path.exists():
        return manifest_path

    candidate_paths = []

    if representation_name:
        candidate_paths.append(
            IF_FINAL_ROW_LEVEL_DIR / f"{feature_set}_{representation_name}_if_candidate_anomalies.parquet"
        )

    candidate_paths.append(
        IF_FINAL_ROW_LEVEL_DIR / f"{feature_set}_if_candidate_anomalies.parquet"
    )

    for candidate_path in candidate_paths:
        if candidate_path.exists():
            return candidate_path

    matches = sorted(IF_FINAL_ROW_LEVEL_DIR.glob(f"{feature_set}*.parquet"))
    if representation_name:
        representation_matches = [
            path for path in matches if representation_name in path.stem
        ]
        if len(representation_matches) == 1:
            return representation_matches[0]

    if len(matches) == 1:
        return matches[0]

    raise AssertionError(
        f"Could not resolve Isolation Forest row-level file for {feature_set}. "
        f"Manifest path tried: {manifest_path}. Candidate paths tried: {[str(path) for path in candidate_paths]}"
    )


def resolve_gmm_row_level_path(feature_set, n_components, covariance_type, tail_threshold):
    tail_candidate_id = build_gmm_tail_candidate_id(
        feature_set,
        n_components,
        covariance_type,
        tail_threshold,
    )
    return GMM_TAIL_ROW_LEVEL_DIR / f"{tail_candidate_id}.parquet"


def filter_to_calibration_universe(row_level_df, feature_set):
    calibration_ids_df = pd.read_parquet(
        ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set],
        columns=[MODEL_ID_COLUMN],
    ).drop_duplicates()

    return (
        row_level_df.merge(
            calibration_ids_df,
            on=MODEL_ID_COLUMN,
            how="inner",
            validate="many_to_one",
        )
        .copy()
    )


if (
    not should_rebuild(REBUILD_GMM_FRAMEWORK_COMPARISONS, GMM_FRAMEWORK_COMPARISON_ROW_LEVEL_PATH)
    and GMM_FRAMEWORK_COMPARISON_MANIFEST_PATH.exists()
    and GMM_FRAMEWORK_COMPARISON_ROW_LEVEL_PATH.exists()
):
    framework_comparison_manifest_df = pd.read_parquet(GMM_FRAMEWORK_COMPARISON_MANIFEST_PATH)
    framework_comparison_row_level_df = pd.read_parquet(GMM_FRAMEWORK_COMPARISON_ROW_LEVEL_PATH)

else:
    assert DBSCAN_SELECTED_CONFIGURATION_PATH.exists(), (
        "The selected DBSCAN configuration file was not found. Please run the DBSCAN final-selection block first."
    )
    assert IF_ANOMALY_EXPORT_MANIFEST_PATH.exists(), (
        "The Isolation Forest export manifest was not found. Please run the Isolation Forest export block first."
    )
    assert GMM_SELECTED_CONFIGURATION_PATH.exists(), (
        "The selected GMM configuration file was not found. Please run the final GMM selection block first."
    )

    dbscan_selected_configurations_df = pd.read_csv(DBSCAN_SELECTED_CONFIGURATION_PATH)
    if_anomaly_export_manifest_df = pd.read_csv(IF_ANOMALY_EXPORT_MANIFEST_PATH)
    gmm_selected_configurations_df = pd.read_csv(GMM_SELECTED_CONFIGURATION_PATH)

    if_row_level_path_column = identify_framework_row_level_path_column(if_anomaly_export_manifest_df)

    dbscan_manifest_df = dbscan_selected_configurations_df[
        ["feature_set", "representation_name", "eps_band", "min_samples"]
    ].drop_duplicates().copy()
    dbscan_manifest_df["framework_label"] = "DBSCAN"
    dbscan_manifest_df["branch_label"] = (
        "DBSCAN | "
        + dbscan_manifest_df["representation_name"].astype(str)
        + " | eps="
        + dbscan_manifest_df["eps_band"].astype(str)
        + " | ms="
        + dbscan_manifest_df["min_samples"].astype(str)
    )
    dbscan_manifest_df["row_level_output_path"] = dbscan_manifest_df.apply(
        lambda row: str(
            resolve_dbscan_row_level_path(
                row["feature_set"],
                row["representation_name"],
            )
        ),
        axis=1,
    )

    if_manifest_df = if_anomaly_export_manifest_df[
        ["feature_set", "representation_name", "contamination", if_row_level_path_column]
    ].drop_duplicates().copy()
    if_manifest_df["framework_label"] = "Isolation Forest"
    if_manifest_df["branch_label"] = (
        "Isolation Forest | "
        + if_manifest_df["representation_name"].astype(str)
        + " | contamination="
        + if_manifest_df["contamination"].map(lambda v: f"{float(v):.3f}")
    )
    if_manifest_df["row_level_output_path"] = if_manifest_df.apply(
        lambda row: str(
            resolve_if_row_level_path(
                row["feature_set"],
                row[if_row_level_path_column],
                row["representation_name"],
            )
        ),
        axis=1,
    )

    gmm_manifest_df = gmm_selected_configurations_df[
        [
            "feature_set",
            "representation_name",
            "n_components",
            "covariance_type",
            "tail_threshold",
        ]
    ].drop_duplicates().copy()
    gmm_manifest_df["framework_label"] = "GMM"
    gmm_manifest_df["branch_label"] = (
        "GMM | k="
        + gmm_manifest_df["n_components"].astype(str)
        + " | "
        + gmm_manifest_df["covariance_type"].str.upper()
        + " | tail="
        + gmm_manifest_df["tail_threshold"].map(lambda v: f"{float(v):.3f}")
    )
    gmm_manifest_df["row_level_output_path"] = gmm_manifest_df.apply(
        lambda row: str(
            resolve_gmm_row_level_path(
                row["feature_set"],
                row["n_components"],
                row["covariance_type"],
                row["tail_threshold"],
            )
        ),
        axis=1,
    )

    framework_comparison_manifest_df = pd.concat(
        [
            dbscan_manifest_df[
                [
                    "framework_label",
                    "feature_set",
                    "representation_name",
                    "branch_label",
                    "row_level_output_path",
                ]
            ],
            if_manifest_df[
                [
                    "framework_label",
                    "feature_set",
                    "representation_name",
                    "branch_label",
                    "row_level_output_path",
                ]
            ],
            gmm_manifest_df[
                [
                    "framework_label",
                    "feature_set",
                    "representation_name",
                    "branch_label",
                    "row_level_output_path",
                ]
            ],
        ],
        ignore_index=True,
    ).sort_values(["feature_set", "framework_label"]).reset_index(drop=True)

    framework_comparison_manifest_df["path_exists"] = framework_comparison_manifest_df[
        "row_level_output_path"
    ].map(lambda path: Path(path).exists())

    missing_framework_paths_df = framework_comparison_manifest_df.loc[
        ~framework_comparison_manifest_df["path_exists"]
    ].copy()

    framework_comparison_manifest_summary_df = (
        framework_comparison_manifest_df.groupby("framework_label", dropna=False)
        .agg(
            retained_branches=("feature_set", "size"),
            paths_found=("path_exists", "sum"),
        )
        .reset_index()
    )
    framework_comparison_manifest_summary_df["status"] = np.where(
        framework_comparison_manifest_summary_df["retained_branches"]
        == framework_comparison_manifest_summary_df["paths_found"],
        "pass",
        "review",
    )

    display(framework_comparison_manifest_summary_df)
    if not missing_framework_paths_df.empty:
        display(missing_framework_paths_df)

    assert framework_comparison_manifest_df["path_exists"].all(), (
        "One or more retained framework row-level outputs is missing."
    )

    comparison_frames = []

    for manifest_row in framework_comparison_manifest_df.to_dict("records"):
        framework_label = manifest_row["framework_label"]
        feature_set = manifest_row["feature_set"]

        row_level_df = pd.read_parquet(manifest_row["row_level_output_path"]).copy()
        row_level_df = filter_to_calibration_universe(row_level_df, feature_set)

        row_level_df["framework_label"] = framework_label
        row_level_df["feature_set"] = feature_set
        row_level_df["branch_label"] = manifest_row["branch_label"]
        row_level_df["anomaly_flag"] = derive_framework_anomaly_flag(row_level_df, framework_label)

        required_columns = [
            MODEL_ID_COLUMN,
            "framework_label",
            "feature_set",
            "branch_label",
            "anomaly_flag",
        ]
        available_columns = [
            column
            for column in DEFAULT_METADATA_COLUMNS
            if column in row_level_df.columns and column not in required_columns
        ]
        comparison_frames.append(row_level_df[required_columns + available_columns].copy())

    framework_comparison_row_level_df = pd.concat(
        comparison_frames,
        ignore_index=True,
    )

    if WRITE_334_OUTPUTS:
        framework_comparison_manifest_df.to_parquet(
            GMM_FRAMEWORK_COMPARISON_MANIFEST_PATH,
            index=False,
        )
        framework_comparison_row_level_df.to_parquet(
            GMM_FRAMEWORK_COMPARISON_ROW_LEVEL_PATH,
            index=False,
        )

framework_alignment_validation_df = pd.DataFrame(
    [
        {
            "frameworks_loaded": framework_comparison_manifest_df["framework_label"].nunique(),
            "expected_frameworks": 3,
            "feature_sets_covered": framework_comparison_manifest_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "retained_branches_loaded": len(framework_comparison_manifest_df),
            "status": (
                "pass"
                if (
                    framework_comparison_manifest_df["framework_label"].nunique() == 3
                    and framework_comparison_manifest_df["feature_set"].nunique() == len(MODEL_FEATURE_SET_NAMES)
                )
                else "review"
            ),
        }
    ]
)

display(framework_alignment_validation_df)

assert framework_alignment_validation_df["status"].eq("pass").all(), (
    "The retained framework comparison table did not load all expected frameworks and feature sets."
)

,frameworks_loaded,expected_frameworks,feature_sets_covered,expected_feature_sets,retained_branches_loaded,status
0,3,3,5,5,15,pass


Findings\. The retained framework handoff is complete: DBSCAN, Isolation Forest, and GMM all contribute five available row\-level branches, and the aligned comparison panel successfully loads all three frameworks across all five feature sets\.

Validate that the retained surfaces are row\-aligned within each modality\. The only thing we need to prove here is that each framework is evaluating the same row universe before we start comparing prevalence, overlap, and concentration\.

In [27]:
# ---------------------------------------------------------------------
# Validate row alignment across retained framework surfaces
# ---------------------------------------------------------------------

assert "framework_comparison_row_level_df" in globals(), (
    "framework_comparison_row_level_df is not in memory. "
    "Please run the retained-framework loading block first."
)

framework_alignment_records = []

for feature_set in MODEL_FEATURE_SET_NAMES:
    feature_df = framework_comparison_row_level_df.loc[
        framework_comparison_row_level_df["feature_set"].eq(feature_set)
    ].copy()

    feature_row_id_sets = {}
    feature_row_counts = {}

    for framework_label in ["DBSCAN", "Isolation Forest", "GMM"]:
        framework_df = feature_df.loc[
            feature_df["framework_label"].eq(framework_label)
        ].copy()

        feature_row_id_sets[framework_label] = set(framework_df[MODEL_ID_COLUMN].tolist())
        feature_row_counts[framework_label] = len(framework_df)

    unique_row_id_sets = list(feature_row_id_sets.values())
    same_row_universe = all(
        row_id_set == unique_row_id_sets[0]
        for row_id_set in unique_row_id_sets[1:]
    )

    framework_alignment_records.append(
        {
            "feature_set": feature_set,
            "dbscan_rows": feature_row_counts.get("DBSCAN", np.nan),
            "if_rows": feature_row_counts.get("Isolation Forest", np.nan),
            "gmm_rows": feature_row_counts.get("GMM", np.nan),
            "same_row_universe": same_row_universe,
            "status": "pass" if same_row_universe else "review",
        }
    )

framework_alignment_validation_df = pd.DataFrame(framework_alignment_records)

display(framework_alignment_validation_df)

assert framework_alignment_validation_df["status"].eq("pass").all(), (
    "One or more modalities has mismatched retained framework row universes."
)

,feature_set,dbscan_rows,if_rows,gmm_rows,same_row_universe,status
0,bus,50000,50000,50000,True,pass
1,subway,50000,50000,50000,True,pass
2,taxi,50000,50000,50000,True,pass
3,fhvhv,50000,50000,50000,True,pass
4,multimodal,50000,50000,50000,True,pass


Findings\. The retained DBSCAN, Isolation Forest, and GMM surfaces are aligned on the same row universe within each modality, so the rest of this section can be read as a true framework comparison rather than a mismatch in underlying panels\.

### Compare anomaly prevalence and row\-level overlap across frameworks

This part answers two basic questions\. First, how large is each framework’s anomaly budget? Second, how much of that anomaly surface is actually shared row\-for\-row? Anomaly share tells us how aggressively each framework is flagging observations\. Jaccard overlap tells us how much two frameworks are flagging the same rows out of everything either one flagged\. Framework\-only share tells us how much unique anomaly mass each method contributes beyond the overlap\.

Compare anomaly prevalence across retained framework branches\. This is the simplest cross\-framework check: how much of each modality panel is each framework labeling as anomalous?

In [28]:
# ---------------------------------------------------------------------
# Compare anomaly prevalence across retained framework branches
# ---------------------------------------------------------------------

assert "framework_comparison_row_level_df" in globals(), (
    "framework_comparison_row_level_df is not in memory. "
    "Please run the retained-framework loading block first."
)

framework_prevalence_df = (
    framework_comparison_row_level_df.groupby(
        ["feature_set", "framework_label", "branch_label"],
        dropna=False,
        observed=False,
    )
    .agg(
        rows_evaluated=(MODEL_ID_COLUMN, "size"),
        anomaly_rows=("anomaly_flag", lambda s: s.fillna(0).astype(int).sum()),
    )
    .reset_index()
    .sort_values(["feature_set", "framework_label"])
    .reset_index(drop=True)
)

framework_prevalence_df["anomaly_share"] = (
    framework_prevalence_df["anomaly_rows"] / framework_prevalence_df["rows_evaluated"]
)

display(
    framework_prevalence_df[
        ["feature_set", "framework_label", "anomaly_rows", "anomaly_share", "branch_label"]
    ].style.format({"anomaly_share": "{:.3f}"})
)

fig = px.bar(
    framework_prevalence_df,
    x="feature_set",
    y="anomaly_share",
    color="framework_label",
    barmode="group",
    text="anomaly_share",
    category_orders={
        "feature_set": ["bus", "subway", "taxi", "fhvhv", "multimodal"],
        "framework_label": ["DBSCAN", "Isolation Forest", "GMM"],
    },
    labels={
        "feature_set": "Feature set",
        "anomaly_share": "Anomaly share",
        "framework_label": "Framework",
    },
    title="Retained GMM vs DBSCAN vs Isolation Forest Anomaly Prevalence",
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside", cliponaxis=False)
fig.update_layout(
    width=960,
    height=480,
    margin=dict(l=60, r=40, t=80, b=50),
)
fig.show()

print("Framework prevalence comparison rendered.")

,feature_set,framework_label,anomaly_rows,anomaly_share,branch_label
0,bus,DBSCAN,885,0.018,DBSCAN | pca_80pct | eps=balanced | ms=15
1,bus,GMM,250,0.005,GMM | k=3 | FULL | tail=0.005
2,bus,Isolation Forest,250,0.005,Isolation Forest | pca_80pct | contamination=0.005
3,fhvhv,DBSCAN,1026,0.021,DBSCAN | pca_80pct | eps=balanced | ms=15
4,fhvhv,GMM,250,0.005,GMM | k=2 | FULL | tail=0.005
5,fhvhv,Isolation Forest,250,0.005,Isolation Forest | pca_80pct | contamination=0.005
6,multimodal,DBSCAN,859,0.017,DBSCAN | pca_80pct | eps=balanced | ms=15
7,multimodal,GMM,250,0.005,GMM | k=2 | FULL | tail=0.005
8,multimodal,Isolation Forest,250,0.005,Isolation Forest | pca_80pct | contamination=0.005
9,subway,DBSCAN,1590,0.032,DBSCAN | pca_80pct | eps=balanced | ms=15


Framework prevalence comparison rendered.


Findings\. The prevalence contrast is straightforward: GMM and Isolation Forest are both operating at the same sparse 0\.5% anomaly budget in every modality, while DBSCAN is materially broader, ranging from 1\.7% to 3\.3%\. So the practical read is that DBSCAN is casting a wider anomaly net, while GMM and Isolation Forest are both serving as much narrower tail\-based detectors\.

Compare row\-level overlap across frameworks\. Jaccard overlap tells us how much two frameworks agree on the exact same rows, while the framework\-specific shares tell us how much of the combined anomaly surface belongs only to one method\. In practical terms, this block tells us whether GMM is mostly repeating the other frameworks or adding a different probabilistic anomaly slice\.

In [29]:
# ---------------------------------------------------------------------
# Compare row-level overlap across frameworks
# ---------------------------------------------------------------------

assert "framework_comparison_row_level_df" in globals(), (
    "framework_comparison_row_level_df is not in memory. "
    "Please run the retained-framework loading block first."
)

framework_flag_cache = {}

for feature_set in MODEL_FEATURE_SET_NAMES:
    for framework_label in ["DBSCAN", "Isolation Forest", "GMM"]:
        scoped_df = (
            framework_comparison_row_level_df.loc[
                framework_comparison_row_level_df["feature_set"].eq(feature_set)
                & framework_comparison_row_level_df["framework_label"].eq(framework_label),
                [MODEL_ID_COLUMN, "anomaly_flag"],
            ]
            .copy()
            .rename(columns={"anomaly_flag": "anomaly_flag"})
            .reset_index(drop=True)
        )

        framework_flag_cache[(feature_set, framework_label)] = scoped_df

pairwise_framework_comparisons = [
    ("GMM", "DBSCAN"),
    ("GMM", "Isolation Forest"),
    ("DBSCAN", "Isolation Forest"),
]

framework_overlap_records = []

for feature_set in MODEL_FEATURE_SET_NAMES:
    for left_framework, right_framework in pairwise_framework_comparisons:
        left_df = framework_flag_cache[(feature_set, left_framework)].rename(
            columns={"anomaly_flag": "left_flag"}
        )
        right_df = framework_flag_cache[(feature_set, right_framework)].rename(
            columns={"anomaly_flag": "right_flag"}
        )

        overlap_df = left_df.merge(
            right_df,
            on=MODEL_ID_COLUMN,
            how="inner",
            validate="one_to_one",
        )

        left_flag = overlap_df["left_flag"].fillna(0).astype(int)
        right_flag = overlap_df["right_flag"].fillna(0).astype(int)

        shared_flag_rows = int(((left_flag == 1) & (right_flag == 1)).sum())
        left_only_rows = int(((left_flag == 1) & (right_flag == 0)).sum())
        right_only_rows = int(((left_flag == 0) & (right_flag == 1)).sum())
        union_flag_rows = shared_flag_rows + left_only_rows + right_only_rows

        framework_overlap_records.append(
            {
                "feature_set": feature_set,
                "left_framework": left_framework,
                "right_framework": right_framework,
                "rows_compared": int(len(overlap_df)),
                "left_flag_count": int(left_flag.sum()),
                "right_flag_count": int(right_flag.sum()),
                "shared_flag_rows": shared_flag_rows,
                "left_only_rows": left_only_rows,
                "right_only_rows": right_only_rows,
                "union_flag_rows": union_flag_rows,
                "jaccard_similarity": (
                    shared_flag_rows / union_flag_rows if union_flag_rows > 0 else np.nan
                ),
                "left_only_share_of_union": (
                    left_only_rows / union_flag_rows if union_flag_rows > 0 else np.nan
                ),
                "right_only_share_of_union": (
                    right_only_rows / union_flag_rows if union_flag_rows > 0 else np.nan
                ),
                "shared_share_of_union": (
                    shared_flag_rows / union_flag_rows if union_flag_rows > 0 else np.nan
                ),
            }
        )

framework_overlap_df = pd.DataFrame(framework_overlap_records).sort_values(
    ["feature_set", "left_framework", "right_framework"]
).reset_index(drop=True)

display(
    framework_overlap_df[
        [
            "feature_set",
            "left_framework",
            "right_framework",
            "jaccard_similarity",
            "left_only_share_of_union",
            "right_only_share_of_union",
            "shared_share_of_union",
        ]
    ].style.format(
        {
            "jaccard_similarity": "{:.3f}",
            "left_only_share_of_union": "{:.3f}",
            "right_only_share_of_union": "{:.3f}",
            "shared_share_of_union": "{:.3f}",
        }
    )
)

gmm_overlap_plot_df = framework_overlap_df.loc[
    framework_overlap_df["left_framework"].eq("GMM")
].copy()

gmm_overlap_plot_df["comparison_label"] = (
    gmm_overlap_plot_df["left_framework"]
    + " vs "
    + gmm_overlap_plot_df["right_framework"]
)

comparison_order = ["GMM vs DBSCAN", "GMM vs Isolation Forest"]
feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]

gmm_overlap_plot_df["comparison_label"] = pd.Categorical(
    gmm_overlap_plot_df["comparison_label"],
    categories=comparison_order,
    ordered=True,
)

gmm_overlap_plot_df["feature_set"] = pd.Categorical(
    gmm_overlap_plot_df["feature_set"],
    categories=feature_set_order,
    ordered=True,
)

gmm_overlap_plot_df = gmm_overlap_plot_df.sort_values(
    ["comparison_label", "feature_set"]
).reset_index(drop=True)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Jaccard overlap", "Shared share of union"],
    horizontal_spacing=0.12,
)

color_map = {
    "GMM vs DBSCAN": BRAND_COLORS["dark_teal"],
    "GMM vs Isolation Forest": BRAND_COLORS["terracotta"],
}

for comparison_label in comparison_order:
    trace_df = gmm_overlap_plot_df.loc[
        gmm_overlap_plot_df["comparison_label"].eq(comparison_label)
    ]

    fig.add_trace(
        go.Bar(
            x=trace_df["feature_set"],
            y=trace_df["jaccard_similarity"],
            name=comparison_label,
            marker_color=color_map[comparison_label],
            text=trace_df["jaccard_similarity"].map(lambda v: f"{v:.3f}"),
            textposition="outside",
            cliponaxis=False,
            legendgroup=comparison_label,
        ),
        row=1,
        col=1,
    )

for comparison_label in comparison_order:
    trace_df = gmm_overlap_plot_df.loc[
        gmm_overlap_plot_df["comparison_label"].eq(comparison_label)
    ]

    fig.add_trace(
        go.Bar(
            x=trace_df["feature_set"],
            y=trace_df["shared_share_of_union"],
            name=comparison_label,
            marker_color=color_map[comparison_label],
            text=trace_df["shared_share_of_union"].map(lambda v: f"{v:.3f}"),
            textposition="outside",
            cliponaxis=False,
            legendgroup=comparison_label,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig.update_layout(
    width=960,
    height=480,
    margin=dict(l=60, r=40, t=80, b=50),
    barmode="group",
)

fig.update_yaxes(title_text="Jaccard", ticklabelstandoff=4, row=1, col=1)
fig.update_yaxes(title_text="Shared share", ticklabelstandoff=4, row=1, col=2)

fig.show()

print("Framework overlap comparison rendered.")

,feature_set,left_framework,right_framework,jaccard_similarity,left_only_share_of_union,right_only_share_of_union,shared_share_of_union
0,bus,DBSCAN,Isolation Forest,0.275,0.719,0.006,0.275
1,bus,GMM,DBSCAN,0.280,0.002,0.718,0.280
2,bus,GMM,Isolation Forest,0.572,0.214,0.214,0.572
3,fhvhv,DBSCAN,Isolation Forest,0.233,0.758,0.009,0.233
4,fhvhv,GMM,DBSCAN,0.244,0.000,0.756,0.244
5,fhvhv,GMM,Isolation Forest,0.429,0.286,0.286,0.429
6,multimodal,DBSCAN,Isolation Forest,0.215,0.726,0.059,0.215
7,multimodal,GMM,DBSCAN,0.213,0.060,0.726,0.213
8,multimodal,GMM,Isolation Forest,0.458,0.271,0.271,0.458
9,subway,DBSCAN,Isolation Forest,0.115,0.848,0.036,0.115


Framework overlap comparison rendered.


Findings\. GMM overlaps much more with Isolation Forest than with DBSCAN in every modality, with GMM\-versus\-Isolation\-Forest Jaccard values ranging from 0\.302 to 0\.572, versus only 0\.131 to 0\.280 against DBSCAN\. The so\-what is that GMM is behaving much more like a fellow sparse probabilistic/isolation\-style detector than like DBSCAN’s broader density\-based surface, even though it is not merely duplicating Isolation Forest one\-for\-one\.

Compare how much of each pairwise anomaly union is shared versus framework\-specific\. This view is meant to answer a slightly different question than Jaccard alone\. Instead of asking only how similar two anomaly surfaces are, it shows how the combined anomaly surface breaks into three pieces: rows flagged only by GMM, rows flagged by both frameworks, and rows flagged only by the comparison framework\. In practical terms, this makes it easy to see whether GMM is contributing a substantial unique anomaly slice or only a very small amount beyond what the other framework already captures\.

In [30]:
# ---------------------------------------------------------------------
# Visualize pairwise union composition for GMM vs DBSCAN and GMM vs IF
# ---------------------------------------------------------------------

gmm_union_composition_df = (
    framework_overlap_df.loc[
        framework_overlap_df["left_framework"].eq("GMM")
        & framework_overlap_df["right_framework"].isin(["DBSCAN", "Isolation Forest"])
    ]
    .copy()
    .sort_values(["right_framework", "feature_set"])
    .reset_index(drop=True)
)

feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
comparison_order = ["DBSCAN", "Isolation Forest"]
comparison_label_map = {
    "DBSCAN": "GMM vs DBSCAN",
    "Isolation Forest": "GMM vs Isolation Forest",
}

gmm_union_composition_df["feature_set"] = pd.Categorical(
    gmm_union_composition_df["feature_set"],
    categories=feature_set_order,
    ordered=True,
)
gmm_union_composition_df["right_framework"] = pd.Categorical(
    gmm_union_composition_df["right_framework"],
    categories=comparison_order,
    ordered=True,
)
gmm_union_composition_df["comparison_label"] = (
    gmm_union_composition_df["right_framework"].map(comparison_label_map)
)

display(
    gmm_union_composition_df[
        [
            "feature_set",
            "comparison_label",
            "left_only_share_of_union",
            "shared_share_of_union",
            "right_only_share_of_union",
        ]
    ].rename(
        columns={
            "left_only_share_of_union": "gmm_only_share",
            "shared_share_of_union": "shared_share",
            "right_only_share_of_union": "other_only_share",
        }
    ).style.format(
        {
            "gmm_only_share": "{:.3f}",
            "shared_share": "{:.3f}",
            "other_only_share": "{:.3f}",
        }
    )
)

stack_plot_df = pd.concat(
    [
        gmm_union_composition_df.assign(
            segment_label="Other only",
            segment_value=gmm_union_composition_df["right_only_share_of_union"],
        ),
        gmm_union_composition_df.assign(
            segment_label="Shared",
            segment_value=gmm_union_composition_df["shared_share_of_union"],
        ),
        gmm_union_composition_df.assign(
            segment_label="GMM only",
            segment_value=gmm_union_composition_df["left_only_share_of_union"],
        ),
    ],
    ignore_index=True,
)

segment_order = ["Other only", "Shared", "GMM only"]
segment_color_map = {
    "Other only": BRAND_COLORS["terracotta"],
    "Shared": BRAND_COLORS["seafoam"],
    "GMM only": BRAND_COLORS["dark_teal"],
}

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["GMM vs DBSCAN", "GMM vs Isolation Forest"],
    horizontal_spacing=0.12,
    shared_yaxes=True,
)

for col_idx, comparison_label in enumerate(
    ["GMM vs DBSCAN", "GMM vs Isolation Forest"],
    start=1,
):
    facet_df = (
        stack_plot_df.loc[stack_plot_df["comparison_label"].eq(comparison_label)]
        .copy()
        .sort_values(["feature_set"])
        .reset_index(drop=True)
    )

    for segment_label in segment_order:
        trace_df = facet_df.loc[facet_df["segment_label"].eq(segment_label)].copy()

        fig.add_trace(
            go.Bar(
                x=trace_df["feature_set"],
                y=trace_df["segment_value"],
                name=segment_label,
                marker_color=segment_color_map[segment_label],
                text=trace_df["segment_value"].map(lambda v: f"{v:.3f}"),
                textposition="inside",
                insidetextanchor="middle",
                legendgroup=segment_label,
                showlegend=(col_idx == 2),
                hovertemplate=(
                    "Feature set=%{x}"
                    + f"<br>Comparison={comparison_label}"
                    + f"<br>Segment={segment_label}"
                    + "<br>Share=%{y:.3f}"
                    + "<extra></extra>"
                ),
            ),
            row=1,
            col=col_idx,
        )

fig.update_yaxes(
    title_text="Share of pairwise anomaly union",
    tickformat=".0%",
    ticklabelstandoff=4,
    row=1,
    col=1,
)
fig.update_yaxes(
    tickformat=".0%",
    ticklabelstandoff=4,
    row=1,
    col=2,
)
fig.update_xaxes(title_text="Feature set", row=1, col=1)
fig.update_xaxes(title_text="Feature set", row=1, col=2)

fig.update_layout(
    width=930,
    height=430,
    margin=dict(l=65, r=40, t=110, b=75),
    barmode="stack",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
        title_text="Union segment",
    ),
)

fig.add_annotation(
    text="How Much Unique Anomaly Surface GMM Adds in Pairwise Framework Comparisons",
    x=0.5,
    y=1.12,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="center",
    yanchor="bottom",
    font=dict(size=18, color=BRAND_COLORS["dark_teal"]),
)

fig.show()

,feature_set,comparison_label,gmm_only_share,shared_share,other_only_share
0,bus,GMM vs DBSCAN,0.002,0.280,0.718
1,fhvhv,GMM vs DBSCAN,0.000,0.244,0.756
2,multimodal,GMM vs DBSCAN,0.060,0.213,0.726
3,subway,GMM vs DBSCAN,0.023,0.131,0.846
4,taxi,GMM vs DBSCAN,0.001,0.152,0.847
5,bus,GMM vs Isolation Forest,0.214,0.572,0.214
6,fhvhv,GMM vs Isolation Forest,0.286,0.429,0.286
7,multimodal,GMM vs Isolation Forest,0.271,0.458,0.271
8,subway,GMM vs Isolation Forest,0.349,0.302,0.349
9,taxi,GMM vs Isolation Forest,0.251,0.497,0.251


Findings\. This view makes the asymmetry much easier to see\. Against DBSCAN, the GMM\-only slice is tiny in every modality, while the DBSCAN\-only slice dominates most of the pairwise union, which tells us that GMM is contributing only a small additional probabilistic layer beyond DBSCAN’s much broader density\-based surface\. Against Isolation Forest, the picture is more balanced: the shared slice is much larger, and the GMM\-only slice is still modest rather than dominant\. So the practical takeaway is that GMM is not redundant, but its distinct contribution is much smaller relative to DBSCAN than its overlap with Isolation Forest might initially suggest\.

### Compare anomaly concentration across local operating slices and zones

Row\-level overlap tells us whether the frameworks flag the same observations\. Concentration metrics tell us whether they still point to the same kinds of operating contexts even when they do not flag the exact same rows\. Slice concentration asks how sharply each framework concentrates anomalies into the shared temporal\_bucket × policy\_geography\_class comparison slices\. Zone concentration asks how geographically diffuse or hotspot\-driven the anomaly surface is\.

Compare anomaly concentration across local operating slices\. min, median, and max slice anomaly share summarize how each framework distributes anomalies across the shared local comparison slices\. In practical terms, a higher median means a framework spreads more anomaly mass across many slices, while a higher maximum means it creates sharper regime\-specific pockets\.

In [31]:
# ---------------------------------------------------------------------
# Compare anomaly concentration across local operating slices
# ---------------------------------------------------------------------

assert "framework_comparison_row_level_df" in globals(), (
    "framework_comparison_row_level_df is not in memory. "
    "Please run the retained-framework loading block first."
)

framework_slice_records = []

for (feature_set, framework_label), row_level_df in (
    framework_comparison_row_level_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
        observed=False,
    )
):
    slice_summary_df = (
        row_level_df.groupby(
            [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN],
            dropna=False,
            observed=False,
        )
        .agg(
            total_rows=(MODEL_ID_COLUMN, "size"),
            anomaly_rows=("anomaly_flag", lambda s: s.fillna(0).astype(int).sum()),
        )
        .reset_index()
    )
    slice_summary_df["anomaly_share"] = (
        slice_summary_df["anomaly_rows"] / slice_summary_df["total_rows"]
    )

    framework_slice_records.append(
        {
            "feature_set": feature_set,
            "framework_label": framework_label,
            "comparison_slices": len(slice_summary_df),
            "min_slice_anomaly_share": float(slice_summary_df["anomaly_share"].min()),
            "median_slice_anomaly_share": float(slice_summary_df["anomaly_share"].median()),
            "max_slice_anomaly_share": float(slice_summary_df["anomaly_share"].max()),
        }
    )

framework_slice_concentration_df = pd.DataFrame(framework_slice_records).sort_values(
    ["feature_set", "framework_label"]
).reset_index(drop=True)

display(
    framework_slice_concentration_df.style.format(
        {
            "min_slice_anomaly_share": "{:.3f}",
            "median_slice_anomaly_share": "{:.3f}",
            "max_slice_anomaly_share": "{:.3f}",
        }
    )
)

slice_plot_df = framework_slice_concentration_df.melt(
    id_vars=["feature_set", "framework_label"],
    value_vars=["median_slice_anomaly_share", "max_slice_anomaly_share"],
    var_name="metric_name",
    value_name="anomaly_share",
)

slice_plot_df["metric_label"] = slice_plot_df["metric_name"].map(
    {
        "median_slice_anomaly_share": "Median slice share",
        "max_slice_anomaly_share": "Max slice share",
    }
)

fig = px.bar(
    slice_plot_df,
    x="feature_set",
    y="anomaly_share",
    color="framework_label",
    facet_col="metric_label",
    barmode="group",
    text="anomaly_share",
    category_orders={
        "feature_set": ["bus", "subway", "taxi", "fhvhv", "multimodal"],
        "framework_label": ["DBSCAN", "Isolation Forest", "GMM"],
    },
    labels={
        "feature_set": "Feature set",
        "anomaly_share": "Anomaly share",
        "framework_label": "Framework",
    },
)

title_map = {
    "Median slice share": "Median slice share",
    "Max slice share": "Max slice share",
}
fig.for_each_annotation(
    lambda a: a.update(text=title_map.get(a.text.split("=")[-1], a.text.split("=")[-1]))
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside", cliponaxis=False)
fig.update_yaxes(ticklabelstandoff=4)
fig.update_layout(
    width=960,
    height=480,
    margin=dict(l=60, r=40, t=110, b=50),
)

fig.add_annotation(
    text="Retained GMM vs DBSCAN vs Isolation Forest Slice Concentration",
    x=0.5,
    y=1.12,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="center",
    yanchor="bottom",
    font=dict(size=18, color=BRAND_COLORS["dark_teal"]),
)

fig.show()

,feature_set,framework_label,comparison_slices,min_slice_anomaly_share,median_slice_anomaly_share,max_slice_anomaly_share
0,bus,DBSCAN,30,0.000,0.019,0.118
1,bus,GMM,30,0.000,0.002,0.063
2,bus,Isolation Forest,30,0.000,0.001,0.074
3,fhvhv,DBSCAN,30,0.005,0.023,0.131
4,fhvhv,GMM,30,0.000,0.002,0.021
5,fhvhv,Isolation Forest,30,0.000,0.003,0.037
6,multimodal,DBSCAN,30,0.000,0.020,0.107
7,multimodal,GMM,30,0.000,0.004,0.037
8,multimodal,Isolation Forest,30,0.000,0.003,0.039
9,subway,DBSCAN,30,0.000,0.017,0.287


Findings\. DBSCAN remains the most locally concentrated framework across operating slices, especially in Subway and Taxi, where its median and maximum slice shares are much larger than either GMM or Isolation Forest\. That does not mean DBSCAN is simply “better” at finding anomalies; it reflects a different anomaly definition\. Here, DBSCAN labels rows as anomalous when they fall outside dense local neighborhoods, while GMM and Isolation Forest were both calibrated to retain only a very sparse 0\.5% anomaly tail\. So the practical takeaway is that DBSCAN is producing a broader and more regime\-concentrated anomaly surface, whereas GMM and Isolation Forest are contributing narrower score\-based tails that still show local structure, but with much less slice\-level concentration\.

Compare anomaly concentration across zones\. top\_zone\_share shows how much of a framework’s anomaly surface lives in its single most dominant zone\. top5\_zone\_share shows how much sits in its top five zones combined\. distinct\_anomaly\_zones shows how geographically wide the framework’s anomaly footprint is\. In practical terms, higher top\-zone shares mean stronger hotspot concentration; higher distinct\-zone counts mean broader geographic spread\.

In [32]:
# ---------------------------------------------------------------------
# Compare anomaly concentration across zones
# ---------------------------------------------------------------------

assert "framework_comparison_row_level_df" in globals(), (
    "framework_comparison_row_level_df is not in memory. "
    "Please run the retained-framework loading block first."
)

framework_zone_records = []

for (feature_set, framework_label), row_level_df in (
    framework_comparison_row_level_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
        observed=False,
    )
):
    anomaly_df = row_level_df.loc[
        row_level_df["anomaly_flag"].fillna(0).astype(int).eq(1)
    ].copy()

    zone_summary_df = (
        anomaly_df.groupby(
            [ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN],
            dropna=False,
            observed=False,
        )
        .agg(anomaly_rows=(MODEL_ID_COLUMN, "size"))
        .reset_index()
        .sort_values(["anomaly_rows", BOROUGH_COLUMN, ZONE_COLUMN], ascending=[False, True, True])
        .reset_index(drop=True)
    )

    total_anomaly_rows = int(zone_summary_df["anomaly_rows"].sum())
    distinct_anomaly_zones = int(zone_summary_df[ZONE_ID_COLUMN].nunique())

    if total_anomaly_rows > 0:
        zone_summary_df["anomaly_share_within_framework"] = (
            zone_summary_df["anomaly_rows"] / total_anomaly_rows
        )
        top_zone_share = float(zone_summary_df["anomaly_share_within_framework"].max())
        top5_zone_share = float(
            zone_summary_df["anomaly_share_within_framework"].head(5).sum()
        )
    else:
        zone_summary_df["anomaly_share_within_framework"] = 0.0
        top_zone_share = np.nan
        top5_zone_share = np.nan

    framework_zone_records.append(
        {
            "feature_set": feature_set,
            "framework_label": framework_label,
            "anomaly_rows": total_anomaly_rows,
            "distinct_anomaly_zones": distinct_anomaly_zones,
            "top_zone_share": top_zone_share,
            "top5_zone_share": top5_zone_share,
        }
    )

framework_zone_concentration_df = pd.DataFrame(framework_zone_records).sort_values(
    ["feature_set", "framework_label"]
).reset_index(drop=True)

display(
    framework_zone_concentration_df.style.format(
        {
            "top_zone_share": "{:.3f}",
            "top5_zone_share": "{:.3f}",
        }
    )
)

zone_plot_df = framework_zone_concentration_df.melt(
    id_vars=["feature_set", "framework_label"],
    value_vars=["top_zone_share", "top5_zone_share"],
    var_name="metric_name",
    value_name="zone_share",
)

zone_plot_df["metric_label"] = zone_plot_df["metric_name"].map(
    {
        "top_zone_share": "Top zone share",
        "top5_zone_share": "Top 5 zone share",
    }
)

fig = px.bar(
    zone_plot_df,
    x="feature_set",
    y="zone_share",
    color="framework_label",
    facet_col="metric_label",
    barmode="group",
    text="zone_share",
    category_orders={
        "feature_set": ["bus", "subway", "taxi", "fhvhv", "multimodal"],
        "framework_label": ["DBSCAN", "Isolation Forest", "GMM"],
    },
    labels={
        "feature_set": "Feature set",
        "zone_share": "Share of anomalies",
        "framework_label": "Framework",
    },
)

title_map = {
    "Top zone share": "Top zone share",
    "Top 5 zone share": "Top 5 zone share",
}
fig.for_each_annotation(
    lambda a: a.update(text=title_map.get(a.text.split("=")[-1], a.text.split("=")[-1]))
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside", cliponaxis=False)
fig.update_yaxes(ticklabelstandoff=4)
fig.update_layout(
    width=960,
    height=480,
    margin=dict(l=60, r=40, t=110, b=50),
)

fig.add_annotation(
    text="Retained GMM vs DBSCAN vs Isolation Forest Zone Concentration",
    x=0.5,
    y=1.12,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="center",
    yanchor="bottom",
    font=dict(size=18, color=BRAND_COLORS["dark_teal"]),
)

fig.show()

,feature_set,framework_label,anomaly_rows,distinct_anomaly_zones,top_zone_share,top5_zone_share
0,bus,DBSCAN,885,108,0.225,0.487
1,bus,GMM,250,27,0.516,0.860
2,bus,Isolation Forest,250,23,0.380,0.848
3,fhvhv,DBSCAN,1026,126,0.108,0.377
4,fhvhv,GMM,250,38,0.156,0.608
5,fhvhv,Isolation Forest,250,59,0.168,0.464
6,multimodal,DBSCAN,859,107,0.119,0.380
7,multimodal,GMM,250,65,0.268,0.536
8,multimodal,Isolation Forest,250,43,0.288,0.628
9,subway,DBSCAN,1590,115,0.115,0.419


Findings\. The spatial footprint comparison reinforces the broader framework story\. DBSCAN spreads anomalies across the widest geography in every modality, especially in Taxi, Subway, FHVHV, Bus, and Multimodal, while both GMM and Isolation Forest are much more selective\. GMM is often more top\-heavy than DBSCAN and in Bus is even more concentrated than Isolation Forest, with over half of all GMM bus anomalies falling into a single zone and 86\.0% falling into its top five zones\. In Subway and Taxi, though, Isolation Forest is the more geographically concentrated sparse detector\. So the practical takeaway is that GMM contributes a narrow probabilistic hotspot layer rather than a citywide anomaly surface, but the exact hotspot pattern still differs meaningfully from Isolation Forest by modality\.

### Record the framework\-comparison takeaway

This last step is just a synthesis step\. We are not choosing the final project\-wide framework winner here\. We are only recording whether GMM looks mostly redundant, partially complementary, or clearly distinct relative to DBSCAN and Isolation Forest within each modality\.

Record a compact framework\-comparison takeaway by modality\. The summary below brings together the most decision\-relevant metrics from the overlap and concentration checks so we can judge whether GMM is mostly repeating the other frameworks or adding a distinct probabilistic anomaly slice\.

In [33]:
# ---------------------------------------------------------------------
# Record the framework-comparison takeaway
# ---------------------------------------------------------------------

gmm_vs_dbscan_overlap_df = framework_overlap_df.loc[
    (framework_overlap_df["left_framework"].eq("GMM"))
    & (framework_overlap_df["right_framework"].eq("DBSCAN"))
].copy()

gmm_vs_if_overlap_df = framework_overlap_df.loc[
    (framework_overlap_df["left_framework"].eq("GMM"))
    & (framework_overlap_df["right_framework"].eq("Isolation Forest"))
].copy()

gmm_overlap_summary_df = (
    gmm_vs_dbscan_overlap_df[
        ["feature_set", "jaccard_similarity", "left_only_share_of_union"]
    ]
    .rename(
        columns={
            "jaccard_similarity": "gmm_vs_dbscan_jaccard",
            "left_only_share_of_union": "gmm_only_share_vs_dbscan_union",
        }
    )
    .merge(
        gmm_vs_if_overlap_df[
            ["feature_set", "jaccard_similarity", "left_only_share_of_union"]
        ].rename(
            columns={
                "jaccard_similarity": "gmm_vs_if_jaccard",
                "left_only_share_of_union": "gmm_only_share_vs_if_union",
            }
        ),
        on="feature_set",
        how="inner",
        validate="one_to_one",
    )
)

gmm_concentration_summary_df = (
    framework_slice_concentration_df.loc[
        framework_slice_concentration_df["framework_label"].eq("GMM"),
        ["feature_set", "median_slice_anomaly_share", "max_slice_anomaly_share"],
    ]
    .merge(
        framework_zone_concentration_df.loc[
            framework_zone_concentration_df["framework_label"].eq("GMM"),
            ["feature_set", "top_zone_share", "top5_zone_share", "distinct_anomaly_zones"],
        ],
        on="feature_set",
        how="inner",
        validate="one_to_one",
    )
)

gmm_framework_takeaway_df = gmm_overlap_summary_df.merge(
    gmm_concentration_summary_df,
    on="feature_set",
    how="inner",
    validate="one_to_one",
)

def classify_gmm_distinctness(row):
    if row["gmm_vs_dbscan_jaccard"] < 0.20 and row["gmm_vs_if_jaccard"] < 0.20:
        return "clearly distinct"
    if row["gmm_vs_dbscan_jaccard"] < 0.35 or row["gmm_vs_if_jaccard"] < 0.35:
        return "partially complementary"
    return "mostly overlapping"

gmm_framework_takeaway_df["framework_takeaway"] = (
    gmm_framework_takeaway_df.apply(classify_gmm_distinctness, axis=1)
)

display(
    gmm_framework_takeaway_df.style.format(
        {
            "gmm_vs_dbscan_jaccard": "{:.3f}",
            "gmm_only_share_vs_dbscan_union": "{:.3f}",
            "gmm_vs_if_jaccard": "{:.3f}",
            "gmm_only_share_vs_if_union": "{:.3f}",
            "median_slice_anomaly_share": "{:.3f}",
            "max_slice_anomaly_share": "{:.3f}",
            "top_zone_share": "{:.3f}",
            "top5_zone_share": "{:.3f}",
        }
    )
)

,feature_set,gmm_vs_dbscan_jaccard,gmm_only_share_vs_dbscan_union,gmm_vs_if_jaccard,gmm_only_share_vs_if_union,median_slice_anomaly_share,max_slice_anomaly_share,top_zone_share,top5_zone_share,distinct_anomaly_zones,framework_takeaway
0,bus,0.280,0.002,0.572,0.214,0.002,0.063,0.516,0.860,27,partially complementary
1,fhvhv,0.244,0.000,0.429,0.286,0.002,0.021,0.156,0.608,38,partially complementary
2,multimodal,0.213,0.060,0.458,0.271,0.004,0.037,0.268,0.536,65,partially complementary
3,subway,0.131,0.023,0.302,0.349,0.003,0.071,0.192,0.544,56,partially complementary
4,taxi,0.152,0.001,0.497,0.251,0.005,0.042,0.120,0.352,79,partially complementary


Findings\. Across all five modalities, the retained GMM surface is best described as partially complementary: it overlaps DBSCAN only modestly, overlaps Isolation Forest more substantially but not redundantly, and preserves its own distinct slice and spatial concentration pattern strongly enough to justify carrying it forward into downstream framework comparison\.

### Section Summary

This section asked whether the retained GMM anomaly surface is mostly repeating DBSCAN and Isolation Forest or adding a meaningfully different probabilistic view of mobility\-state irregularity\. The answer is the latter, but in a measured way\. GMM is much sparser than DBSCAN, since it operates at the retained 0\.5% tail threshold, and it does not reproduce DBSCAN’s broader density\-based anomaly footprint across rows, slices, or zones\. At the same time, GMM is closer to Isolation Forest than to DBSCAN, which makes sense because both are sparse score\-based detectors, but the row\-level overlaps are still far from complete and the spatial concentration patterns differ enough to keep GMM from being redundant\. The aggregate takeaway is that GMM adds a credible third anomaly\-detection lens: narrower and more hotspot\-oriented than DBSCAN, related to but distinct from Isolation Forest, and coherent enough to advance into the final downstream cross\-framework comparison\.

\-\-\-

## 3\.3\.4\.5 Generate and Export Candidate GMM Anomalies

This step turns the retained GMM branch into a real downstream anomaly package by applying the selected probabilistic specification across the full modeling universe for each modality\. The goal is no longer calibration\. It is to materialize one metadata\-enriched row\-level anomaly file per modality, preserve the exact retained GMM settings used to generate it, and write a compact manifest that later notebooks can load directly\.

The export should stay simple: one final row\-level anomaly table per modality, plus a compact manifest that records exactly which retained GMM configuration produced each file\. Because the retained branch is already settled, the expected outcome here is mechanical rather than interpretive: the selected configurations should write cleanly, preserve the final anomaly labels and scores, and produce a small, explicit handoff surface for later framework comparison\.

In [34]:
# ---------------------------------------------------------------------
# Define helper logic for full-universe retained GMM export
# ---------------------------------------------------------------------

import gc

def resolve_first_available_column(df, candidates, label):
    column = next((col for col in candidates if col in df.columns), None)
    assert column is not None, (
        f"Could not identify the column for {label}. "
        f"Tried {candidates}. Available columns: {df.columns.tolist()}"
    )
    return column


def get_gmm_final_prepared_panel_path(feature_set):
    return GMM_FINAL_PREPARED_PANELS_DIR / f"{feature_set}_gmm_prepared_panel.parquet"


def get_gmm_final_temp_output_path(feature_set):
    return GMM_FINAL_ROW_LEVEL_TEMP_DIR / f"{feature_set}_gmm_candidate_anomalies.parquet"


def get_gmm_final_output_path(feature_set):
    return GMM_FINAL_ROW_LEVEL_DIR / f"{feature_set}_gmm_candidate_anomalies.parquet"


def load_full_row_metadata(feature_set):
    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set]
    assert metadata_path.exists(), f"Missing full row metadata for {feature_set}: {metadata_path}"
    return pd.read_parquet(metadata_path)


def load_selected_gmm_configuration_df():
    assert GMM_SELECTED_CONFIGURATION_PATH.exists(), (
        f"Missing selected GMM configuration file: {GMM_SELECTED_CONFIGURATION_PATH}"
    )

    config_df = pd.read_csv(GMM_SELECTED_CONFIGURATION_PATH).copy()

    rename_map = {}

    n_components_col = next(
        (
            col for col in [
                "n_components",
                "component_count",
                "selected_n_components",
            ]
            if col in config_df.columns
        ),
        None,
    )
    covariance_type_col = next(
        (
            col for col in [
                "covariance_type",
                "selected_covariance_type",
            ]
            if col in config_df.columns
        ),
        None,
    )
    tail_threshold_col = next(
        (
            col for col in [
                "tail_threshold",
                "retained_tail_threshold",
                "anomaly_tail_threshold",
                "tail_share",
            ]
            if col in config_df.columns
        ),
        None,
    )

    assert n_components_col is not None, (
        f"Could not find the selected GMM component-count column. "
        f"Available columns: {config_df.columns.tolist()}"
    )
    assert covariance_type_col is not None, (
        f"Could not find the selected GMM covariance-type column. "
        f"Available columns: {config_df.columns.tolist()}"
    )
    assert tail_threshold_col is not None, (
        f"Could not find the selected GMM tail-threshold column. "
        f"Available columns: {config_df.columns.tolist()}"
    )

    if n_components_col != "n_components":
        rename_map[n_components_col] = "n_components"
    if covariance_type_col != "covariance_type":
        rename_map[covariance_type_col] = "covariance_type"
    if tail_threshold_col != "tail_threshold":
        rename_map[tail_threshold_col] = "tail_threshold"

    if rename_map:
        config_df = config_df.rename(columns=rename_map)

    assert "feature_set" in config_df.columns
    assert "representation_name" in config_df.columns
    assert "n_components" in config_df.columns
    assert "covariance_type" in config_df.columns
    assert "tail_threshold" in config_df.columns

    return config_df


def build_full_universe_gmm_prepared_panel(feature_set, rebuild=False):
    prepared_panel_path = get_gmm_final_prepared_panel_path(feature_set)

    if prepared_panel_path.exists() and not rebuild:
        return prepared_panel_path

    full_metadata_df = load_full_row_metadata(feature_set).copy()

    if feature_set != "taxi" or TAXI_PCA_HANDLING_DECISION != "split_prepost_period":
        merged_df, pca_columns, _ = get_final_pca_representation_df(
            feature_set,
            full_metadata_df,
        )
    else:
        assert PRE_POST_CP_COLUMN in full_metadata_df.columns, (
            f"Expected '{PRE_POST_CP_COLUMN}' in taxi full metadata."
        )

        taxi_period_frames = []
        pca_columns = None

        for period_value in ["pre_cp", "post_cp"]:
            scoped_metadata_df = (
                full_metadata_df.loc[
                    full_metadata_df[PRE_POST_CP_COLUMN].eq(period_value)
                ]
                .copy()
                .reset_index(drop=True)
            )

            if len(scoped_metadata_df) == 0:
                continue

            scoped_merged_df, scoped_pca_columns, _ = get_final_pca_representation_df(
                feature_set,
                scoped_metadata_df,
            )

            if pca_columns is None:
                pca_columns = scoped_pca_columns

            taxi_period_frames.append(scoped_merged_df)

        assert taxi_period_frames, "Taxi full-universe prepared-panel build produced no rows."
        merged_df = (
            pd.concat(taxi_period_frames, ignore_index=True)
            .sort_values([DATE_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN, TEMPORAL_BUCKET_COLUMN])
            .reset_index(drop=True)
        )

    assert pca_columns is not None and len(pca_columns) > 0, (
        f"No PCA columns were resolved for {feature_set}."
    )

    prepared_panel_df = merged_df[
        [*DEFAULT_METADATA_COLUMNS, *pca_columns]
    ].copy()

    prepared_panel_df.to_parquet(prepared_panel_path, index=False)
    return prepared_panel_path


def load_final_gmm_export_manifest():
    if "gmm_final_row_level_manifest_df" in globals():
        return gmm_final_row_level_manifest_df.copy()

    assert GMM_ANOMALY_EXPORT_MANIFEST_PATH.exists(), (
        f"Missing GMM export manifest: {GMM_ANOMALY_EXPORT_MANIFEST_PATH}"
    )
    return pd.read_csv(GMM_ANOMALY_EXPORT_MANIFEST_PATH)

Export the retained GMM anomaly rows into final downstream\-ready files\. This block takes the already selected modality\-specific GMM configurations, resolves their retained tail\-calibration outputs, and writes one final row\-level parquet per modality into the 3\.3\.4\.final\_tables directory\.

In [35]:
# ---------------------------------------------------------------------
# Export final retained GMM anomaly row-level outputs
# Full-universe, rerunnable per modality
# ---------------------------------------------------------------------

selected_gmm_configuration_df = load_selected_gmm_configuration_df()

gmm_final_export_records = []
gmm_final_manifest_records = []

total_modalities = len(MODEL_FEATURE_SET_NAMES)
run_start = perf_counter()

for modality_index, feature_set_name in enumerate(MODEL_FEATURE_SET_NAMES, start=1):
    modality_start = perf_counter()

    config_row = (
        selected_gmm_configuration_df.loc[
            selected_gmm_configuration_df["feature_set"].eq(feature_set_name)
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(config_row) == 1, (
        f"Expected exactly one retained GMM configuration for {feature_set_name}, "
        f"found {len(config_row)}."
    )

    config_row = config_row.iloc[0]

    final_output_path = get_gmm_final_output_path(feature_set_name)
    temp_output_path = get_gmm_final_temp_output_path(feature_set_name)

    print(
        f"[{modality_index}/{total_modalities}] {feature_set_name}: "
        f"rep={config_row['representation_name']} | "
        f"k={int(config_row['n_components'])} | "
        f"cov={str(config_row['covariance_type'])} | "
        f"tail={float(config_row['tail_threshold']):.3f}"
    )

    if final_output_path.exists() and not REBUILD_GMM_FINAL_OUTPUTS:
        load_existing_start = perf_counter()

        existing_df = pd.read_parquet(
            final_output_path,
            columns=[
                MODEL_ID_COLUMN,
                TEMPORAL_BUCKET_COLUMN,
                POLICY_GEOGRAPHY_COLUMN,
                "anomaly_flag",
            ],
        )

        anomaly_rows = int(existing_df["anomaly_flag"].fillna(0).astype(int).sum())
        comparison_groups_present = int(
            existing_df[
                [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN]
            ].drop_duplicates().shape[0]
        )

        gmm_final_export_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": config_row["representation_name"],
                "n_components": int(config_row["n_components"]),
                "covariance_type": str(config_row["covariance_type"]),
                "tail_threshold": float(config_row["tail_threshold"]),
                "row_level_output_path": str(final_output_path),
                "rows_written": int(len(existing_df)),
                "anomaly_rows": anomaly_rows,
                "comparison_groups_present": comparison_groups_present,
                "status": "loaded_existing",
            }
        )

        gmm_final_manifest_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": config_row["representation_name"],
                "n_components": int(config_row["n_components"]),
                "covariance_type": str(config_row["covariance_type"]),
                "tail_threshold": float(config_row["tail_threshold"]),
                "row_level_output_path": str(final_output_path),
            }
        )

        print(
            f"  loaded existing in {perf_counter() - load_existing_start:.1f}s | "
            f"rows={len(existing_df):,} | anomalies={anomaly_rows:,}"
        )

        del existing_df
        gc.collect()
        continue

    print(f"  building full-universe export for {feature_set_name}...")

    prepared_panel_start = perf_counter()
    prepared_panel_path = build_full_universe_gmm_prepared_panel(
        feature_set_name,
        rebuild=REBUILD_GMM_FINAL_OUTPUTS,
    )
    prepared_panel_df = pd.read_parquet(prepared_panel_path)
    prepared_panel_seconds = perf_counter() - prepared_panel_start

    pca_columns = [col for col in prepared_panel_df.columns if str(col).startswith("PC")]
    assert pca_columns, f"No PCA columns found in prepared panel for {feature_set_name}."

    print(
        f"  prepared panel ready | rows={len(prepared_panel_df):,} | "
        f"pcs={len(pca_columns)} | time={prepared_panel_seconds:.1f}s"
    )

    matrix_start = perf_counter()
    X = prepared_panel_df[pca_columns].to_numpy(dtype=np.float32, copy=False)
    print(
        f"  matrix materialized | shape={X.shape[0]:,} x {X.shape[1]} | "
        f"time={perf_counter() - matrix_start:.1f}s"
    )

    print("  fitting GaussianMixture...")
    fit_start = perf_counter()

    gmm_model = GaussianMixture(
        n_components=int(config_row["n_components"]),
        covariance_type=str(config_row["covariance_type"]),
        random_state=GMM_RANDOM_STATE,
        n_init=GMM_N_INIT,
        max_iter=GMM_MAX_ITER,
        reg_covar=GMM_REG_COVAR,
    )

    gmm_model.fit(X)
    fit_seconds = perf_counter() - fit_start
    print(f"  model fit complete | time={fit_seconds:.1f}s")

    print("  scoring rows...")
    score_start = perf_counter()
    log_likelihood_score = gmm_model.score_samples(X).astype(np.float32)
    score_seconds = perf_counter() - score_start
    print(f"  scoring complete | time={score_seconds:.1f}s")

    print("  deriving tail cutoff and anomaly flags...")
    threshold_start = perf_counter()
    tail_threshold = float(config_row["tail_threshold"])
    score_cutoff = float(np.quantile(log_likelihood_score, tail_threshold))
    anomaly_flag = (log_likelihood_score <= score_cutoff).astype(np.int8)
    threshold_seconds = perf_counter() - threshold_start

    anomaly_rows = int(anomaly_flag.sum())
    print(
        f"  thresholding complete | cutoff={score_cutoff:.4f} | "
        f"anomalies={anomaly_rows:,} | time={threshold_seconds:.1f}s"
    )

    print("  assembling output table...")
    assemble_start = perf_counter()
    output_df = prepared_panel_df[DEFAULT_METADATA_COLUMNS].copy()
    output_df["feature_set"] = feature_set_name
    output_df["framework_name"] = "gmm"
    output_df["representation_name"] = config_row["representation_name"]
    output_df["n_components"] = int(config_row["n_components"])
    output_df["covariance_type"] = str(config_row["covariance_type"])
    output_df["tail_threshold"] = tail_threshold
    output_df["log_likelihood_score"] = log_likelihood_score
    output_df["anomaly_flag"] = anomaly_flag
    output_df["is_gmm_anomaly"] = anomaly_flag
    print(f"  output table assembled | time={perf_counter() - assemble_start:.1f}s")

    print("  writing temp parquet...")
    write_temp_start = perf_counter()
    output_df.to_parquet(temp_output_path, index=False)
    print(f"  temp parquet written | time={perf_counter() - write_temp_start:.1f}s")

    print("  writing final parquet...")
    write_final_start = perf_counter()
    output_df.to_parquet(final_output_path, index=False)
    print(f"  final parquet written | time={perf_counter() - write_final_start:.1f}s")

    comparison_groups_present = int(
        output_df[
            [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN]
        ].drop_duplicates().shape[0]
    )

    gmm_final_export_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": config_row["representation_name"],
            "n_components": int(config_row["n_components"]),
            "covariance_type": str(config_row["covariance_type"]),
            "tail_threshold": tail_threshold,
            "row_level_output_path": str(final_output_path),
            "rows_written": int(len(output_df)),
            "anomaly_rows": anomaly_rows,
            "comparison_groups_present": comparison_groups_present,
            "status": "written",
        }
    )

    gmm_final_manifest_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": config_row["representation_name"],
            "n_components": int(config_row["n_components"]),
            "covariance_type": str(config_row["covariance_type"]),
            "tail_threshold": tail_threshold,
            "row_level_output_path": str(final_output_path),
        }
    )

    modality_seconds = perf_counter() - modality_start
    total_elapsed = perf_counter() - run_start

    print(
        f"  finished {feature_set_name} | rows={len(output_df):,} | "
        f"anomalies={anomaly_rows:,} | groups={comparison_groups_present} | "
        f"modality_time={modality_seconds:.1f}s | total_elapsed={total_elapsed:.1f}s"
    )

    del prepared_panel_df, X, output_df, log_likelihood_score, anomaly_flag, gmm_model
    gc.collect()

gmm_final_export_summary_df = (
    pd.DataFrame(gmm_final_export_records)
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

gmm_final_row_level_manifest_df = (
    pd.DataFrame(gmm_final_manifest_records)
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

if WRITE_334_OUTPUTS:
    manifest_write_start = perf_counter()
    gmm_final_row_level_manifest_df.to_csv(
        GMM_ANOMALY_EXPORT_MANIFEST_PATH,
        index=False,
    )
    print(f"Manifest written in {perf_counter() - manifest_write_start:.1f}s.")

display(gmm_final_export_summary_df.round({"tail_threshold": 3}))
display(gmm_final_row_level_manifest_df.round({"tail_threshold": 3}))

print(f"GMM final export run complete in {perf_counter() - run_start:.1f}s.")

[1/5] bus: rep=pca_80pct | k=3 | cov=full | tail=0.005
  loaded existing in 0.4s | rows=1,472,947 | anomalies=7,365
[2/5] subway: rep=pca_80pct | k=3 | cov=full | tail=0.005
  loaded existing in 0.3s | rows=911,455 | anomalies=4,558
[3/5] taxi: rep=pca_80pct | k=3 | cov=full | tail=0.005
  loaded existing in 0.4s | rows=1,541,800 | anomalies=7,709
[4/5] fhvhv: rep=pca_80pct | k=2 | cov=full | tail=0.005
  loaded existing in 0.3s | rows=1,541,800 | anomalies=7,710
[5/5] multimodal: rep=pca_80pct | k=2 | cov=full | tail=0.005
  loaded existing in 0.4s | rows=1,541,800 | anomalies=7,709
Manifest written in 0.7s.


,feature_set,representation_name,n_components,covariance_type,tail_threshold,row_level_output_path,rows_written,anomaly_rows,comparison_groups_present,status
0,bus,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/bus_gmm_candidate_anomalies.parquet,1472947,7365,30,loaded_existing
1,fhvhv,pca_80pct,2,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/fhvhv_gmm_candidate_anomalies.parquet,1541800,7710,30,loaded_existing
2,multimodal,pca_80pct,2,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/multimodal_gmm_candidate_anomalies.parquet,1541800,7709,30,loaded_existing
3,subway,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/subway_gmm_candidate_anomalies.parquet,911455,4558,30,loaded_existing
4,taxi,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/taxi_gmm_candidate_anomalies.parquet,1541800,7709,30,loaded_existing


,feature_set,representation_name,n_components,covariance_type,tail_threshold,row_level_output_path
0,bus,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/bus_gmm_candidate_anomalies.parquet
1,fhvhv,pca_80pct,2,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/fhvhv_gmm_candidate_anomalies.parquet
2,multimodal,pca_80pct,2,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/multimodal_gmm_candidate_anomalies.parquet
3,subway,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/subway_gmm_candidate_anomalies.parquet
4,taxi,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/taxi_gmm_candidate_anomalies.parquet


GMM final export run complete in 3.9s.


Findings\. The final GMM export should write exactly one retained branch per modality, with each output preserving the selected PCA\-based structure and the retained 0\.005 tail threshold\. If this block passes, then the notebook has successfully converted the selected probabilistic anomaly surfaces into final row\-level files rather than leaving them only as intermediate calibration artifacts\.

Record the final GMM export manifest and validate the downstream handoff\. This last block makes the retained GMM branch explicit by saving a compact manifest with paths and selected hyperparameters, then confirming that every modality has exactly one final export and that all output files exist\.

In [36]:
# ---------------------------------------------------------------------
# Record the final GMM export manifest and validate the downstream handoff
# ---------------------------------------------------------------------

assert "gmm_final_export_summary_df" in globals(), (
    "gmm_final_export_summary_df is not in memory. Please run the final GMM export block first."
)

gmm_anomaly_export_manifest_df = (
    gmm_final_export_summary_df.assign(framework_name="gmm")[
        [
            "framework_name",
            "feature_set",
            "representation_name",
            "n_components",
            "covariance_type",
            "tail_threshold",
            "row_level_output_path",
        ]
    ]
    .copy()
    .sort_values("feature_set")
    .reset_index(drop=True)
)

gmm_export_validation_df = pd.DataFrame(
    [
        {
            "feature_sets_exported": gmm_anomaly_export_manifest_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "retained_configuration_rows": len(gmm_anomaly_export_manifest_df),
            "paths_found": int(
                gmm_anomaly_export_manifest_df["row_level_output_path"]
                .map(lambda p: Path(p).exists())
                .sum()
            ),
            "status": (
                "pass"
                if (
                    gmm_anomaly_export_manifest_df["feature_set"].nunique()
                    == len(MODEL_FEATURE_SET_NAMES)
                    and len(gmm_anomaly_export_manifest_df) == len(MODEL_FEATURE_SET_NAMES)
                    and gmm_anomaly_export_manifest_df["row_level_output_path"]
                    .map(lambda p: Path(p).exists())
                    .all()
                )
                else "review"
            ),
        }
    ]
)

if WRITE_334_OUTPUTS:
    gmm_anomaly_export_manifest_df.to_csv(
        GMM_ANOMALY_EXPORT_MANIFEST_PATH,
        index=False,
    )

display(
    gmm_anomaly_export_manifest_df.style.format(
        {
            "tail_threshold": "{:.3f}",
        }
    )
)
display(gmm_export_validation_df)

assert gmm_export_validation_df["status"].eq("pass").all(), (
    "One or more final GMM anomaly exports is missing or incomplete."
)

,framework_name,feature_set,representation_name,n_components,covariance_type,tail_threshold,row_level_output_path
0,gmm,bus,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/bus_gmm_candidate_anomalies.parquet
1,gmm,fhvhv,pca_80pct,2,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/fhvhv_gmm_candidate_anomalies.parquet
2,gmm,multimodal,pca_80pct,2,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/multimodal_gmm_candidate_anomalies.parquet
3,gmm,subway,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/subway_gmm_candidate_anomalies.parquet
4,gmm,taxi,pca_80pct,3,full,0.005,/datasets/_deepnote_work/pipeline_data/3.3.4.final_tables/gmm_candidate_anomaly_row_level_parts/taxi_gmm_candidate_anomalies.parquet


,feature_sets_exported,expected_feature_sets,retained_configuration_rows,paths_found,status
0,5,5,5,5,pass


Findings\. The final GMM handoff should now be fully explicit: five retained modality\-specific exports, one per feature set, all PCA\-based, all using full covariance, and all written with the selected final tail threshold\. If this validation passes, then GMM is ready to participate cleanly in downstream anomaly\-framework comparison rather than remaining a calibration\-only artifact\.

\-\-\-

## 3\.3\.4\.6 Review Aggregate Coherence of Retained GMM Anomalies

This section is a lightweight sanity check on the retained GMM surfaces\. We are not reopening model selection, tail\-threshold calibration, or cross\-framework bakeoffs here\. We are simply confirming that the final GMM anomaly outputs form a plausible spatial and temporal footprint in aggregate, so they can advance cleanly into downstream framework comparison\.

### Summarize retained GMM anomaly concentration across operating slices and zones

Start with one compact aggregate summary\. This block reuses the saved framework\-comparison row universe from the previous section and filters it down to the retained GMM surface\. The goal is to check whether GMM looks structurally coherent before we map it: concentrated enough to be interpretable, but not so diffuse that it reads like arbitrary noise\.

In [37]:
# ---------------------------------------------------------------------
# Summarize retained GMM anomaly concentration across slices and zones
# using final full-universe exports
# ---------------------------------------------------------------------

gmm_manifest_df = load_final_gmm_export_manifest()

required_manifest_columns = [
    "feature_set",
    "row_level_output_path",
]
missing_manifest_columns = [
    col for col in required_manifest_columns if col not in gmm_manifest_df.columns
]
assert not missing_manifest_columns, (
    f"Missing required GMM manifest columns: {missing_manifest_columns}"
)

retained_gmm_row_level_frames = []

for manifest_row in gmm_manifest_df.itertuples(index=False):
    row_level_path = Path(manifest_row.row_level_output_path)
    assert row_level_path.exists(), f"Missing final GMM row-level export: {row_level_path}"

    branch_df = pd.read_parquet(row_level_path).copy()

    anomaly_flag_col = resolve_first_available_column(
        branch_df,
        ["anomaly_flag", "is_gmm_anomaly"],
        "GMM anomaly flag",
    )

    if anomaly_flag_col != "anomaly_flag":
        branch_df["anomaly_flag"] = branch_df[anomaly_flag_col]

    branch_df[ZONE_ID_COLUMN] = branch_df[ZONE_ID_COLUMN].astype(str)
    branch_df["feature_set"] = str(manifest_row.feature_set)

    retained_gmm_row_level_frames.append(branch_df)

retained_gmm_row_level_df = (
    pd.concat(retained_gmm_row_level_frames, ignore_index=True)
    .sort_values(
        ["feature_set", DATE_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN, TEMPORAL_BUCKET_COLUMN]
    )
    .reset_index(drop=True)
)

gmm_slice_summary_df = (
    retained_gmm_row_level_df.groupby(
        ["feature_set", TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN],
        dropna=False,
        observed=False,
    )
    .agg(
        total_rows=(MODEL_ID_COLUMN, "size"),
        anomaly_rows=("anomaly_flag", lambda s: s.fillna(0).astype(int).sum()),
    )
    .reset_index()
)
gmm_slice_summary_df["anomaly_share"] = (
    gmm_slice_summary_df["anomaly_rows"] / gmm_slice_summary_df["total_rows"]
)

gmm_slice_coherence_df = (
    gmm_slice_summary_df.groupby("feature_set", dropna=False, observed=False)
    .agg(
        comparison_slices=("anomaly_share", "size"),
        min_slice_anomaly_share=("anomaly_share", "min"),
        median_slice_anomaly_share=("anomaly_share", "median"),
        max_slice_anomaly_share=("anomaly_share", "max"),
    )
    .reset_index()
)

gmm_zone_summary_df = (
    retained_gmm_row_level_df.loc[
        retained_gmm_row_level_df["anomaly_flag"].fillna(0).astype(int).eq(1)
    ]
    .groupby(
        ["feature_set", ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN],
        dropna=False,
        observed=False,
    )
    .agg(anomaly_rows=(MODEL_ID_COLUMN, "size"))
    .reset_index()
)

gmm_zone_summary_df["total_feature_set_anomaly_rows"] = (
    gmm_zone_summary_df.groupby("feature_set", dropna=False, observed=False)["anomaly_rows"]
    .transform("sum")
)
gmm_zone_summary_df["anomaly_share_within_modality"] = (
    gmm_zone_summary_df["anomaly_rows"]
    / gmm_zone_summary_df["total_feature_set_anomaly_rows"]
)

gmm_zone_coherence_df = (
    gmm_zone_summary_df.groupby("feature_set", dropna=False, observed=False)
    .agg(
        anomaly_rows=("anomaly_rows", "sum"),
        top_zone_share=("anomaly_share_within_modality", "max"),
        top5_zone_share=(
            "anomaly_share_within_modality",
            lambda s: float(s.sort_values(ascending=False).head(5).sum()),
        ),
        distinct_anomaly_zones=(ZONE_ID_COLUMN, "nunique"),
    )
    .reset_index()
)

gmm_aggregate_coherence_df = (
    gmm_slice_coherence_df.merge(
        gmm_zone_coherence_df,
        on="feature_set",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("feature_set")
    .reset_index(drop=True)
)

display(
    gmm_aggregate_coherence_df[
        [
            "feature_set",
            "comparison_slices",
            "min_slice_anomaly_share",
            "median_slice_anomaly_share",
            "max_slice_anomaly_share",
            "top_zone_share",
            "top5_zone_share",
            "distinct_anomaly_zones",
        ]
    ].style.format(
        {
            "min_slice_anomaly_share": "{:.3f}",
            "median_slice_anomaly_share": "{:.3f}",
            "max_slice_anomaly_share": "{:.3f}",
            "top_zone_share": "{:.3f}",
            "top5_zone_share": "{:.3f}",
        }
    )
)

gmm_aggregate_plot_df = pd.concat(
    [
        gmm_aggregate_coherence_df.assign(
            metric_label="Median slice share",
            metric_value=gmm_aggregate_coherence_df["median_slice_anomaly_share"],
        ),
        gmm_aggregate_coherence_df.assign(
            metric_label="Max slice share",
            metric_value=gmm_aggregate_coherence_df["max_slice_anomaly_share"],
        ),
        gmm_aggregate_coherence_df.assign(
            metric_label="Top zone share",
            metric_value=gmm_aggregate_coherence_df["top_zone_share"],
        ),
    ],
    ignore_index=True,
)

metric_order = ["Median slice share", "Max slice share", "Top zone share"]
feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]

gmm_aggregate_plot_df["metric_label"] = pd.Categorical(
    gmm_aggregate_plot_df["metric_label"],
    categories=metric_order,
    ordered=True,
)
gmm_aggregate_plot_df["feature_set"] = pd.Categorical(
    gmm_aggregate_plot_df["feature_set"],
    categories=feature_set_order,
    ordered=True,
)

fig = px.bar(
    gmm_aggregate_plot_df.sort_values(["metric_label", "feature_set"]),
    x="feature_set",
    y="metric_value",
    facet_col="metric_label",
    color="metric_label",
    text="metric_value",
    category_orders={
        "feature_set": feature_set_order,
        "metric_label": metric_order,
    },
    labels={
        "feature_set": "Feature set",
        "metric_value": "Share",
        "metric_label": "Metric",
    },
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
    cliponaxis=False,
    showlegend=False,
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(ticklabelstandoff=4)
fig.update_layout(
    width=960,
    height=420,
    margin=dict(l=60, r=40, t=110, b=50),
)

fig.add_annotation(
    text="Retained GMM Aggregate Concentration Summary",
    x=0.5,
    y=1.12,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="center",
    yanchor="bottom",
    font=dict(size=18, color=BRAND_COLORS["dark_teal"]),
)

fig.show()

,feature_set,comparison_slices,min_slice_anomaly_share,median_slice_anomaly_share,max_slice_anomaly_share,top_zone_share,top5_zone_share,distinct_anomaly_zones
0,bus,30,0.000,0.002,0.082,0.459,0.806,90
1,fhvhv,30,0.000,0.003,0.016,0.143,0.591,218
2,multimodal,30,0.001,0.004,0.054,0.282,0.558,246
3,subway,30,0.000,0.003,0.064,0.166,0.518,121
4,taxi,30,0.001,0.006,0.050,0.104,0.340,257


Findings\. The retained GMM surfaces still look aggregate\-coherent, but the modalities are clearly not behaving the same way\. Bus remains the sharpest hotspot case by far: its hottest slice reaches 0\.082, nearly half of all Bus anomalies fall in one zone, and 80\.6% fall in the top five zones\. Multimodal is the next clearest structured case, with a relatively high max\-slice share and a strong top\-zone concentration, while Subway also remains meaningfully concentrated but less extremely so\. Taxi is still the broadest surface overall, with the largest zone footprint and the weakest top\-zone concentration, even though its median slice share is the highest of the five\. FHVHV sits in between: it is spatially broader than Bus or Subway, but not as diffuse as Taxi\. So the practical takeaway is that the retained GMM outputs do not look flat or arbitrary; they form interpretable hotspot patterns, but with materially different concentration profiles by modality\.

### Render one aggregate choropleth for the final retained GMM surface

Now render one aggregate choropleth for the retained GMM surface\. This is a visual sanity check on whether the anomaly mass clusters into a plausible citywide footprint rather than scattering arbitrarily across the map\. To keep the review deterministic, we map the modality with the highest retained top\_zone\_share, since that usually gives the clearest aggregate hotspot pattern\.

In [38]:
# ---------------------------------------------------------------------
# Render one aggregate choropleth for the final retained GMM surface
# ---------------------------------------------------------------------

if "TAXI_ZONE_GEOMETRY_PATH" not in globals():
    TAXI_ZONE_GEOMETRY_PATH = PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"
if "HARMONIZED_TAXI_ZONES_PATH" not in globals():
    HARMONIZED_TAXI_ZONES_PATH = (
        PIPELINE_DATA_DIR / "1.3.1.final_tables" / "nyc_taxi_zones_harmonized.parquet"
    )


def load_harmonized_taxi_zone_geometry_for_gmm():
    candidate_paths = [
        HARMONIZED_TAXI_ZONES_PATH,
        TAXI_ZONE_GEOMETRY_PATH,
    ]

    last_error = None
    for candidate_path in candidate_paths:
        candidate_path = Path(candidate_path)
        if not candidate_path.exists():
            continue

        try:
            gdf = gpd.read_parquet(candidate_path)
        except Exception as exc:
            last_error = exc
            continue

        gdf = gdf.copy()

        rename_map = {}
        if ZONE_ID_COLUMN not in gdf.columns:
            for candidate in ["LocationID", "location_id", "locationid", "OBJECTID", "objectid"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = ZONE_ID_COLUMN
                    break
        if BOROUGH_COLUMN not in gdf.columns:
            for candidate in ["Borough", "borough_name"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = BOROUGH_COLUMN
                    break
        if ZONE_COLUMN not in gdf.columns:
            for candidate in ["Zone", "zone_name", "taxi_zone"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = ZONE_COLUMN
                    break

        if rename_map:
            gdf = gdf.rename(columns=rename_map)

        assert ZONE_ID_COLUMN in gdf.columns, (
            f"Geometry file {candidate_path} does not contain '{ZONE_ID_COLUMN}'."
        )

        if BOROUGH_COLUMN not in gdf.columns:
            gdf[BOROUGH_COLUMN] = ""
        if ZONE_COLUMN not in gdf.columns:
            gdf[ZONE_COLUMN] = ""

        gdf[ZONE_ID_COLUMN] = gdf[ZONE_ID_COLUMN].astype(str)
        return gdf

    raise AssertionError(
        f"Could not load taxi-zone geometry from any candidate path. Last error: {last_error}"
    )


gmm_map_feature_set = (
    gmm_aggregate_coherence_df.sort_values(
        ["top_zone_share", "max_slice_anomaly_share", "feature_set"],
        ascending=[False, False, True],
    )
    .iloc[0]["feature_set"]
)

gmm_map_zone_df = (
    gmm_zone_summary_df.loc[
        gmm_zone_summary_df["feature_set"].eq(gmm_map_feature_set),
        [
            ZONE_ID_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "anomaly_rows",
            "anomaly_share_within_modality",
        ],
    ]
    .copy()
)
gmm_map_zone_df[ZONE_ID_COLUMN] = gmm_map_zone_df[ZONE_ID_COLUMN].astype(str)
gmm_map_zone_df["anomaly_share_pct_within_modality"] = (
    100.0 * gmm_map_zone_df["anomaly_share_within_modality"]
)

taxi_zone_gdf = load_harmonized_taxi_zone_geometry_for_gmm()

# Simplify geometry to keep Deepnote output size manageable.
taxi_zone_plot_gdf = taxi_zone_gdf.copy().to_crs("EPSG:2263")
taxi_zone_plot_gdf["geometry"] = taxi_zone_plot_gdf["geometry"].simplify(
    tolerance=150,
    preserve_topology=True,
)
taxi_zone_plot_gdf = taxi_zone_plot_gdf.to_crs("EPSG:4326")

taxi_zone_geojson = json.loads(
    taxi_zone_plot_gdf[[ZONE_ID_COLUMN, "geometry"]].to_json()
)

minx, miny, maxx, maxy = taxi_zone_plot_gdf.total_bounds
map_center = {
    "lon": float((minx + maxx) / 2.0),
    "lat": float((miny + maxy) / 2.0),
}

feature_zone_df = (
    taxi_zone_plot_gdf[[ZONE_ID_COLUMN, "geometry"]]
    .merge(
        gmm_map_zone_df[
            [
                ZONE_ID_COLUMN,
                BOROUGH_COLUMN,
                ZONE_COLUMN,
                "anomaly_rows",
                "anomaly_share_pct_within_modality",
            ]
        ],
        on=ZONE_ID_COLUMN,
        how="left",
    )
    .fillna(
        {
            "anomaly_rows": 0,
            "anomaly_share_pct_within_modality": 0.0,
            BOROUGH_COLUMN: "",
            ZONE_COLUMN: "",
        }
    )
)

brand_map_scale = [
    [0.00, BRAND_COLORS["ice"]],
    [0.35, BRAND_COLORS["pale_peach"]],
    [0.70, BRAND_COLORS["terracotta"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

fig = go.Figure(
    go.Choroplethmapbox(
        geojson=taxi_zone_geojson,
        locations=feature_zone_df[ZONE_ID_COLUMN],
        z=feature_zone_df["anomaly_share_pct_within_modality"],
        featureidkey=f"properties.{ZONE_ID_COLUMN}",
        colorscale=brand_map_scale,
        zmin=0,
        zmax=float(gmm_map_zone_df["anomaly_share_pct_within_modality"].max()),
        marker_opacity=0.85,
        marker_line_width=0.25,
        marker_line_color=BRAND_COLORS["dark_teal"],
        colorbar=dict(
            title=dict(
                text="Share of modality anomalies (%)",
                font=dict(color=BRAND_COLORS["dark_teal"]),
            ),
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            x=1.01,
        ),
        customdata=np.column_stack(
            [
                feature_zone_df[BOROUGH_COLUMN],
                feature_zone_df[ZONE_COLUMN],
                feature_zone_df["anomaly_rows"],
                feature_zone_df["anomaly_share_pct_within_modality"],
            ]
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Anomaly rows: %{customdata[2]:,.0f}<br>"
            "Share of modality anomalies: %{customdata[3]:.3f}%<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=dict(
        text=f"Final GMM Anomaly Concentration by Taxi Zone: {gmm_map_feature_set.upper()}",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    paper_bgcolor="white",
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=980,
    height=760,
    margin=dict(l=20, r=20, t=80, b=20),
    mapbox=dict(
        style="carto-positron",
        zoom=9.3,
        center=map_center,
    ),
)

fig.show()

print(f"{gmm_map_feature_set.upper()} choropleth rendered.")

/tmp/ipykernel_401/1091493885.py:149: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Choroplethmapbox(


BUS choropleth rendered.


Findings\. The aggregate Bus choropleth confirms that the retained GMM bus surface is not scattering anomalies randomly across the city\. Instead, it is heavily concentrated in a very small set of zones, led by a dominant hotspot in Queens and a secondary cluster across the Jamaica Bay / coastal Queens area\. In practical terms, that means the final bus GMM surface is behaving like a focused hotspot detector rather than a diffuse citywide flagger, which is consistent with the high top\-zone and top\-5 concentration we saw in the summary table\.

### Render one slice\-specific choropleth for a modality with meaningful local concentration

The aggregate map can hide local operating context, so this second map zooms into the single strongest retained GMM operating slice\. That lets us check whether the local hotspot structure still looks coherent once we condition on the shared temporal\_bucket x policy\_geography\_class comparison universe\.

In [39]:
# ---------------------------------------------------------------------
# Render Taxi anomaly concentration for one temporal bucket at a time
# ---------------------------------------------------------------------

assert "retained_gmm_row_level_df" in globals(), (
    "retained_gmm_row_level_df is not in memory. Please run the aggregate GMM concentration summary block first."
)
assert GMM_TEMPORAL_BUCKET_TO_REVIEW in [
    "weekday_am_peak",
    "weekday_midday",
    "weekday_evening",
    "weekday_pm_peak",
    "weekday_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_evening",
    "weekend_pm_peak",
    "weekend_overnight",
], (
    f"Unsupported temporal bucket: {GMM_TEMPORAL_BUCKET_TO_REVIEW}"
)

if "TAXI_ZONE_GEOMETRY_PATH" not in globals():
    TAXI_ZONE_GEOMETRY_PATH = PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"
if "HARMONIZED_TAXI_ZONES_PATH" not in globals():
    HARMONIZED_TAXI_ZONES_PATH = (
        PIPELINE_DATA_DIR / "1.3.1.final_tables" / "nyc_taxi_zones_harmonized.parquet"
    )


def load_harmonized_taxi_zone_geometry_for_gmm():
    candidate_paths = [
        HARMONIZED_TAXI_ZONES_PATH,
        TAXI_ZONE_GEOMETRY_PATH,
    ]

    last_error = None
    for candidate_path in candidate_paths:
        candidate_path = Path(candidate_path)
        if not candidate_path.exists():
            continue

        try:
            gdf = gpd.read_parquet(candidate_path)
        except Exception as exc:
            last_error = exc
            continue

        gdf = gdf.copy()

        rename_map = {}
        if ZONE_ID_COLUMN not in gdf.columns:
            for candidate in ["LocationID", "location_id", "locationid", "OBJECTID", "objectid"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = ZONE_ID_COLUMN
                    break
        if BOROUGH_COLUMN not in gdf.columns:
            for candidate in ["Borough", "borough_name"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = BOROUGH_COLUMN
                    break
        if ZONE_COLUMN not in gdf.columns:
            for candidate in ["Zone", "zone_name", "taxi_zone"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = ZONE_COLUMN
                    break

        if rename_map:
            gdf = gdf.rename(columns=rename_map)

        assert ZONE_ID_COLUMN in gdf.columns, (
            f"Geometry file {candidate_path} does not contain '{ZONE_ID_COLUMN}'."
        )

        if BOROUGH_COLUMN not in gdf.columns:
            gdf[BOROUGH_COLUMN] = ""
        if ZONE_COLUMN not in gdf.columns:
            gdf[ZONE_COLUMN] = ""

        gdf[ZONE_ID_COLUMN] = gdf[ZONE_ID_COLUMN].astype(str)
        return gdf

    raise AssertionError(
        f"Could not load taxi-zone geometry from any candidate path. Last error: {last_error}"
    )


taxi_zone_gdf = load_harmonized_taxi_zone_geometry_for_gmm()

bucket_source_df = retained_gmm_row_level_df.loc[
    retained_gmm_row_level_df["feature_set"].eq("taxi")
    & retained_gmm_row_level_df[TEMPORAL_BUCKET_COLUMN].eq(GMM_TEMPORAL_BUCKET_TO_REVIEW)
].copy()

bucket_total_rows = len(bucket_source_df)

bucket_zone_source_df = (
    bucket_source_df.loc[bucket_source_df["anomaly_flag"].eq(1)]
    .groupby([ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN], dropna=False)
    .size()
    .rename("anomaly_rows")
    .reset_index()
)

bucket_zone_source_df[ZONE_ID_COLUMN] = bucket_zone_source_df[ZONE_ID_COLUMN].astype(str)

if bucket_total_rows > 0:
    bucket_zone_source_df["anomaly_share_pct_within_taxi_bucket"] = (
        100.0 * bucket_zone_source_df["anomaly_rows"] / bucket_total_rows
    )
else:
    bucket_zone_source_df["anomaly_share_pct_within_taxi_bucket"] = 0.0

taxi_zone_plot_gdf = taxi_zone_gdf.copy().to_crs("EPSG:2263")
taxi_zone_plot_gdf["geometry"] = taxi_zone_plot_gdf["geometry"].simplify(
    tolerance=150,
    preserve_topology=True,
)
taxi_zone_plot_gdf = taxi_zone_plot_gdf.to_crs("EPSG:4326")

taxi_zone_geojson = json.loads(
    taxi_zone_plot_gdf[[ZONE_ID_COLUMN, "geometry"]].to_json()
)

minx, miny, maxx, maxy = taxi_zone_plot_gdf.total_bounds
map_center = {
    "lon": float((minx + maxx) / 2.0),
    "lat": float((miny + maxy) / 2.0),
}

bucket_zone_df = (
    taxi_zone_plot_gdf[[ZONE_ID_COLUMN, "geometry"]]
    .merge(
        bucket_zone_source_df[
            [
                ZONE_ID_COLUMN,
                BOROUGH_COLUMN,
                ZONE_COLUMN,
                "anomaly_rows",
                "anomaly_share_pct_within_taxi_bucket",
            ]
        ],
        on=ZONE_ID_COLUMN,
        how="left",
    )
    .fillna(
        {
            "anomaly_rows": 0,
            "anomaly_share_pct_within_taxi_bucket": 0.0,
            BOROUGH_COLUMN: "",
            ZONE_COLUMN: "",
        }
    )
)

brand_map_scale = [
    [0.00, BRAND_COLORS["ice"]],
    [0.35, BRAND_COLORS["pale_peach"]],
    [0.70, BRAND_COLORS["terracotta"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

fig = go.Figure(
    go.Choroplethmapbox(
        geojson=taxi_zone_geojson,
        locations=bucket_zone_df[ZONE_ID_COLUMN],
        z=bucket_zone_df["anomaly_share_pct_within_taxi_bucket"],
        featureidkey=f"properties.{ZONE_ID_COLUMN}",
        colorscale=brand_map_scale,
        zmin=0,
        zmax=float(bucket_zone_source_df["anomaly_share_pct_within_taxi_bucket"].max()) if len(bucket_zone_source_df) > 0 else 0.0,
        marker_opacity=0.85,
        marker_line_width=0.25,
        marker_line_color=BRAND_COLORS["dark_teal"],
        colorbar=dict(
            title=dict(
                text="Share of Taxi bucket anomalies (%)",
                font=dict(color=BRAND_COLORS["dark_teal"]),
            ),
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            x=1.01,
        ),
        customdata=np.column_stack(
            [
                bucket_zone_df[BOROUGH_COLUMN],
                bucket_zone_df[ZONE_COLUMN],
                bucket_zone_df["anomaly_rows"],
                bucket_zone_df["anomaly_share_pct_within_taxi_bucket"],
            ]
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Taxi anomaly rows in bucket: %{customdata[2]:,.0f}<br>"
            "Share of bucket anomalies: %{customdata[3]:.3f}%<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=dict(
        text=f"Taxi GMM Anomaly Concentration by Taxi Zone: {GMM_TEMPORAL_BUCKET_TO_REVIEW}",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    paper_bgcolor="white",
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=980,
    height=760,
    margin=dict(l=20, r=20, t=80, b=20),
    mapbox=dict(
        style="carto-positron",
        zoom=9.3,
        center=map_center,
    ),
)

fig.show()

print(f"Taxi GMM choropleth rendered for {GMM_TEMPORAL_BUCKET_TO_REVIEW}.")

/tmp/ipykernel_401/3039596646.py:164: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Choroplethmapbox(


Taxi GMM choropleth rendered for weekday_pm_peak.


Findings\. The Taxi weekday\_pm\_peak choropleth shows that the retained GMM taxi surface is also locally coherent within a specific operating slice, but in a different way from Bus\. Rather than one outer\-borough super\-hotspot dominating the map, the anomaly mass is concentrated into a small Manhattan\-centered set of zones with a few supporting airport\-adjacent pockets\. Practically, that tells us the Taxi GMM anomalies are not arbitrary within this time bucket; they are locking onto a recognizable evening operating pattern, which is exactly what we wanted this slice\-level sanity check to confirm\.

### Record the aggregate coherence takeaway

Finish with one compact synthesis table\. We are not choosing a global winning framework here; we are only recording whether the retained GMM surface still looks coherent enough in aggregate to advance into downstream comparison\.

In [40]:
# ---------------------------------------------------------------------
# Record the aggregate coherence takeaway
# ---------------------------------------------------------------------

assert "gmm_aggregate_coherence_df" in globals(), (
    "gmm_aggregate_coherence_df was not created. Please run the aggregate GMM concentration block first."
)

gmm_aggregate_coherence_takeaway_df = gmm_aggregate_coherence_df.copy()

def classify_gmm_aggregate_coherence(row):
    if (
        row["max_slice_anomaly_share"] >= 0.050
        and row["top_zone_share"] >= 0.120
        and row["distinct_anomaly_zones"] <= 90
    ):
        return "coherent and hotspot-oriented"
    if (
        row["max_slice_anomaly_share"] >= 0.025
        and row["top_zone_share"] >= 0.080
    ):
        return "coherent but moderate"
    return "review more closely"

gmm_aggregate_coherence_takeaway_df["aggregate_takeaway"] = (
    gmm_aggregate_coherence_takeaway_df.apply(
        classify_gmm_aggregate_coherence,
        axis=1,
    )
)

display(
    gmm_aggregate_coherence_takeaway_df[
        [
            "feature_set",
            "median_slice_anomaly_share",
            "max_slice_anomaly_share",
            "top_zone_share",
            "top5_zone_share",
            "distinct_anomaly_zones",
            "aggregate_takeaway",
        ]
    ].style.format(
        {
            "median_slice_anomaly_share": "{:.3f}",
            "max_slice_anomaly_share": "{:.3f}",
            "top_zone_share": "{:.3f}",
            "top5_zone_share": "{:.3f}",
        }
    )
)

,feature_set,median_slice_anomaly_share,max_slice_anomaly_share,top_zone_share,top5_zone_share,distinct_anomaly_zones,aggregate_takeaway
0,bus,0.002,0.082,0.459,0.806,90,coherent and hotspot-oriented
1,fhvhv,0.003,0.016,0.143,0.591,218,review more closely
2,multimodal,0.004,0.054,0.282,0.558,246,coherent but moderate
3,subway,0.003,0.064,0.166,0.518,121,coherent but moderate
4,taxi,0.006,0.050,0.104,0.340,257,coherent but moderate


Findings\. The aggregate review says the retained GMM surfaces are usable, but they are not equally sharp across modalities\. Bus is still the clearest hotspot case by far: its anomalies are tightly concentrated into a small set of slices and zones, which makes the probabilistic signal easy to read as focused disruption pockets\. Multimodal, Subway, and Taxi all still look coherent, but they are less dominated by one overwhelming hotspot and instead read as moderate\-concentration surfaces with broader spatial footprints\. FHVHV remains the softest case: its slice shares are low, its top zone is not especially dominant, and its anomaly surface spreads across the widest geography short of Taxi and Multimodal\. So the practical takeaway is that none of the retained GMM branches look broken, but FHVHV is still the modality where the anomaly structure feels least decisive\.

Review the leading FHVHV anomaly zones to see whether the retained GMM surface is at least anchored in recognizable operating contexts\. This is a lightweight sanity check: we are not reopening calibration, only checking whether the softer FHVHV footprint still resolves into interpretable hotspot locations\.

In [41]:
# ---------------------------------------------------------------------
# Review the top retained GMM anomaly zones for FHVHV
# ---------------------------------------------------------------------

assert "gmm_zone_summary_df" in globals(), (
    "gmm_zone_summary_df is not in memory. Please run the aggregate GMM concentration summary block first."
)

fhvhv_gmm_top_zones_df = (
    gmm_zone_summary_df.loc[
        gmm_zone_summary_df["feature_set"].eq("fhvhv"),
        [
            "feature_set",
            ZONE_ID_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "anomaly_rows",
            "anomaly_share_within_modality",
        ],
    ]
    .copy()
    .sort_values(
        ["anomaly_share_within_modality", "anomaly_rows", BOROUGH_COLUMN, ZONE_COLUMN],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

fhvhv_gmm_top_zones_df["zone_rank"] = np.arange(1, len(fhvhv_gmm_top_zones_df) + 1)

fhvhv_gmm_top10_zones_df = (
    fhvhv_gmm_top_zones_df.loc[fhvhv_gmm_top_zones_df["zone_rank"].le(10)]
    .copy()
    .reset_index(drop=True)
)

display(
    fhvhv_gmm_top10_zones_df[
        [
            "zone_rank",
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "anomaly_rows",
            "anomaly_share_within_modality",
        ]
    ].style.format(
        {
            "anomaly_share_within_modality": "{:.3f}",
        }
    )
)

fhvhv_gmm_top10_summary_df = pd.DataFrame(
    [
        {
            "top10_zone_rows": int(fhvhv_gmm_top10_zones_df["anomaly_rows"].sum()),
            "top10_share_of_modality": float(
                fhvhv_gmm_top10_zones_df["anomaly_share_within_modality"].sum()
            ),
            "distinct_boroughs_in_top10": int(
                fhvhv_gmm_top10_zones_df[BOROUGH_COLUMN].nunique()
            ),
        }
    ]
)

display(
    fhvhv_gmm_top10_summary_df.style.format(
        {
            "top10_share_of_modality": "{:.3f}",
        }
    )
)

,zone_rank,borough,zone,anomaly_rows,anomaly_share_within_modality
0,1,Staten Island,Freshkills Park,1099,0.143
1,2,Queens,Jamaica Bay,936,0.121
2,3,Queens,Willets Point,849,0.110
3,4,Queens,Breezy Point/Fort Tilden/Riis Beach,839,0.109
4,5,Staten Island,Great Kills Park,834,0.108
5,6,Manhattan,Battery Park,713,0.092
6,7,Bronx,Rikers Island,330,0.043
7,8,Queens,Astoria Park,258,0.033
8,9,Brooklyn,Marine Park/Floyd Bennett Field,255,0.033
9,10,Queens,LaGuardia Airport,212,0.027


,top10_zone_rows,top10_share_of_modality,distinct_boroughs_in_top10
0,6325,0.820,5


Findings\. This helps soften the earlier concern about FHVHV\. The retained GMM FHVHV surface is not collapsing into one suspicious super\-zone; instead, it spreads across a set of recognizable edge and gateway contexts, including Freshkills Park, Great Kills Park, Breezy Point/Fort Tilden/Riis Beach, Jamaica Bay, Willets Point, Battery Park, and LaGuardia Airport\. The top 10 zones account for 82\.0% of FHVHV anomalies, but they span 5 borough groupings rather than one narrow pocket\. So the practical read is that FHVHV looks less like a degenerate single\-hotspot detector and more like a coherent but geographically distributed anomaly surface, which is a reasonable pattern for this modality\.

### Section Summary

The retained GMM anomaly surfaces pass the aggregate coherence check\. Bus is the clearest hotspot case, with anomalies concentrating into a small number of operating slices and zones\. Subway and Multimodal are also coherent, but with more moderate concentration, while Taxi is the broadest of the retained surfaces\. FHVHV turned out not to be broken or arbitrary; its anomaly surface is simply broader, spreading across a recognizable set of edge, airport, park, and gateway\-adjacent zones rather than collapsing into one dominant hotspot\. Taken together, the summary tables, zone review, and choropleths show that the final GMM outputs form plausible aggregate mobility footprints and are coherent enough to advance into downstream framework comparison\.

\-\-\-

## Close

With GMM complete, we now have three calibrated anomaly frameworks built inside the same shared comparison universe: density\-based DBSCAN, isolation\-based Isolation Forest, and probabilistic GMM\. Each one was narrowed to a single retained branch per modality, exported into final row\-level outputs, and checked for enough coherence to move forward\. That gives the next notebook a clean starting point: not more calibration, but direct cross\-framework comparison of how these three anomaly philosophies agree, differ, and complement one another across the same mobility\-state panel\.

### 3\.3\.4 Final Tables Inventory

selected\_gmm\_structures\.csv
Retained GMM structure decisions after structure calibration, recording the selected component count and covariance type for each modality\.

selected\_gmm\_tail\_thresholds\.csv
Retained tail\-threshold decisions after likelihood\-tail calibration, recording the selected anomaly share cutoff for each modality\.

selected\_gmm\_configurations\.csv
Final retained GMM configuration per modality, combining the selected PCA branch, structure choice, and retained tail threshold\.

gmm\_anomaly\_export\_manifest\.csv
Compact manifest pointing downstream notebooks to the final GMM row\-level anomaly exports for each modality\.

gmm\_candidate\_anomaly\_row\_level\_parts/
Directory containing the final metadata\-enriched GMM anomaly row\-level parquet outputs, one file per modality\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>